# Estimativa temporal da intensidade de ciclones tropicais — versão RÁPIDA (1 seed)
### CNN encoder + LSTM + cabeças KAN multitarefa — para ir melhorando o modelo no Kaggle

Esta é a versão de **desenvolvimento** do `pipeline_tc_kaggle.ipynb`: o mesmo modelo, o mesmo pré-processamento e os mesmos dados, mas
com **uma divisão única dos dados no lugar dos 5 folds** e **só a proposta** por padrão. Diferenças:

1. **Sem folds** (removidos em 2026-10-23; veja «Folds: o que foi removido» abaixo). O dataset **inteiro** é dividido **uma vez** em treino,
   validação (para o *early stopping*) e teste. Por padrão a divisão é **por ano, como no TCC** (`DIVISAO = "ano"`): treino 2003–2014,
   validação 2015–2016, teste 2017, então o número do teste é comparável ao do TCC. `DIVISAO = "ciclone"` sorteia ciclones inteiros (80/10/10).
2. **Só a proposta** (`cnn_lstm_kan`) por padrão. As três variantes de controle (`cnn_lstm_mlp`, `cnn_kan_1frame`, `cnn_lstm_kan_vmax`) continuam
   disponíveis em `VARIANTES_RODAR`.
3. **Uma seed só**, a que você escolher na variável `SEED` (a pipeline completa treina 5).
4. **O treino é registrado e desenhado época a época**: perda de treino × validação, RMSE de treino (em modo de avaliação) × validação e
   taxa de aprendizado, por variante (seção 7.4b); mais figuras de diagnóstico do teste (7.4c).
5. **Tudo é empacotado no fim** (seção 10): figuras, tabelas, predições, pesos e um `RELATORIO.md`, num `.zip` para baixar.

**Hiperparâmetros.** Antes desta versão houve um ajuste de hiperparâmetros de treino (`ajustar_hiperparametros.py`; seções 9.12 e 10.11 do `CLAUDE.md`), feito no Mac com a validação por ano e só uma parte dos dados. Testou-se, um de cada vez, `dropout` 0,1→0,3, `weight_decay` 1e-4→5e-2, `prob_rotacao` 0,55→1,0, peso de ΔVmax na perda 0,5→0,2 e `lr` 1e-3→2e-3. **Só o peso de ΔVmax melhorou o Vmax** (0,5 → 0,2: RMSE de Vmax na validação 9,66 → 9,41 nós com 25 épocas e 1 seed; a de ΔVmax ficou praticamente igual, 16,69 → 16,80); os outros quatro pioraram o Vmax. Essa é a **única** diferença para os valores de antes. É uma indicação, não uma prova (dados parciais, 1 seed): para conferir no Kaggle, rode a mesma seed com `HIPERPARAMETROS = "ajustados"` e com `"antigos"` (painel da seção 1) e compare no mesmo teste.

**Folds: o que foi removido.** A pipeline completa (`pipeline_tc_kaggle.ipynb`) usa validação cruzada agrupada por ciclone (k = 5): os ciclones
são divididos em 5 grupos (*folds*) e, para cada grupo, um modelo é treinado nos outros 4 e testado nele, de modo que todo ciclone é testado uma vez.
Aqui isso foi tirado: há um treino por (variante, seed). Para reativar os folds nesta versão é preciso trazer de volta `dividir_folds` (balanceia os
ciclones entre os grupos), o laço fold × seed × variante de `executar_protocolo`, o argumento `fold` de `treinar_execucao`, o `f<fold>` no nome dos
arquivos de `runs/` e as tabelas e figuras por fold (`comparar_por_fold`, `12_desempenho_por_fold`...). A versão deste notebook **com** folds está guardada
em `versoes_anteriores/pipeline_tc_kaggle_rapido_com_folds.ipynb` (na raiz do projeto).

O protocolo de **5 seeds** fica para quando o modelo estiver ajustado: na pipeline completa — ou aqui mesmo, com `SEED = (0, 1, 2, 3, 4)`.

```
 janela de 5 quadros (t-12h … t)         cada quadro = tensor (4, 64, 64)
 canais: IR, PMW, DTA, SDTA
 ┌────┬────┬────┬────┬────┐
 │t-12│t-9 │t-6 │t-3 │ t  │
 └─┬──┴─┬──┴─┬──┴─┬──┴─┬──┘
   ▼    ▼    ▼    ▼    ▼
 ╔═════════════════════════╗
 ║ CNN encoder (pesos      ║   → um vetor latente e(t_i) por quadro
 ║ COMPARTILHADOS)         ║
 ╚════════════╤════════════╝
              ▼
         ╔═════════╗
         ║  LSTM   ║   h_t = LSTM(e(t_i), h_{t-1})  → estado final h(t)
         ╚════╤════╝
      ┌───────┴────────┐
      ▼                ▼
  Cabeça KAN 1     Cabeça KAN 3
  Vmax(t)          ΔVmax = Vmax(t+24h) − Vmax(t)   (intensificação rápida)
```

**Escopo:** a cabeça de **Pmin (pressão mínima) foi deixada de fora** de propósito (decisão do projeto). O código é multitarefa com
**duas** tarefas: `vmax` (principal) e `dvmax`.

**Ciclones duplicados:** o próprio notebook os detecta e remove (seção 2.2), a partir da tabela de metadados do TCIR — não há
arquivo extra para anexar.

---
## Como usar no Kaggle
1. **Dados:** crie/anexe um *Dataset* com os 3 arquivos do TCIR (`TCIR-ATLN_EPAC_WPAC-001.h5`, `TCIR-CPAC_IO_SH-002.h5`,
   `TCIR-ALL_2017-003.h5`). O notebook procura `TCIR-*.h5` sozinho em `/kaggle/input`.
2. **Acelerador:** *Settings → Accelerator →* **GPU T4** (a P100 pode falhar com PyTorch novo). A internet pode ficar **desligada**: não
   há nenhum `pip install` (só PyTorch, NumPy, pandas, SciPy, h5py e Matplotlib, que já vêm no Kaggle).
3. **Teste rápido primeiro:** o notebook vem com `MODO = "teste"` (seção 1). Rode tudo (poucos minutos) para validar o ambiente. Os
   números do modo teste **não valem nada**.
4. **Rodada de verdade:** na seção 1 troque para `MODO = "completo"`, escolha `SEED`, dê um `NOME_EXPERIMENTO` e anote em `NOTAS` o que mudou;
   depois *Save Version → Save & Run All (Commit)*. Cada treino é gravado ao terminar: se a sessão cair, rode de novo e ele retoma de onde parou.
5. **Baixar:** painel *Output* → `resultados_<NOME_EXPERIMENTO>_seed<SEED>.zip` (as figuras também ficam soltas em
   `<NOME_EXPERIMENTO>_seed<SEED>/figuras/`). Comece pelo `RELATORIO.md`.
6. **Mudou o modelo?** Mudando só a configuração (`Config`), a assinatura das execuções muda sozinha e nada é reaproveitado por engano.
   Mudando o **código** (uma classe do modelo, a perda...), a assinatura *não* muda: use outro `NOME_EXPERIMENTO` a cada versão. (Cada
   *Save & Run All* do Kaggle já começa de uma pasta vazia; isso só importa se você re-executar na mesma sessão ou usar `dirs_retomada`.)

**Como ler o resultado de 1 seed.** Duas rodadas com a mesma configuração e seeds diferentes não dão o mesmo número (e, com LSTM, nem a mesma
seed repete o treino na GPU). Com um teste único, a margem de ruído de uma seed é de ≈ 0,45 nó em Vmax e ≈ 0,67 nó em ΔVmax (medida na
execução completa anterior, um fold por vez: `ruido_entre_seeds.py`): só leve a sério uma diferença entre duas rodadas de 1 seed se passar disso,
e confirme com mais seeds antes de adotar uma mudança que ganhe pouco. O teste é **um conjunto só** (por padrão, os ciclones de 2017, ≈ 94): ele
responde à mesma pergunta do TCC, mas é pequeno, então o intervalo do bootstrap por ciclone (seção 7.3b) é largo.

**Onde ficam os resultados:** `/kaggle/working/<NOME_EXPERIMENTO>_seed<SEED>/` (execuções, tabelas, figuras, relatório) e o `.zip` ao lado dela.

## 0. Ambiente: bibliotecas e versões

Só dependemos de bibliotecas que já vêm instaladas no Kaggle. A leitura dos arquivos `.h5` é feita direto com
`h5py` (sem PyTables), porque `pandas.read_hdf` falha em algumas combinações de versões.

In [ ]:
import copy
import glob
import hashlib
import io
import json
import math
import os
import pickle
import random
import re
import shutil
import sys
import time
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # leitura de /kaggle/input (somente leitura): evita erro de trava de arquivo
import h5py
import matplotlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats as sps

if "ipykernel" not in sys.modules and "get_ipython" not in globals():
    matplotlib.use("Agg")  # execução como script (sem tela): só grava as figuras
import matplotlib.pyplot as plt

try:  # no notebook mostra tabelas bonitas; como script cai para print
    from IPython.display import display
except Exception:  # pragma: no cover
    display = print

warnings.filterwarnings("ignore", category=UserWarning, module="torch")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

print(f"python {sys.version.split()[0]} | torch {torch.__version__} | numpy {np.__version__} | "
      f"pandas {pd.__version__} | h5py {h5py.__version__}")


def log(msg: str) -> None:
    """Imprime com carimbo de hora (útil nos logs longos do Kaggle)."""
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

## 1. Configuração (único lugar para mexer)

Os parâmetros do experimento ficam na célula abaixo. O que muda a cada rodada está no **painel de controle**, no topo dela; o resto
(`Config`) são os hiperparâmetros da pipeline: os valores padrão seguem a metodologia (janela de 5 quadros a cada 3 h, tensor 4×64×64,
rotação só no treino, divisão por ano como no TCC) e os itens que ela deixa em aberto estão marcados com **(a confirmar)**.

| variável do painel | para quê |
|---|---|
| `MODO` | `"teste"` roda uma versão minúscula (poucos ciclones, 2 épocas) só para ver se tudo funciona; `"completo"` é o treino de verdade, com o dataset inteiro |
| `SEED` | a seed do treino (inteiro). Muda os pesos iniciais, a ordem dos lotes e as rotações sorteadas; **não** muda a divisão dos dados. Aceita também uma tupla, ex.: `(0, 1, 2, 3, 4)` |
| `DIVISAO` | `"ano"` (padrão, como no TCC): treino 2003–2014, validação 2015–2016, teste 2017. `"ciclone"`: sorteia ciclones inteiros (80% treino, 10% validação, 10% teste) |
| `VARIANTES_RODAR` | quais modelos treinar. Padrão: só a proposta, `cnn_lstm_kan`. Cada variante a mais custa um treino de 3–5 min na T4 por seed |
| `NOME_EXPERIMENTO`, `NOTAS` | rótulo da rodada (nome da pasta e do `.zip`) e uma frase sobre o que mudou; vão para o `RELATORIO.md` e o `config.json` |

In [ ]:
# =============================================================================================================================
# PAINEL DE CONTROLE — o que você muda a cada rodada (o resto da configuração está na classe Config, logo abaixo)
# =============================================================================================================================
MODO = "teste"             # "teste" = verificação rápida do ambiente (minutos; números SEM valor) | "completo" = treino de verdade
SEED = 0                   # <<< seed do treino (inteiro: 0, 1, 2…). Aceita também uma tupla, ex.: (0, 1, 2, 3, 4), para rodar várias
DIVISAO = "ano"            # "ano" = como no TCC: treino 2003–2014, validação 2015–2016, teste 2017 | "ciclone" = sorteia ciclones inteiros (80/10/10)
VARIANTES_RODAR = ("cnn_lstm_kan",)   # só a proposta. Controles (opcionais): "cnn_lstm_mlp" (sem KAN), "cnn_kan_1frame" (sem LSTM), "cnn_lstm_kan_vmax" (sem multitarefa)
NOME_EXPERIMENTO = "base"  # rótulo da rodada: vira o nome da pasta de saída e do .zip (ex.: "lstm256", "dropout02")
NOTAS = ""                 # uma frase sobre o que você mudou nesta rodada (vai para o RELATORIO.md e o config.json)
HIPERPARAMETROS = "ajustados"   # "ajustados" = os do ajuste de 2026-10-23 (seção 9.12 do CLAUDE.md) | "antigos" = os de antes: serve para comparar no MESMO teste.
                           # O ajuste só mudou UM valor: o peso de ΔVmax na perda (0,5 → 0,2); o resto do Config é igual nas duas opções.

if os.environ.get("TCIR_VARIANTES"):          # (usado pelos scripts locais; no Kaggle vale o painel)
    VARIANTES_RODAR = tuple(v.strip() for v in os.environ["TCIR_VARIANTES"].split(",") if v.strip())
SEEDS = tuple(int(s) for s in (os.environ["TCIR_SEEDS"].split(",") if os.environ.get("TCIR_SEEDS") else
                               (SEED if isinstance(SEED, (list, tuple, set)) else (SEED,))))
assert SEEDS and len(set(SEEDS)) == len(SEEDS), "SEED: um inteiro ou uma tupla de inteiros diferentes"
assert HIPERPARAMETROS in ("ajustados", "antigos"), 'HIPERPARAMETROS: "ajustados" ou "antigos"'


@dataclass
class Config:
    # ------------------------------------------------------------------ execução
    modo: str = "teste"                    # "completo" | "teste"
    dir_dados: Optional[str] = None        # pasta com os TCIR-*.h5 (None = procura sozinho)
    dir_saida: Optional[str] = None        # onde gravar tudo (None = /kaggle/working/<NOME_EXPERIMENTO>_seed<SEED> ou ./saida_rapida/<...>)
    detectar_duplicatas: bool = True       # detecta e remove registros redundantes do TCIR (seção 2.2); False = não mexe nos dados
    tol_duplicata_graus: float = 0.25      # IDs diferentes, mesmo instante e posição a ≤ isto (graus) = mesmo ciclone físico
    dirs_retomada: Tuple[str, ...] = ()    # pastas de saídas anteriores (para reaproveitar execuções já concluídas)
    seed_estrutural: int = 2024            # semente do sorteio de ciclones (divisao="ciclone"), das subamostras e do bootstrap (NÃO muda entre seeds de treino)
    usar_cache: bool = True                # grava os quadros extraídos em disco (≈ 2,5 GB); desligue se faltar espaço
    permitir_cpu: bool = False             # o modo "completo" recusa rodar sem GPU (levaria dias); True ignora a trava
    max_horas_treino: Optional[float] = None  # orçamento de tempo para NOVOS treinos nesta sessão (horas). None = automático:
                                           # 10 h no Kaggle (a sessão é limitada a ~12 h) e sem limite fora dele; 0 = sem limite
    nome_experimento: str = "base"         # rótulo da rodada: nome da pasta de saída e do .zip (NÃO muda o resultado)
    notas: str = ""                        # anotação livre sobre a rodada: vai para config.json e RELATORIO.md (NÃO muda o resultado)

    # ------------------------------------------------------------------ dados / pré-processamento
    canal_ir: int = 0                      # ordem dos canais no TCIR: 0=IR, 1=WV, 2=VIS, 3=PMW
    canal_pmw: int = 3
    recorte_armazenado: int = 92           # lado (px) do recorte central guardado ANTES da rotação (≥ 64·√2 ≈ 90,5)
    recorte_final: int = 64                # lado (px) do tensor que entra na rede: (4, 64, 64)
    invalido_acima_de: float = 1000.0      # pixels NaN ou > 1000 são considerados inválidos (como no TCC)
    max_frac_nan_ir: float = 0.10          # quadro com mais de 10% de IR ausente é "inválido" (descartado das janelas)
    tratamento_invalidos: str = "tcc"      # pixels NaN/>1000 dentro de quadros válidos: "tcc" = vira 0 antes do z-score (igual ao TCC);
                                           # "mascara" = vira a média (z=0) e DTA/SDTA ficam 0 nesses pixels
    passo_horas: int = 3                   # espaçamento regular entre quadros consecutivos
    tam_janela: int = 5                    # nº de quadros da sequência: t-12h, t-9h, t-6h, t-3h, t
    horizonte_horas: int = 24              # Δ da intensificação rápida (a confirmar)
    limiar_ri_nos: float = 30.0            # RI = aumento de Vmax ≥ 30 nós em 24 h (Kaplan & DeMaria, 2003)
    prob_rotacao: float = 0.55             # chance de girar uma amostra de TREINO (anotação manuscrita lida como 0,55; a confirmar)
    max_ciclones: Optional[int] = None     # limita nº de ciclones (só para testes rápidos)

    # ------------------------------------------------------------------ modelo
    dim_embedding: int = 128               # tamanho do vetor latente e(t_i) de cada quadro
    dim_oculta: int = 128                  # tamanho do estado oculto h(t) da LSTM
    canais_cnn: Tuple[int, ...] = (32, 64, 128, 256)  # 4 convoluções com stride 2 (estilo CNN-TC): 64→32→16→8→4
    dropout: float = 0.1
    kan_ocultas: Tuple[int, ...] = (16,)   # largura das camadas ocultas da cabeça KAN: [128 → 16 → 1]
    kan_grid: int = 5                      # nº de intervalos do grid das B-splines
    kan_ordem: int = 3                     # grau das B-splines (3 = cúbica)
    mlp_ocultas: Tuple[int, ...] = (64,)   # cabeça MLP usada na ablação "sem KAN"

    # ------------------------------------------------------------------ treino
    tam_lote: int = 128
    tam_lote_aval: int = 256               # lote usado só na avaliação/explicabilidade (menor se a GPU for pequena)
    lr: float = 1e-3
    weight_decay: float = 1e-4
    epocas_max: int = 25
    paciencia: int = 8                     # early stopping: épocas sem melhora na validação interna
    clip_grad: float = 1.0
    pesos_tarefas: Dict[str, float] = field(default_factory=lambda: {"vmax": 1.0, "dvmax": 0.2})   # peso de ΔVmax ajustado em 2026-10-23 (era 0.5; seção 9.12)
    usar_amp: bool = True                  # precisão mista (só vale em GPU CUDA)
    log_por_epoca: int = 5                 # imprime uma linha a cada N épocas (0 = só o resumo de cada execução)
    n_janelas_aval_treino: int = 4000      # janelas FIXAS do treino usadas para medir o erro de treino em modo de avaliação a cada época
                                           # (só diagnóstico das curvas de treino; não muda o treino nem o resultado; 0 = desliga)

    # ------------------------------------------------------------------ divisão dos dados (sem folds: uma divisão só)
    divisao: str = "ano"                   # "ano" = como no TCC: treino ≤ ano_fim_treino, validação até ano_fim_validacao, teste depois
                                           # "ciclone" = sorteia CICLONES inteiros (frac_validacao e frac_teste dos ciclones)
    ano_fim_treino: int = 2014             # divisao="ano": treino 2003–2014 (TCC)
    ano_fim_validacao: int = 2016          # divisao="ano": validação 2015–2016 (early stopping); teste = 2017 (TCC)
    frac_validacao: float = 0.10           # divisao="ciclone": fração dos ciclones reservada para a validação (early stopping)
    frac_teste: float = 0.10               # divisao="ciclone": fração dos ciclones reservada para o teste

    # ------------------------------------------------------------------ protocolo
    seeds: Tuple[int, ...] = (0, 1, 2, 3, 4)           # seeds do treino (controlam a estocasticidade; a divisão dos dados não depende delas)
    variantes: Tuple[str, ...] = ("cnn_lstm_kan", "cnn_lstm_mlp", "cnn_kan_1frame", "cnn_lstm_kan_vmax")
    variante_referencia: str = "cnn_lstm_kan"          # modelo proposto: os outros são comparados contra ele
    alfa: float = 0.05                     # nível de significância dos testes (antes da correção de Holm)
    bootstrap_reps: int = 2000             # réplicas do bootstrap por ciclone (teste principal, seção 7.3b)
    baseline_sem_imagens: bool = True      # compara a rede com previsores de ΔVmax SEM imagens (seção 7.2b)
    outlier_erro_nos: float = 40.0         # |Vmax previsto − real| (nós) a partir do qual a janela entra no diagnóstico da seção 7.5

    # ------------------------------------------------------------------ explicabilidade
    gradcam_amostras: int = 256            # nº de janelas para o mapa Grad-CAM médio
    spline_topk: int = 4                   # nº de variáveis latentes mostradas nas curvas spline


cfg = Config(
    modo=os.environ.get("TCIR_MODO", MODO),
    dir_dados=os.environ.get("TCIR_DIR") or None,
    dir_saida=os.environ.get("TCIR_SAIDA") or None,
    seeds=SEEDS,                                   # as seeds do PAINEL (a pipeline completa usa 5)
    divisao=os.environ.get("TCIR_DIVISAO") or DIVISAO,
    variantes=tuple(VARIANTES_RODAR),
    variante_referencia="cnn_lstm_kan" if "cnn_lstm_kan" in VARIANTES_RODAR else tuple(VARIANTES_RODAR)[0],
    nome_experimento=NOME_EXPERIMENTO,
    notas=NOTAS,
    log_por_epoca=1,                               # uma linha de log por época (a pipeline completa imprime a cada 5)
)
if HIPERPARAMETROS == "antigos":              # os valores de ANTES do ajuste de 2026-10-23, para comparar com os ajustados no mesmo teste
    cfg.pesos_tarefas = {"vmax": 1.0, "dvmax": 0.5}
    print("ATENÇÃO: HIPERPARAMETROS = \"antigos\" — usando os hiperparâmetros de antes do ajuste (só para comparação).")

# Variantes de modelo que podem ser comparadas estatisticamente (todas usam o MESMO tronco CNN):
VARIANTES = {
    "cnn_lstm_kan": dict(
        descricao="PROPOSTA — CNN encoder compartilhado + LSTM + cabeças KAN (Vmax e ΔVmax)",
        temporal="lstm", cabeca="kan", tarefas=("vmax", "dvmax")),
    "cnn_lstm_mlp": dict(
        descricao="Ablação do KAN — igual à proposta, mas com cabeças MLP",
        temporal="lstm", cabeca="mlp", tarefas=("vmax", "dvmax")),
    "cnn_kan_1frame": dict(
        descricao="Ablação do LSTM — só o quadro atual (sem sequência), cabeças KAN",
        temporal="nenhum", cabeca="kan", tarefas=("vmax", "dvmax")),
    "cnn_lstm_kan_vmax": dict(
        descricao="Ablação da multitarefa — LSTM + KAN treinados só para Vmax",
        temporal="lstm", cabeca="kan", tarefas=("vmax",)),
}

if cfg.modo == "teste":  # versão minúscula, só para checar o encanamento (≈ minutos, até em CPU)
    cfg.max_ciclones = 36
    cfg.divisao = "ciclone"                  # com 36 ciclones talvez nenhum caia em 2015–2017: o teste sorteia ciclones
    cfg.frac_validacao = cfg.frac_teste = 0.25
    cfg.epocas_max = 2
    cfg.paciencia = 2
    cfg.gradcam_amostras = 32
    cfg.tam_lote = 64
    cfg.log_por_epoca = 1
    cfg.bootstrap_reps = 200

assert cfg.tratamento_invalidos in ("tcc", "mascara"), 'tratamento_invalidos: "tcc" ou "mascara"'
assert cfg.divisao in ("ano", "ciclone"), 'divisao: "ano" ou "ciclone"'
assert cfg.variante_referencia in cfg.variantes, "a variante de referência precisa estar em cfg.variantes"
assert all(v in VARIANTES for v in cfg.variantes), f"variantes válidas: {list(VARIANTES)}"
assert cfg.recorte_armazenado >= int(math.ceil(cfg.recorte_final * math.sqrt(2))), \
    "o recorte armazenado precisa cobrir o recorte final em QUALQUER rotação (≥ lado·√2)"
print(f"modo = {cfg.modo} | variantes = {cfg.variantes} | divisão = {cfg.divisao} | "
      f"seeds = {cfg.seeds} | épocas ≤ {cfg.epocas_max}")

### 1.1 Dispositivo, pastas e sementes

* **Dispositivo:** GPU CUDA se existir (Kaggle), senão MPS (Mac) ou CPU.
* **Sementes:** `fixar_semente` controla a aleatoriedade do Python, NumPy e PyTorch. Uma *seed* fixa **não**
  garante resultados idênticos em GPU (operações paralelas de ponto flutuante não são determinísticas) — é por isso
  que o protocolo repete cada treino com 5 seeds diferentes: para medir quanto da diferença entre modelos é efeito real
  e quanto é ruído do treino.

In [ ]:
if os.environ.get("TCIR_DISPOSITIVO"):                       # só para testes locais: "cpu", "mps" ou "cuda"
    device = torch.device(os.environ["TCIR_DISPOSITIVO"])
elif torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
usar_amp = bool(cfg.usar_amp and device.type == "cuda")
if cfg.modo == "completo" and device.type == "cpu" and not cfg.permitir_cpu:
    raise RuntimeError(
        "Modo 'completo' sem GPU: o protocolo inteiro levaria dias na CPU. No Kaggle ligue a GPU em Settings → Accelerator "
        "(P100 ou T4) e reinicie; para rodar na CPU mesmo assim, use Config(permitir_cpu=True); para só testar, MODO = 'teste'.")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True  # tamanhos fixos de entrada → cuDNN escolhe os kernels mais rápidos
    print("GPU:", torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("dispositivo:", device, "| precisão mista:", usar_amp)

EM_KAGGLE = os.path.exists("/kaggle/input")
# Uma pasta de saída por rodada: <nome do experimento>_seed<seed>. Rodadas diferentes não se misturam e o .zip se identifica sozinho.
NOME_RODADA = (re.sub(r"[^0-9A-Za-z_.-]+", "-", cfg.nome_experimento).strip("-") or "rodada") + "_seed" + "-".join(str(s) for s in cfg.seeds)
DIR_SAIDA = Path(cfg.dir_saida or (f"/kaggle/working/{NOME_RODADA}" if EM_KAGGLE else f"saida_rapida/{NOME_RODADA}"))
if cfg.modo == "teste":
    DIR_SAIDA = DIR_SAIDA / "teste"  # não mistura o teste com a execução de verdade
if cfg.max_horas_treino is None and EM_KAGGLE and cfg.modo == "completo":
    cfg.max_horas_treino = 10.0     # deixa folga p/ as células de estatística/figuras antes do limite da sessão
for sub in ("cache", "runs", "tabelas", "figuras"):
    (DIR_SAIDA / sub).mkdir(parents=True, exist_ok=True)
print("saída em:", DIR_SAIDA.resolve())
(DIR_SAIDA / "config.json").write_text(json.dumps(asdict(cfg), indent=2, ensure_ascii=False, default=str), encoding="utf-8")  # reprodutibilidade
if cfg.modo == "teste":
    print("\n" + "!" * 78 + "\n  MODO TESTE: poucos ciclones e 2 épocas — só confere se o ambiente funciona.\n"
          "  Os números NÃO valem como resultado. Para o experimento real: MODO = \"completo\".\n" + "!" * 78)


def fixar_semente(seed: int) -> None:
    """Fixa as sementes de Python, NumPy e PyTorch (CPU e GPU)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def formatar_tempo(segundos: float) -> str:
    h, resto = divmod(int(segundos), 3600)
    m, s = divmod(resto, 60)
    return f"{h}h{m:02d}m{s:02d}s" if h else f"{m}m{s:02d}s"

## 2. Etapa 1 — Dados (TCIR)

O **TCIR** (*Tropical Cyclone for Image-to-Intensity Regression*) é distribuído em 3 arquivos HDF5. Cada um tem:

* `matrix` — tensor `(N, 201, 201, 4)` em `float32`: uma imagem de 201×201 px (≈ 7,8 km/px) por linha, com 4 canais
  na ordem **IR, WV, VIS, PMW** (infravermelho, vapor d'água, luz visível, chuva por micro-ondas);
* `info` — tabela com uma linha por imagem: bacia, `ID` do ciclone, lon/lat, `time` (AAAAMMDDHH), `Vmax` (nós),
  `MSLP` (pressão) etc.

As imagens são de **3 em 3 horas** para cada ciclone — a base de toda a modelagem temporal.

### 2.1 Leitura da tabela `info`
O pandas grava a tabela num formato próprio ("fixed"), com as colunas de texto dentro de um *pickle*. Lemos isso direto
com `h5py` e desserializamos **apenas** os objetos NumPy esperados (um *unpickler* restrito, que recusa qualquer outra
coisa — desserializar pickle arbitrário é inseguro).

In [ ]:
class _UnpicklerRestrito(pickle.Unpickler):
    """Desserializa SOMENTE o necessário para recuperar um array NumPy de objetos (strings)."""
    _PERMITIDOS = {("numpy", "dtype"), ("numpy", "ndarray"),
                   ("numpy.core.multiarray", "_reconstruct"), ("numpy._core.multiarray", "_reconstruct")}

    def find_class(self, module, name):
        if (module, name) not in self._PERMITIDOS:
            raise pickle.UnpicklingError(f"objeto não permitido no pickle: {module}.{name}")
        if module == "numpy.core.multiarray":  # NumPy 2 renomeou o módulo para numpy._core
            try:
                from numpy._core.multiarray import _reconstruct  # type: ignore
                return _reconstruct
            except Exception:
                pass
        return super().find_class(module, name)


def ler_info_tcir(caminho: Path) -> pd.DataFrame:
    """Lê a tabela `info` de um .h5 do TCIR. Layout 'fixed' do pandas: axis0 = nomes das colunas,
    axis1 = índice, e blocos blockK_items/blockK_values que agrupam as colunas por tipo."""
    with h5py.File(caminho, "r") as f:
        g = f["info"]
        ordem = [c.decode() for c in g["axis0"][:]]
        indice = g["axis1"][:]
        dados, k = {}, 0
        while f"block{k}_items" in g:
            nomes = [c.decode() for c in g[f"block{k}_items"][:]]
            valores = g[f"block{k}_values"][:]
            if valores.dtype == object:  # bloco de strings (bacia, ID, time): pickle dentro do HDF5
                valores = _UnpicklerRestrito(io.BytesIO(valores[0].tobytes())).load()
            for j, nome in enumerate(nomes):
                dados[nome] = valores[:, j]
            k += 1
    return pd.DataFrame(dados, index=indice)[ordem]


def localizar_arquivos_tcir(cfg: Config) -> List[Path]:
    """Acha os TCIR-*.h5 (na pasta configurada ou em /kaggle/input, ./data, ../data)."""
    raizes = [Path(cfg.dir_dados)] if cfg.dir_dados else [Path("/kaggle/input"), Path("data"), Path("../data"), Path(".")]
    for raiz in raizes:
        if raiz.exists():
            achados, vistos = [], set()
            for p in sorted(p for p in raiz.rglob("*.h5") if "tcir" in p.name.lower()):
                if p.name in vistos:  # o mesmo arquivo anexado por dois datasets: usa só o primeiro (senão tudo duplicaria)
                    log(f"ATENÇÃO: {p} repete o nome de um arquivo já encontrado — ignorado")
                    continue
                vistos.add(p.name); achados.append(p)
            if achados:
                log("arquivos TCIR: " + ", ".join(str(p) for p in achados))
                return achados
    raise FileNotFoundError("Nenhum arquivo TCIR-*.h5 encontrado. Anexe o dataset no Kaggle ou ajuste cfg.dir_dados.")

### 2.2 Duplicatas: detecção e remoção feitas aqui mesmo

É preciso checar "fotos duplicadas do mesmo ciclone no mesmo instante". O notebook faz essa checagem
sozinho, só com a tabela `info` (instantâneo, sem ler imagens), em duas verificações:

* **A — mesmo ID e mesmo instante** (a definição literal);
* **B — IDs diferentes, mesmo instante e mesma posição** (≤ `tol_duplicata_graus`, distância angular): o mesmo ciclone
  gravado duas vezes, o que acontece quando ele cruza de uma bacia para outra e cada centro de previsão lhe dá um ID.
  Em cada instante só há poucos ciclones ativos no planeta, e ciclones vizinhos de verdade ficam a mais de 0,25° um do outro.

Depois decide:

1. **une os IDs** que formam o mesmo ciclone físico (*union-find*) num único `id_ciclone` (o ID com mais registros;
   empate: o menor ID), e a série temporal é a união das linhas;
2. **remove** as linhas repetidas: onde sobra mais de uma linha para o mesmo (`id_ciclone`, instante), fica uma só;
3. usa o `id_ciclone` como **grupo** na divisão dos dados — as duas cópias ficam sempre do mesmo lado (treino, validação ou teste): sem vazamento.

Por fim lê os **pixels** dos pares removidos e confere se são idênticos bit a bit (só informativo: poucas dezenas de
imagens). A remoção vale para **todos** os dados (e não só para o treino) porque a janela deslizante exige instantes
únicos e regulares; na prática ela afeta pouquíssimos registros. `cfg.detectar_duplicatas = False` desliga tudo.

In [ ]:
def _distancia_graus(lon1, lat1, lon2, lat2):
    """Distância angular (graus) entre pontos da esfera — haversine; aceita vetores (broadcast)."""
    lon1, lat1, lon2, lat2 = map(np.deg2rad, (lon1, lat1, lon2, lat2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return np.rad2deg(2 * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0))))


def detectar_duplicatas(tabela: pd.DataFrame, tol_graus: float):
    """Devolve (id_ciclone, duplicata) alinhados a `tabela`: o ID unificado de cada linha e se ela deve ser removida.
    Só usa metadados (id_original, instante, lon, lat) — mesma regra do verificar_duplicatas.py."""
    ids = tabela["id_original"].to_numpy()
    lon = ((tabela["lon"].to_numpy(dtype=float) + 180.0) % 360.0) - 180.0   # CPAC usa 0..360
    lat = tabela["lat"].to_numpy(dtype=float)

    # Verificação B: pares de IDs diferentes, mesmo instante, posição a ≤ tol_graus → mesmo ciclone físico (union-find)
    pai: Dict[str, str] = {}

    def achar(x: str) -> str:
        pai.setdefault(x, x)
        while pai[x] != x:
            pai[x] = pai[pai[x]]            # compressão de caminho
            x = pai[x]
        return x

    for pos in tabela.groupby("instante").indices.values():     # poucos ciclones por instante: comparar todos os pares é barato
        if len(pos) < 2 or len(set(ids[pos])) < 2:
            continue
        d = _distancia_graus(lon[pos][:, None], lat[pos][:, None], lon[pos][None, :], lat[pos][None, :])
        for a, b in zip(*np.nonzero(np.triu(d <= tol_graus, k=1))):
            if ids[pos[a]] != ids[pos[b]]:
                ra, rb = achar(ids[pos[a]]), achar(ids[pos[b]])
                if ra != rb:
                    pai[rb] = ra

    # ID canônico de cada ciclone físico: o ID com mais registros (empate: o menor ID)
    n_por_id = tabela.groupby("id_original").size()
    raiz = tabela["id_original"].map(achar)
    canonico = {r: sorted(g, key=lambda x: (-n_por_id[x], x))[0]
                for r, g in tabela.assign(_raiz=raiz).groupby("_raiz")["id_original"].unique().items()}
    id_ciclone = raiz.map(canonico)

    # Verificações A e B: mais de uma linha para o mesmo (id_ciclone, instante) → fica uma só (a do ID com mais registros;
    # desempate: menor ID, arquivo, linha)
    t = pd.DataFrame({"id_ciclone": id_ciclone, "instante": tabela["instante"], "n_id": tabela["id_original"].map(n_por_id),
                      "id": tabela["id_original"], "arquivo": tabela["arquivo"], "linha": tabela["linha"]})
    ordem = t.sort_values(["id_ciclone", "instante", "n_id", "id", "arquivo", "linha"],
                          ascending=[True, True, False, True, True, True]).index
    rank = t.loc[ordem].groupby(["id_ciclone", "instante"]).cumcount()
    duplicata = pd.Series(False, index=tabela.index)
    duplicata.loc[ordem] = (rank > 0).to_numpy()
    return id_ciclone, duplicata


def conferir_pixels_duplicatas(tabela: pd.DataFrame, arquivos: Dict[str, Path]):
    """Informativo: para cada linha removida, lê a imagem dela e a da linha mantida (mesmo id_ciclone e instante) e diz se
    são idênticas bit a bit. Devolve (n_pares, n_identicos). Lê só algumas dezenas de imagens."""
    rem = tabela[tabela["duplicata"]]
    if rem.empty:
        return 0, 0
    mantidas = tabela[~tabela["duplicata"]].set_index(["id_ciclone", "instante"])
    abertos, n_ident = {}, 0
    try:
        for _, r in rem.iterrows():
            m = mantidas.loc[(r["id_ciclone"], r["instante"])]
            imgs = []
            for nome, linha in ((r["arquivo"], int(r["linha"])), (m["arquivo"], int(m["linha"]))):
                if nome not in abertos:
                    abertos[nome] = h5py.File(arquivos[nome], "r")
                imgs.append(abertos[nome]["matrix"][linha])
            n_ident += int(np.array_equal(imgs[0], imgs[1], equal_nan=True))
    finally:
        for f in abertos.values():
            f.close()
    return len(rem), n_ident


def carregar_tabela_bruta(cfg: Config):
    """Lê o `info` de todos os arquivos e devolve a tabela (1 linha por imagem) já com as duplicatas detectadas."""
    arquivos = localizar_arquivos_tcir(cfg)
    partes = []
    for arq in arquivos:
        d = ler_info_tcir(arq)
        partes.append(pd.DataFrame({
            "arquivo": arq.name,
            "linha": np.arange(len(d)),                      # posição da imagem dentro do arquivo
            "id_original": d["ID"].astype(str).to_numpy(),
            "bacia": d["data_set"].astype(str).to_numpy(),
            "instante": pd.to_datetime(d["time"].astype(str).to_numpy(), format="%Y%m%d%H"),
            "lon": d["lon"].to_numpy(dtype=float),
            "lat": d["lat"].to_numpy(dtype=float),
            "vmax": d["Vmax"].to_numpy(dtype=float),         # nós
        }))
    tabela = pd.concat(partes, ignore_index=True)
    tabela["ano"] = tabela["id_original"].str[:4].astype(int)

    # ---- duplicatas: detectadas aqui mesmo, só com os metadados (seção 2.2)
    tabela["id_ciclone"] = tabela["id_original"]
    tabela["duplicata"] = False
    resumo_dup = {"detectadas": bool(cfg.detectar_duplicatas), "registros_removidos": 0, "ids_unificados": 0,
                  "pares_conferidos_nos_pixels": 0, "pares_identicos_bit_a_bit": 0}
    if cfg.detectar_duplicatas:
        tabela["id_ciclone"], tabela["duplicata"] = detectar_duplicatas(tabela, cfg.tol_duplicata_graus)
        n_uni = int((tabela["id_ciclone"] != tabela["id_original"]).groupby(tabela["id_original"]).any().sum())
        resumo_dup.update(registros_removidos=int(tabela["duplicata"].sum()), ids_unificados=n_uni)
        log(f"duplicatas: {resumo_dup['registros_removidos']} registros a remover, {n_uni} ID(s) unificado(s) com outro "
            f"(tolerância de posição {cfg.tol_duplicata_graus}°)")
        try:  # a conferência dos pixels é só informativa: se falhar, a remoção (feita pelos metadados) continua valendo
            n_pares, n_ident = conferir_pixels_duplicatas(tabela, {a.name: a for a in arquivos})
            resumo_dup.update(pares_conferidos_nos_pixels=n_pares, pares_identicos_bit_a_bit=n_ident)
            log(f"  conferência de pixels: {n_ident} de {n_pares} registros removidos são idênticos bit a bit à cópia mantida")
        except Exception as erro:
            log(f"  (conferência de pixels ignorada: {erro})")
    else:
        log("ATENÇÃO: cfg.detectar_duplicatas = False — seguindo SEM remover duplicatas (risco de vazamento entre treino e teste).")

    # ---- modo teste: subconjunto pequeno de ciclones (sempre inclui os ciclones duplicados, p/ exercitar a lógica)
    if cfg.max_ciclones:
        rng = np.random.default_rng(cfg.seed_estrutural)
        ids = np.array(sorted(tabela["id_ciclone"].unique()))
        obrigatorios = set(tabela.loc[tabela["id_ciclone"] != tabela["id_original"], "id_ciclone"])
        obrigatorios |= set(tabela.loc[tabela["duplicata"], "id_ciclone"])
        restantes = [i for i in ids if i not in obrigatorios]
        escolhidos = set(rng.choice(restantes, size=max(cfg.max_ciclones - len(obrigatorios), 1), replace=False)) | obrigatorios
        tabela = tabela[tabela["id_ciclone"].isin(escolhidos)].reset_index(drop=True)
        log(f"modo teste: {len(escolhidos)} ciclones, {len(tabela)} imagens")

    tabela["pos_frame"] = np.arange(len(tabela))  # posição da imagem no array de quadros extraído
    return tabela, {a.name: a for a in arquivos}, resumo_dup


tabela_bruta, arquivos_h5, resumo_duplicatas = carregar_tabela_bruta(cfg)
print(f"{len(tabela_bruta)} imagens | {tabela_bruta['id_original'].nunique()} IDs originais → "
      f"{tabela_bruta['id_ciclone'].nunique()} ciclones | anos {tabela_bruta['ano'].min()}–{tabela_bruta['ano'].max()}")
display(tabela_bruta.head(3))

### 2.3 Extração dos quadros: bandas IR + PMW, recorte central e limpeza

Para cada imagem de 201×201×4 fazemos (igual ao TCC, Figura 3):

1. **Seleção de bandas:** ficam só **IR** e **PMW**. A luz visível (VIS) some à noite (vem toda NaN) e o vapor
   d'água (WV) é muito parecido com o IR (risco de *overfitting*).
2. **Recorte central:** guardamos o centro `92×92` (e não já 64×64). Motivo: a rotação aleatória do *data augmentation*
   é feita **antes** do corte final 64×64, como no TCC. Um recorte de 92 px contém o círculo que circunscreve o quadrado
   64×64 (raio 64·√2/2 ≈ 45,3 px < 46 px), então, **qualquer que seja o ângulo**, o 64×64 final nunca
   tem cantos pretos.
3. **Limpeza:** pixels `NaN` ou `> 1000` são marcados como inválidos. No cache eles ficam como `NaN` (o `float16` suporta
   NaN) e a *política* de tratamento é aplicada na hora de montar o lote (`cfg.tratamento_invalidos`): `"tcc"` (padrão)
   troca por `0` antes do z-score, exatamente como no TCC. Guardamos também a **fração de IR ausente** de cada quadro,
   usada adiante para descartar quadros "vazios".

O resultado `(N, 2, 92, 92)` em `float16` (≈ 2,5 GB para o TCIR inteiro) vai para um **cache** em disco — a leitura dos
~48 GB de HDF5 só acontece uma vez.

In [ ]:
def _blocos_contiguos(v: np.ndarray):
    """[(ini, fim), ...] com os trechos de índices consecutivos de um vetor crescente (leitura sequencial é mais rápida)."""
    if len(v) == 0:
        return []
    quebras = np.flatnonzero(np.diff(v) != 1) + 1
    inis = np.r_[0, quebras]
    fins = np.r_[quebras, len(v)]
    return list(zip(inis, fins))


def extrair_quadros(tabela: pd.DataFrame, arquivos: Dict[str, Path], cfg: Config, lote: int = 256):
    """Lê do HDF5 só IR e PMW no recorte central e devolve (quadros float16 (N,2,R,R) com NaN nos pixels inválidos,
    fração de pixels inválidos do IR (N,))."""
    N, R = len(tabela), cfg.recorte_armazenado
    c0 = 100 - R // 2                      # o centro da imagem 201×201 é o pixel 100 → recorte [c0, c0+R)
    c1 = c0 + R
    quadros = np.empty((N, 2, R, R), dtype=np.float16)
    frac_inv_ir = np.empty(N, dtype=np.float32)
    t0, lidos, ultimo_log = time.time(), 0, time.time()
    for nome, g in tabela.groupby("arquivo", sort=False):
        linhas, pos = g["linha"].to_numpy(), g["pos_frame"].to_numpy()
        ordem = np.argsort(linhas)
        linhas, pos = linhas[ordem], pos[ordem]
        with h5py.File(arquivos[nome], "r") as f:
            m = f["matrix"]
            for ini, fim in _blocos_contiguos(linhas):
                for a in range(ini, fim, lote):
                    b = min(fim, a + lote)
                    # lê só as LINHAS do recorte (trecho contíguo no disco = rápido) e corta as colunas na memória
                    bloco = m[linhas[a]:linhas[b - 1] + 1, c0:c1]                         # (n, R, 201, 4)
                    x = bloco[:, :, c0:c1, :][..., [cfg.canal_ir, cfg.canal_pmw]]         # (n, R, R, 2)
                    x = np.moveaxis(x, -1, 1)                                             # (n, 2, R, R)
                    invalido = ~np.isfinite(x) | (x > cfg.invalido_acima_de)
                    frac_inv_ir[pos[a:b]] = invalido[:, 0].mean(axis=(1, 2))
                    quadros[pos[a:b]] = np.where(invalido, np.nan, x).astype(np.float16)   # inválido → NaN (tratado depois)
                    lidos += b - a
                    if time.time() - ultimo_log > 20:  # progresso a cada ~20 s
                        ultimo_log = time.time()
                        log(f"  extraindo quadros: {lidos}/{N} ({lidos / max(ultimo_log - t0, 1e-9):.0f} img/s)")
    return quadros, frac_inv_ir


def obter_quadros(tabela: pd.DataFrame, arquivos: Dict[str, Path], cfg: Config):
    """Usa o cache em disco se existir (também procura nas pastas de retomada); senão extrai e grava."""
    chave_txt = "|".join(tabela["arquivo"] + ":" + tabela["linha"].astype(str)) + \
                f"|v2|{cfg.recorte_armazenado}|{cfg.canal_ir}|{cfg.canal_pmw}|{cfg.invalido_acima_de}"
    chave = hashlib.md5(chave_txt.encode()).hexdigest()[:12]
    if cfg.usar_cache:
        for base in [DIR_SAIDA] + [Path(d) for d in cfg.dirs_retomada]:
            qf, ff = base / "cache" / f"quadros_{chave}.npy", base / "cache" / f"fracinv_{chave}.npy"
            if qf.exists() and ff.exists():
                log(f"cache de quadros encontrado: {qf}")
                return np.load(qf), np.load(ff)
    log(f"extraindo {len(tabela)} quadros do HDF5 (leitura única; depois fica em cache)...")
    quadros, frac = extrair_quadros(tabela, arquivos, cfg)
    if cfg.usar_cache:
        qf, ff = DIR_SAIDA / "cache" / f"quadros_{chave}.npy", DIR_SAIDA / "cache" / f"fracinv_{chave}.npy"
        try:
            np.save(qf, quadros)
            np.save(ff, frac)
            log(f"cache gravado ({quadros.nbytes / 1e9:.2f} GB)")
        except OSError as erro:  # ex.: disco cheio — o cache é só uma otimização, então segue sem ele
            for arq in (qf, ff):
                arq.unlink(missing_ok=True)
            log(f"ATENÇÃO: não foi possível gravar o cache ({erro}); seguindo sem cache (use cfg.usar_cache=False para silenciar)")
    return quadros, frac


quadros_np, frac_invalido_ir = obter_quadros(tabela_bruta, arquivos_h5, cfg)
print("quadros:", quadros_np.shape, quadros_np.dtype, f"| {quadros_np.nbytes / 1e9:.2f} GB")

### 2.4 Janelas deslizantes (sequências de 5 quadros) e rótulos

Cada **amostra** é uma janela de **5 quadros** do mesmo ciclone, espaçados de exatamente 3 h:
`t-12h, t-9h, t-6h, t-3h, t`. Regras (todas decorrem da LSTM esperar passos regulares):

* **Quadros inválidos** (IR com mais de 10% ausente — satélite sem dado) e **duplicatas removidas** saem da série.
  Quem estava ao lado fica com um buraco de ≥ 6 h, e uma janela **não pode atravessar buracos**: a LSTM só
  enxerga "o próximo item" e trataria um intervalo de 6 h como se fosse de 3 h.
* **DTA/SDTA do quadro mais antigo** precisam de 2 quadros anteriores (`t-15h` e `t-18h`). Por isso cada janela exige
  **7 quadros consecutivos** (`t-18h … t`); os dois extras só entram no cálculo dos canais de diferença.
* **Rótulos** (vêm do *best track*, nunca entram como entrada da rede):
  * `vmax` = Vmax(t) — tarefa principal (*nowcasting*);
  * `dvmax` = Vmax(t+24h) − Vmax(t) — intensificação rápida (*forecasting*). Se o ciclone não tem rótulo em `t+24h`
    (acabou antes), vira `NaN` e a perda dessa tarefa **ignora** a amostra (máscara).

In [ ]:
def construir_janelas(tabela: pd.DataFrame, frac_inv_ir: np.ndarray, cfg: Config):
    t = tabela.loc[~tabela["duplicata"]].copy()
    t["frac_inv_ir"] = frac_inv_ir[t["pos_frame"].to_numpy()]
    t["quadro_valido"] = t["frac_inv_ir"] <= cfg.max_frac_nan_ir
    t = t.sort_values(["id_ciclone", "instante"]).reset_index(drop=True)
    # Guarda de sanidade (não é a verificação de duplicatas, que é feita à parte):
    assert not t.duplicated(["id_ciclone", "instante"]).any(), \
        "há (ciclone, instante) repetido: confira cfg.detectar_duplicatas (seção 2.2)"

    passo = pd.to_timedelta(cfg.passo_horas, unit="h")
    v = t[t["quadro_valido"]].reset_index(drop=True)
    # "corrida" = trecho contínuo de um mesmo ciclone com quadros válidos exatamente a cada `passo` horas
    nova_corrida = (v["id_ciclone"] != v["id_ciclone"].shift()) | (v["instante"].diff() != passo)
    v["corrida"] = nova_corrida.cumsum()
    v["pos_corrida"] = v.groupby("corrida").cumcount()

    n_prev = cfg.tam_janela + 1                       # quadros anteriores ao alvo: 4 da janela + 2 p/ DTA/SDTA = 6
    alvo = np.flatnonzero((v["pos_corrida"] >= n_prev).to_numpy())
    idx_v = alvo[:, None] + np.arange(-n_prev, 1)[None, :]            # (W, 7) posições dentro de `v`
    idx_quadros = v["pos_frame"].to_numpy()[idx_v]                    # (W, 7) posições no array de quadros

    # rótulo futuro: procura Vmax(id, t+Δ) na série COMPLETA (o quadro futuro pode até ser inválido; o rótulo vale)
    serie_vmax = t.set_index(["id_ciclone", "instante"])["vmax"]
    a = v.iloc[alvo]
    fut = pd.MultiIndex.from_arrays([a["id_ciclone"].to_numpy(),
                                     (a["instante"] + pd.to_timedelta(cfg.horizonte_horas, unit="h")).to_numpy()])
    vmax_fut = serie_vmax.reindex(fut).to_numpy(dtype=float)

    jan = pd.DataFrame({
        "id_ciclone": a["id_ciclone"].to_numpy(),
        "instante": a["instante"].to_numpy(),
        "ano": a["ano"].to_numpy(),
        "bacia": a["bacia"].to_numpy(),
        "vmax": a["vmax"].to_numpy(dtype=np.float32),
        "dvmax": (vmax_fut - a["vmax"].to_numpy()).astype(np.float32),   # NaN quando não há rótulo em t+Δ
    })
    return jan, idx_quadros.astype(np.int64), t


janelas, idx_quadros, tabela_serie = construir_janelas(tabela_bruta, frac_invalido_ir, cfg)

n_inv = int((~tabela_serie["quadro_valido"]).sum())
print(f"quadros na série: {len(tabela_serie)} (inválidos: {n_inv} = {100 * n_inv / len(tabela_serie):.2f}%)")
print(f"janelas: {len(janelas)} de {janelas['id_ciclone'].nunique()} ciclones | "
      f"com rótulo ΔVmax: {int(janelas['dvmax'].notna().sum())} | "
      f"RI (ΔVmax ≥ {cfg.limiar_ri_nos:.0f} nós): {int((janelas['dvmax'] >= cfg.limiar_ri_nos).sum())} "
      f"({100 * (janelas['dvmax'] >= cfg.limiar_ri_nos).sum() / max(janelas['dvmax'].notna().sum(), 1):.1f}% das rotuladas)")
display(janelas[["ano", "vmax", "dvmax"]].describe().T[["count", "mean", "std", "min", "50%", "max"]].round(2))
assert len(janelas) > 50, "poucas janelas para treinar — confira os dados/modo de teste"

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
ax[0].hist(janelas["vmax"], bins=40, color="#3b6fb6"); ax[0].set_title("Vmax(t) das janelas (nós)")
ax[1].hist(janelas["dvmax"].dropna(), bins=60, color="#c96a1b"); ax[1].set_title("ΔVmax = Vmax(t+24h) − Vmax(t) (nós)")
ax[1].axvline(cfg.limiar_ri_nos, color="k", ls="--", lw=1); ax[1].text(cfg.limiar_ri_nos + 1, ax[1].get_ylim()[1] * 0.8, "RI")
ax[2].bar(*np.unique(janelas["ano"], return_counts=True), color="#5a8f5a"); ax[2].set_title("janelas por ano")
plt.tight_layout(); plt.savefig(DIR_SAIDA / "figuras" / "00_distribuicoes.png", dpi=110); plt.show()

## 3. Etapa 1 (cont.) — O tensor de entrada `(4, 64, 64)` por quadro

Montamos, **na GPU e a cada lote**, o tensor de cada janela. Tudo é vetorizado, então o treino não fica preso a
`DataLoader`/CPU. Para cada janela, entram 7 quadros `(IR, PMW)` guardados; saem 5 quadros de 4 canais.

| passo | o que faz | por quê |
|---|---|---|
| 0. **pixels inválidos** (NaN) | política `"tcc"` (padrão): viram 0 **antes** do z-score. Política `"mascara"`: viram a média (z = 0) e o DTA/SDTA vale 0 neles | o TCC faz a primeira. Ela cria picos (≈ −7σ no IR) onde falta dado — por exemplo, faixas de varredura do satélite; a segunda evita isso |
| 1. **z-score** por canal | `x_norm = (x − μ) / σ`, com μ e σ **globais do treino** | põe IR e PMW (Kelvin vs. mm/h) na mesma escala. Global (e não por imagem) para não apagar a informação de *temperatura absoluta* do topo das nuvens. μ/σ vêm só do treino (sem vazar a validação nem o teste) |
| 2. **DTA** | `DTA_n = \|IR_n − IR_{n−1}\|` | "velocidade" de mudança das nuvens entre dois quadros (3 h) |
| 3. **SDTA** | `SDTA_n = \|IR_n − 2·IR_{n−1} + IR_{n−2}\|` | "aceleração": diferença entre duas diferenças consecutivas |
| 4. **rotação aleatória** | ângulo θ ~ U(0°, 360°), aplicado com prob. `prob_rotacao`, **só no treino** | ciclones não têm uma rotação canônica (invariância angular) |
| 5. **recorte** 64×64 | centro do quadro girado | a região central do ciclone é a que importa (≈ 2° de lat/lon) |

**Três cuidados de consistência** (ver seção 12 da metodologia), garantidos *por construção* no código abaixo:

1. **Mesma rotação em todos os canais** do quadro e **em todos os 5 quadros da janela**: empilhamos os `5 quadros × 4
   canais = 20 canais` numa só imagem e giramos essa imagem de uma vez. Se cada quadro girasse diferente, a sequência
   ficaria espacialmente incoerente e a LSTM não aprenderia a dinâmica.
2. **Ordem: diferença → rotação.** DTA/SDTA são calculados na grade original (pixels alinhados) *antes* de girar. Girar cada
   quadro do par isoladamente e só então subtrair destruiria o alinhamento e mediria o artefato da rotação, não o
   movimento do ciclone.
3. **Rotação antes do corte** (como no TCC): giramos o recorte 92×92 e só depois ficamos com o 64×64 central — sem cantos
   vazios.

In [ ]:
@dataclass
class Dados:
    """Tudo que o treino precisa, já no dispositivo (GPU)."""
    quadros: torch.Tensor      # (N, 2, R, R) float16 — IR e PMW limpos, recorte central R×R
    idx: torch.Tensor          # (W, 7) int64 — posições dos 7 quadros (t-18h … t) de cada janela
    idx_np: np.ndarray         # a mesma matriz na CPU (para montar listas de quadros sem ir e voltar da GPU)
    y_vmax: torch.Tensor       # (W,) float32 — Vmax(t) em nós
    y_dvmax: torch.Tensor      # (W,) float32 — Vmax(t+Δ) − Vmax(t); NaN onde não há rótulo futuro
    jan: pd.DataFrame          # metadados (ciclone, instante, ano, bacia, rótulos)


def montar_dados(quadros_np: np.ndarray, idx_quadros: np.ndarray, janelas: pd.DataFrame) -> Dados:
    return Dados(
        quadros=torch.from_numpy(quadros_np).to(device),
        idx=torch.from_numpy(idx_quadros).to(device),
        idx_np=idx_quadros,
        y_vmax=torch.from_numpy(janelas["vmax"].to_numpy(np.float32)).to(device),
        y_dvmax=torch.from_numpy(janelas["dvmax"].to_numpy(np.float32)).to(device),
        jan=janelas,
    )


dados = montar_dados(quadros_np, idx_quadros, janelas)
if device.type != "cpu":
    del quadros_np  # a cópia na memória da CPU não é mais necessária
print(f"dados no dispositivo: quadros {tuple(dados.quadros.shape)} ({dados.quadros.element_size() * dados.quadros.numel() / 1e9:.2f} GB) | "
      f"janelas {tuple(dados.idx.shape)}")

In [ ]:
def estatisticas_canais(dados: Dados, pos_quadros: np.ndarray, cfg: Config, n_max: int = 3000, seed: int = 0):
    """Média e desvio-padrão GLOBAIS de (IR, PMW) numa amostra dos quadros de treino.
    Política "tcc": os pixels inválidos entram como 0 (como no TCC). Política "mascara": são ignorados."""
    rng = np.random.default_rng(seed)
    sel = rng.choice(pos_quadros, size=min(n_max, len(pos_quadros)), replace=False)
    x = dados.quadros[torch.as_tensor(sel, device=dados.quadros.device)].float()          # (n, 2, R, R)
    if cfg.tratamento_invalidos == "tcc":
        x = torch.nan_to_num(x, nan=0.0)
        return x.mean(dim=(0, 2, 3)), x.std(dim=(0, 2, 3)).clamp_min(1e-6)                # tensores de forma (2,)
    valido = ~torch.isnan(x)
    n = valido.sum(dim=(0, 2, 3)).clamp_min(1)
    xz = torch.where(valido, x, torch.zeros_like(x))
    mu = xz.sum(dim=(0, 2, 3)) / n
    var = (torch.where(valido, (x - mu.view(1, 2, 1, 1)) ** 2, torch.zeros_like(x))).sum(dim=(0, 2, 3)) / n
    return mu, var.sqrt().clamp_min(1e-6)


def rotacionar_e_recortar(x: torch.Tensor, angulos_graus: torch.Tensor, lado_saida: int) -> torch.Tensor:
    """Gira cada amostra de `x` (B, C, H, W) em torno do centro pelo seu ângulo e recorta `lado_saida`×`lado_saida`.

    Usa uma transformação afim: `affine_grid` cria, para cada pixel de saída, a coordenada correspondente na imagem de
    entrada (já girada e encolhida pelo fator lado_saida/H = recorte); `grid_sample` interpola bilinearmente.
    Com ângulo 0 cada pixel de saída cai exatamente num pixel de entrada → recorte central exato, sem borrar.
    """
    B, C, H, W = x.shape
    ang = torch.deg2rad(angulos_graus.to(x.dtype))
    s = lado_saida / H                                   # fração do quadro que o recorte final cobre (64/92)
    cos, sen = torch.cos(ang) * s, torch.sin(ang) * s
    theta = torch.zeros(B, 2, 3, dtype=x.dtype, device=x.device)
    theta[:, 0, 0], theta[:, 0, 1] = cos, -sen
    theta[:, 1, 0], theta[:, 1, 1] = sen, cos
    grade = F.affine_grid(theta, size=(B, C, lado_saida, lado_saida), align_corners=False)
    return F.grid_sample(x, grade, mode="bilinear", padding_mode="zeros", align_corners=False)


def montar_lote(dados: Dados, ib: torch.Tensor, mu: torch.Tensor, sd: torch.Tensor, cfg: Config,
                treino: bool) -> torch.Tensor:
    """Monta o tensor de entrada (B, 5, 4, 64, 64) das janelas de índices `ib`.

    canais: 0=IR(z), 1=PMW(z), 2=DTA, 3=SDTA — quadros em ordem cronológica (t-12h … t).
    """
    x = dados.quadros[dados.idx[ib]].float()                         # (B, 7, 2, R, R): t-18h … t (NaN = pixel inválido)
    if cfg.tratamento_invalidos == "tcc":
        x = torch.nan_to_num(x, nan=0.0)                              # TCC: NaN/>1000 → 0 ANTES do z-score
    z = (x - mu.view(1, 1, 2, 1, 1)) / sd.view(1, 1, 2, 1, 1)         # 1. z-score por canal
    ir, pmw = z[:, :, 0], z[:, :, 1]                                  # (B, 7, R, R)
    # 2./3. Diferenças temporais calculadas na grade ORIGINAL (antes da rotação), para os 5 quadros da janela
    #       (índices 2…6 dos 7 lidos); o quadro n-1 é o índice 1…5 e o n-2 é o índice 0…4.
    dta = (ir[:, 2:] - ir[:, 1:-1]).abs()
    sdta = (ir[:, 2:] - 2.0 * ir[:, 1:-1] + ir[:, :-2]).abs()
    t = torch.stack([ir[:, 2:], pmw[:, 2:], dta, sdta], dim=2)        # (B, 5, 4, R, R)
    # política "mascara": o NaN se propaga pelas contas (DTA/SDTA ficam NaN se algum quadro envolvido era inválido
    # naquele pixel); aqui viram 0 = valor médio. Na política "tcc" não há NaN e esta linha não faz nada.
    t = torch.nan_to_num(t, nan=0.0)
    B, T, C, R, _ = t.shape
    t = t.reshape(B, T * C, R, R)               # 5 quadros × 4 canais = 20 "canais": UMA rotação vale para todos
    lado = cfg.recorte_final
    if treino and cfg.prob_rotacao > 0:
        girar = torch.rand(B, device=t.device) < cfg.prob_rotacao     # só uma fração das amostras é girada
        ang = torch.rand(B, device=t.device) * 360.0 * girar          # ângulo 0 = sem rotação
        t = rotacionar_e_recortar(t, ang, lado)
    else:                                                             # validação/teste: sem aumento de dados
        m = (R - lado) // 2
        t = t[:, :, m:m + lado, m:m + lado]
    return t.reshape(B, T, C, lado, lado)

### 3.1 Conferência visual do tensor
Uma janela de treino real: cada coluna é um quadro (t-12h … t) e cada linha um canal. A segunda figura mostra **a mesma
janela girada** — repare que todos os canais e todos os quadros giram juntos pelo mesmo ângulo.

In [ ]:
_ib = torch.tensor([len(janelas) // 2], device=device)
_mu, _sd = estatisticas_canais(dados, np.unique(idx_quadros), cfg, n_max=2000)
print("μ(IR, PMW) =", _mu.cpu().numpy().round(2), "| σ =", _sd.cpu().numpy().round(2))
_normal = montar_lote(dados, _ib, _mu, _sd, cfg, treino=False)[0].cpu().numpy()
_cfg_giro = copy.copy(cfg); _cfg_giro.prob_rotacao = 1.0
torch.manual_seed(3)
_girado = montar_lote(dados, _ib, _mu, _sd, _cfg_giro, treino=True)[0].cpu().numpy()
_nomes = ["IR (z-score)", "PMW (z-score)", "DTA", "SDTA"]
for titulo, arr in (("sem rotação (validação/teste)", _normal), ("com rotação aleatória (treino)", _girado)):
    fig, ax = plt.subplots(4, 5, figsize=(11, 9))
    for c in range(4):
        for f in range(5):
            lo, hi = np.percentile(arr[f, c], [1, 99])  # limites robustos: um pixel extremo não "apaga" o resto
            ax[c, f].imshow(arr[f, c], cmap="gray_r" if c == 0 else "viridis", vmin=lo, vmax=hi); ax[c, f].axis("off")
            if c == 0: ax[c, f].set_title(f"t-{(4 - f) * cfg.passo_horas}h" if f < 4 else "t", fontsize=10)
            if f == 0: ax[c, f].text(-0.15, 0.5, _nomes[c], transform=ax[c, f].transAxes, rotation=90, va="center", ha="right", fontsize=10)
    fig.suptitle(f"Janela {janelas.id_ciclone.iloc[len(janelas) // 2]} — {titulo}", y=0.995)
    plt.tight_layout(); plt.savefig(DIR_SAIDA / "figuras" / f"01_tensor_{'giro' if 'com' in titulo else 'normal'}.png", dpi=100); plt.show()

## 4. Etapas 2, 3 e 4 — O modelo: CNN encoder → LSTM → cabeças KAN

### 4.1 CNN encoder com pesos compartilhados (Etapa 2)
O encoder transforma **cada quadro** `(4, 64, 64)` num vetor latente `e(t_i)` de `dim_embedding` números. Ele é *um
só* — aplicamos a mesma rede (mesmos pesos) aos 5 quadros, juntando os 5 quadros no eixo do lote
(`(B·5, 4, 64, 64)`). É como olhar 5 fotos com os mesmos óculos: não aprendemos 5 formas de enxergar, e a tarefa de
ligar as 5 descrições numa "história" é da LSTM.

A arquitetura segue o espírito do **CNN-TC** do TCC: 4 convoluções com *stride* 2 (a imagem encolhe 64→32→16→8→4,
**sem pooling** — imagens de ciclone não têm bordas nítidas e o pooling joga informação fora), ReLU, e uma camada
totalmente conectada no fim. Diferenças deliberadas: adicionamos *BatchNorm* nas convoluções (treino mais estável) e um pouco
de *dropout*; **tiramos a camada final de regressão** — o encoder devolve o vetor, não a velocidade; e terminamos com uma
**LayerNorm** no embedding. Esta última tem um motivo prático, observado nos testes: sem ela os embeddings chegam a valores
~10, os portões da LSTM saturam e as variáveis de `h(t)` ficam coladas em ±1 — o treino fica instável no início e as splines
da KAN ficam sem informação (só se vê a curva nas bordas). Com a LayerNorm o embedding tem escala ~1 e `h(t)` ocupa todo o
intervalo (−1, 1).

### 4.2 LSTM — a dinâmica temporal (Etapa 3)
Uma LSTM lê os 5 vetores **em ordem** (t-12h → t) mantendo uma memória. A cada passo ela decide, com três "válvulas"
(portões, saídas entre 0 e 1), o que esquecer, o que gravar e o que mostrar:

$$f_t=\sigma(W_f[h_{t-1},x_t]+b_f)\quad i_t=\sigma(W_i[h_{t-1},x_t]+b_i)\quad \tilde c_t=\tanh(W_c[h_{t-1},x_t]+b_c)$$
$$c_t=f_t\odot c_{t-1}+i_t\odot\tilde c_t\qquad o_t=\sigma(W_o[h_{t-1},x_t]+b_o)\qquad h_t=o_t\odot\tanh(c_t)$$

O estado de célula $c_t$ só sofre somas e multiplicações por portões, então o gradiente atravessa muitos passos sem
sumir (o problema da RNN simples). Na notação do diagrama: $h_t=\mathrm{LSTM}(e(t_i),h_{t-1})$. O **estado final
`h(t)`** resume "como o ciclone evoluiu até agora" e é a **única** entrada das cabeças — nunca olhamos quadros do futuro
(sem vazamento no ΔVmax). Como $h_t=o_t\odot\tanh(c_t)$, ele fica sempre em (−1, 1) — faixa que combina com as KANs.

### 4.3 KAN — Kolmogorov-Arnold Network (Etapa 4)
Num MLP, cada neurônio soma entradas com **pesos** e aplica uma ativação **fixa** (ReLU): a não-linearidade mora no nó.
Numa **KAN** a lógica se inverte: cada **aresta** carrega uma **função aprendível** $\varphi_{j,i}$ de uma variável, e o nó
só soma:

$$x_{l+1,j}=\sum_i \varphi_{l,j,i}(x_{l,i}),\qquad \varphi(x)=w_b\,\mathrm{silu}(x)+\sum_{m} c_m\,B_m(x)$$

$B_m$ são **B-splines** (curvas polinomiais por partes, grau 3 = cúbicas, sobre um grid de nós igualmente espaçados) e os
$c_m$ são os coeficientes aprendidos; o termo $w_b\,\mathrm{silu}(x)$ é um "caminho base" suave que ajuda o treino. Isso vem do
Teorema de Kolmogorov-Arnold (toda função contínua multivariada é soma de funções de uma variável). Vantagem para o
projeto: cada $\varphi$ pode ser **desenhada** — é a explicabilidade nativa ("funções spline da KAN"). Custo: cada aresta
guarda `grid+ordem` coeficientes, não um escalar. Como a KAN fica só no final (entrada = `h(t)`, 128 números), o custo é
desprezível frente ao CNN.

As B-splines são calculadas pela **recursão de Cox-de Boor**: o grau 0 é um "degrau" (1 dentro do intervalo entre dois nós,
0 fora) e cada grau seguinte mistura dois vizinhos do grau anterior. A soma de todas as $B_m(x)$ é 1 dentro do grid
(partição da unidade) — cada ponto é uma média ponderada de ~4 coeficientes vizinhos.

In [ ]:
class CodificadorCNN(nn.Module):
    """CNN encoder: (N, 4, 64, 64) → vetor latente (N, dim_embedding). Estilo CNN-TC (stride 2, sem pooling)."""

    def __init__(self, canais_entrada: int = 4, canais=(32, 64, 128, 256), dim_embedding: int = 128,
                 lado_entrada: int = 64, dropout: float = 0.1):
        super().__init__()
        blocos, c_in = [], canais_entrada
        for c in canais:
            blocos += [nn.Conv2d(c_in, c, kernel_size=3, stride=2, padding=1, bias=False),
                       nn.BatchNorm2d(c), nn.ReLU()]
            c_in = c
        self.convs = nn.Sequential(*blocos)               # o último bloco (ReLU) é o alvo do Grad-CAM
        lado = lado_entrada // (2 ** len(canais))          # 64 → 4
        self.fc = nn.Sequential(nn.Flatten(), nn.Dropout(dropout),
                                nn.Linear(c_in * lado * lado, dim_embedding),
                                nn.LayerNorm(dim_embedding))      # embedding com escala ~1 → portões da LSTM não saturam

    def forward(self, x):
        return self.fc(self.convs(x))


class CamadaKAN(nn.Module):
    """Uma camada Kolmogorov-Arnold: y_j = Σ_i φ_{j,i}(x_i), com φ(x) = w_b·silu(x) + Σ_m c_m·B_m(x).

    Parâmetros: `peso_base` (n_out, n_in) e `coef` (n_out, n_in, grid+ordem) — uma curva por aresta (j, i).
    O grid é igual para todas as entradas: `grid` intervalos em `faixa`, estendido `ordem` nós para cada lado.
    """

    def __init__(self, n_in: int, n_out: int, grid: int = 5, ordem: int = 3, faixa=(-1.0, 1.0)):
        super().__init__()
        self.n_in, self.n_out, self.grid, self.ordem = n_in, n_out, grid, ordem
        h = (faixa[1] - faixa[0]) / grid
        nos = torch.arange(-ordem, grid + ordem + 1, dtype=torch.float32) * h + faixa[0]   # grid + 2·ordem + 1 nós
        self.register_buffer("nos", nos)
        self.peso_base = nn.Parameter(torch.empty(n_out, n_in))
        self.coef = nn.Parameter(torch.empty(n_out, n_in, grid + ordem))
        nn.init.kaiming_uniform_(self.peso_base, a=math.sqrt(5))          # mesma inicialização do nn.Linear
        nn.init.normal_(self.coef, mean=0.0, std=0.1 / math.sqrt(n_in))   # splines começam quase planas (função ≈ caminho base)

    def bases(self, x: torch.Tensor) -> torch.Tensor:
        """B-splines B_m(x) por Cox-de Boor. x: (..., n_in) → (..., n_in, grid+ordem)."""
        t = self.nos
        x = x.unsqueeze(-1)
        b = ((x >= t[:-1]) & (x < t[1:])).to(x.dtype)               # grau 0: degraus (1 só no intervalo que contém x)
        for k in range(1, self.ordem + 1):                          # cada grau mistura dois vizinhos do grau anterior
            b = ((x - t[:-(k + 1)]) / (t[k:-1] - t[:-(k + 1)])) * b[..., :-1] \
                + ((t[k + 1:] - x) / (t[k + 1:] - t[1:-k])) * b[..., 1:]
        return b

    def forward(self, x: torch.Tensor) -> torch.Tensor:              # x: (B, n_in) → (B, n_out)
        base = F.linear(F.silu(x), self.peso_base)                   # Σ_i w_b[j,i]·silu(x_i)
        spline = F.linear(self.bases(x).flatten(1), self.coef.flatten(1))   # Σ_i Σ_m c[j,i,m]·B_m(x_i)
        return base + spline

    # ---- utilidades de explicabilidade (seção 9) ------------------------------------------------
    @torch.no_grad()
    def valores_arestas(self, x: torch.Tensor) -> torch.Tensor:
        """φ_{j,i}(x_i) de TODAS as arestas para cada amostra: (N, n_out, n_in)."""
        base = F.silu(x).unsqueeze(1) * self.peso_base.unsqueeze(0)
        spline = torch.einsum("nim,oim->noi", self.bases(x), self.coef)
        return base + spline

    @torch.no_grad()
    def curva(self, j: int, i: int, xs: torch.Tensor) -> torch.Tensor:
        """A função aprendida φ_{j,i} avaliada em uma grade 1-D `xs` (para desenhar a spline da aresta i→j)."""
        spline = self.bases(xs.view(-1, 1))[:, 0, :] @ self.coef[j, i]
        return F.silu(xs) * self.peso_base[j, i] + spline


class CabecaKAN(nn.Module):
    """Cabeça de saída feita de camadas KAN: [dim_in → ocultas… → 1]. Camadas mais fundas recebem entradas não limitadas,
    então usam um grid mais largo; a primeira recebe h(t) ∈ (−1, 1)."""

    def __init__(self, dim_in: int, ocultas=(16,), grid: int = 5, ordem: int = 3):
        super().__init__()
        dims = [dim_in, *ocultas, 1]
        self.camadas = nn.ModuleList(
            CamadaKAN(a, b, grid, ordem, faixa=(-1.0, 1.0) if i == 0 else (-3.0, 3.0))
            for i, (a, b) in enumerate(zip(dims[:-1], dims[1:])))

    def forward(self, h):
        for camada in self.camadas:
            h = camada(h)
        return h


class CabecaMLP(nn.Module):
    """Cabeça MLP (ablação "sem KAN"): [dim_in → ocultas… → 1] com ReLU."""

    def __init__(self, dim_in: int, ocultas=(64,)):
        super().__init__()
        dims, camadas = [dim_in, *ocultas], []
        for a, b in zip(dims[:-1], dims[1:]):
            camadas += [nn.Linear(a, b), nn.ReLU()]
        camadas.append(nn.Linear(dims[-1], 1))
        self.rede = nn.Sequential(*camadas)

    def forward(self, h):
        return self.rede(h)


class ModeloTC(nn.Module):
    """CNN encoder (compartilhado) → LSTM → cabeças multitarefa.

    forward(x): x (B, T, 4, H, W) → ({tarefa: (B,)}, h(t) (B, dim_oculta))
    """

    def __init__(self, cfg: Config, variante: str):
        super().__init__()
        v = VARIANTES[variante]
        self.temporal, self.tarefas = v["temporal"], tuple(v["tarefas"])
        self.encoder = CodificadorCNN(4, cfg.canais_cnn, cfg.dim_embedding, cfg.recorte_final, cfg.dropout)
        if self.temporal == "lstm":
            self.lstm = nn.LSTM(cfg.dim_embedding, cfg.dim_oculta, num_layers=1, batch_first=True)
        else:  # sem sequência: projeta o embedding do quadro atual p/ o mesmo espaço (−1, 1) do estado da LSTM
            self.proj = nn.Linear(cfg.dim_embedding, cfg.dim_oculta)
        criar = (lambda: CabecaKAN(cfg.dim_oculta, cfg.kan_ocultas, cfg.kan_grid, cfg.kan_ordem)) \
            if v["cabeca"] == "kan" else (lambda: CabecaMLP(cfg.dim_oculta, cfg.mlp_ocultas))
        self.cabecas = nn.ModuleDict({t: criar() for t in self.tarefas})   # UMA cabeça independente por tarefa

    def forward(self, x: torch.Tensor):
        B, T = x.shape[:2]
        if self.temporal == "nenhum":
            x, T = x[:, -1:], 1                                           # só o quadro atual
        e = self.encoder(x.flatten(0, 1)).view(B, T, -1)                  # (B, T, D): MESMOS pesos em todos os quadros
        if self.temporal == "lstm":
            _, (h_n, _) = self.lstm(e)                                    # lê e(t-12h) … e(t) em ordem
            h = h_n[-1]                                                   # estado oculto final h(t)
        else:
            h = torch.tanh(self.proj(e[:, -1]))
        tipo = h.device.type if h.device.type in ("cuda", "cpu") else "cpu"
        with torch.autocast(device_type=tipo, enabled=False):             # cabeças sempre em float32 (splines são sensíveis)
            h = h.float()
            saidas = {t: cab(h).squeeze(-1) for t, cab in self.cabecas.items()}
        return saidas, h


def contar_parametros(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

### 4.4 Teste de sanidade do modelo
Confere (i) que as B-splines formam partição da unidade, (ii) as formas dos tensores de cada variante e (iii) que o
gradiente chega ao encoder (se o gradiente não chegasse, o Grad-CAM e o treino ponta-a-ponta não funcionariam).

In [ ]:
_kan = CamadaKAN(3, 2).to(device)
_x = torch.rand(1000, 3, device=device) * 1.98 - 0.99                # dentro do grid (−1, 1)
assert torch.allclose(_kan.bases(_x).sum(-1), torch.ones(1000, 3, device=device), atol=1e-5), "B-splines devem somar 1"
_va = _kan.valores_arestas(_x[:8])                                   # φ_{j,i}(x_i) de cada aresta
assert torch.allclose(_va.sum(-1), _kan(_x[:8]), atol=1e-5), "a saída do nó deve ser a soma das φ das arestas"
assert torch.allclose(_kan.curva(0, 1, _x[:8, 1]), _va[:, 0, 1], atol=1e-5), "curva() deve coincidir com a aresta (0←1)"

_xb = torch.randn(4, 5, 4, cfg.recorte_final, cfg.recorte_final, device=device)
linhas = []
for nome in VARIANTES:
    _m = ModeloTC(cfg, nome).to(device)
    _saidas, _h = _m(_xb)
    sum(s.sum() for s in _saidas.values()).backward()
    _g = _m.encoder.convs[0].weight.grad
    assert _g is not None and torch.isfinite(_g).all() and _g.abs().sum() > 0, "gradiente não chegou ao encoder"
    assert _h.abs().max() <= 1.0 + 1e-6, "h(t) deveria ficar em (−1, 1)"
    linhas.append(dict(variante=nome, tarefas="+".join(_m.tarefas), parametros=contar_parametros(_m),
                       saida_h=tuple(_h.shape), saidas={k: tuple(v.shape) for k, v in _saidas.items()}))
display(pd.DataFrame(linhas))
del _m, _xb, _saidas, _h

## 5. Treino e avaliação de UMA execução (variante × seed)

### 5.1 Divisão dos dados: treino × validação × teste (sem folds)
Janelas do mesmo ciclone são quase a mesma imagem (mesma tempestade, janelas sobrepostas em 4 dos 5 quadros). Se as janelas fossem
sorteadas soltas, vizinhas cairiam em lados opostos (treino × teste) e o modelo seria "testado" em imagens que já viu — **vazamento de
dados** que infla o desempenho. Por isso a divisão **nunca separa um ciclone**: todas as janelas dele ficam do mesmo lado.

* `DIVISAO = "ano"` (padrão, **como no TCC**): treino = ciclones de 2003–2014; validação = 2015–2016 (só para o *early stopping*: parar quando o
  erro de validação deixa de melhorar e ficar com os melhores pesos); teste = 2017. Usa o dataset inteiro, é a mesma divisão do TCC (o número do
  teste é comparável ao dele) e respeita o tempo: treina no passado e testa num ano que o modelo nunca viu. O preço é que o teste é um conjunto
  só (≈ 94 ciclones), então o número tem incerteza maior que a média de 5 folds da pipeline completa.
* `DIVISAO = "ciclone"`: sorteia ciclones inteiros (`frac_teste`, `frac_validacao`; `seed_estrutural` fixa o sorteio). Não respeita o tempo, mas mistura
  todos os anos nos três conjuntos.

A divisão é **a mesma para todas as variantes e seeds** (não depende da seed do treino), então todas são testadas exatamente nas mesmas janelas
(comparação pareada). A validação NÃO é usada para nada além de escolher a época; o teste só é previsto depois do treino.

In [ ]:
@dataclass
class Particao:
    """Índices (de janelas) de treino, validação (early stopping) e teste. Nenhum ciclone aparece em mais de um dos três."""
    treino: np.ndarray
    val: np.ndarray
    teste: np.ndarray
    descricao: str


def dividir_treino_val_teste(janelas: pd.DataFrame, cfg: Config) -> Particao:
    """Divide as janelas em treino / validação / teste SEM separar nenhum ciclone (todas as janelas de um ciclone ficam do mesmo lado).
    divisao="ano": pelo ano do ciclone (treino ≤ ano_fim_treino; validação até ano_fim_validacao; teste depois), como no TCC.
    divisao="ciclone": sorteia ciclones inteiros (frac_teste e frac_validacao dos ciclones; o resto é treino) com `seed_estrutural`."""
    ano, ids = janelas["ano"].to_numpy(), janelas["id_ciclone"].to_numpy()
    papel = np.zeros(len(janelas), dtype=np.int8)                       # 0 = treino, 1 = validação, 2 = teste
    if cfg.divisao == "ano":
        papel[ano > cfg.ano_fim_treino] = 1
        papel[ano > cfg.ano_fim_validacao] = 2
        descricao = (f"por ano (como no TCC): treino ≤ {cfg.ano_fim_treino}, validação {cfg.ano_fim_treino + 1}–{cfg.ano_fim_validacao}, "
                     f"teste ≥ {cfg.ano_fim_validacao + 1}")
    elif cfg.divisao == "ciclone":
        unicos = np.random.default_rng(cfg.seed_estrutural).permutation(np.unique(ids))
        n_te = max(1, int(round(len(unicos) * cfg.frac_teste)))
        n_val = max(1, int(round(len(unicos) * cfg.frac_validacao)))
        papel[np.isin(ids, unicos[:n_te])] = 2
        papel[np.isin(ids, unicos[n_te:n_te + n_val])] = 1
        descricao = (f"ciclones sorteados (seed_estrutural = {cfg.seed_estrutural}): {100 * (1 - cfg.frac_teste - cfg.frac_validacao):.0f}% treino, "
                     f"{100 * cfg.frac_validacao:.0f}% validação, {100 * cfg.frac_teste:.0f}% teste")
    else:
        raise ValueError(f'cfg.divisao deve ser "ano" ou "ciclone" (veio {cfg.divisao!r})')
    p = Particao(np.flatnonzero(papel == 0), np.flatnonzero(papel == 1), np.flatnonzero(papel == 2), descricao)
    for nome, idx in (("treino", p.treino), ("validação", p.val), ("teste", p.teste)):
        assert len(idx) > 0, f"a divisão deixou o conjunto de {nome} vazio ({descricao}): confira os anos/frações e os dados"
    return p


particao = dividir_treino_val_teste(janelas, cfg)
# garantia anti-vazamento: nenhum ciclone aparece em mais de um conjunto
_ids_conjunto = [set(janelas["id_ciclone"].to_numpy()[i]) for i in (particao.treino, particao.val, particao.teste)]
assert not (_ids_conjunto[0] & _ids_conjunto[1]) and not (_ids_conjunto[0] & _ids_conjunto[2]) and not (_ids_conjunto[1] & _ids_conjunto[2]), \
    "ciclone dividido entre treino, validação e teste!"
tab_particao = pd.DataFrame({
    nome: dict(ciclones=len(ids_), janelas=len(idx), anos=f"{janelas['ano'].to_numpy()[idx].min()}–{janelas['ano'].to_numpy()[idx].max()}",
               vmax_medio=round(float(janelas["vmax"].to_numpy()[idx].mean()), 1), com_dvmax=int(janelas["dvmax"].notna().to_numpy()[idx].sum()),
               ri=int((janelas["dvmax"].to_numpy()[idx] >= cfg.limiar_ri_nos).sum()))
    for nome, idx, ids_ in (("treino", particao.treino, _ids_conjunto[0]), ("validação", particao.val, _ids_conjunto[1]),
                            ("teste", particao.teste, _ids_conjunto[2]))}).T
print("divisão dos dados —", particao.descricao)
display(tab_particao)

### 5.2 Perda multitarefa e métricas

Cada tarefa tem sua **cabeça** e sua **perda** (erro quadrático médio sobre alvos padronizados); a perda total é a soma
ponderada $L=w_{V}\,\mathrm{MSE}_{V_{max}}+w_{\Delta}\,\mathrm{MSE}_{\Delta V_{max}}$ (pesos em `cfg.pesos_tarefas`; a tarefa principal pesa mais).
Como o tronco (CNN + LSTM) é compartilhado, os gradientes das duas tarefas se somam nele — é isso que o regulariza.

**Máscara:** janelas sem rótulo em `t+24h` (o ciclone acabou antes) têm `ΔVmax = NaN`; a perda de `dvmax` simplesmente as
ignora, mas elas continuam treinando a tarefa `vmax`. Os alvos são **padronizados** (média 0, desvio 1, do treino) para que
as duas perdas tenham escalas comparáveis; na avaliação voltamos para nós.

**Métricas (em nós):** RMSE (a mesma do TCC e da literatura), MAE e viés (erro médio com sinal). Para a intensificação rápida
(RI), além do erro de regressão, tratamos "ΔVmax ≥ 30 nós" como evento e medimos POD (acertos/eventos reais), FAR (alarmes
falsos/alarmes) e CSI. Como o MSE encolhe as previsões para a média (quase nunca sai "+30"), o RI é medido de três jeitos:
com o limiar ingênuo de 30 nós; com um **limiar ajustado na validação interna** (sufixo `_aj`; nunca olha o teste); e **sem
limiar nenhum** (AUC-ROC e precisão média do ΔVmax previsto usado como escore de RI).

In [ ]:
def metricas_regressao(y: np.ndarray, p: np.ndarray) -> dict:
    ok = ~(np.isnan(y) | np.isnan(p))
    if not ok.any():
        return dict(rmse=np.nan, mae=np.nan, vies=np.nan, n=0)
    e = p[ok] - y[ok]
    return dict(rmse=float(np.sqrt(np.mean(e ** 2))), mae=float(np.mean(np.abs(e))), vies=float(np.mean(e)), n=int(ok.sum()))


def metricas_ri(y_dv: np.ndarray, p_dv: np.ndarray, limiar: float, limiar_prev: Optional[float] = None) -> dict:
    """Intensificação rápida como detecção de evento. O evento REAL é ΔVmax ≥ `limiar`; o alarme é ΔVmax_previsto ≥ `limiar_prev`
    (None = o mesmo `limiar`). Um `limiar_prev` menor que `limiar` compensa o encolhimento das previsões em direção à média."""
    ok = ~(np.isnan(y_dv) | np.isnan(p_dv))
    real, prev = y_dv[ok] >= limiar, p_dv[ok] >= (limiar if limiar_prev is None else limiar_prev)
    tp, fn, fp = int((real & prev).sum()), int((real & ~prev).sum()), int((~real & prev).sum())
    div = lambda a, b: float(a / b) if b > 0 else np.nan
    return dict(pod=div(tp, tp + fn), far=div(fp, tp + fp), csi=div(tp, tp + fn + fp), eventos=int(real.sum()), alarmes=int(prev.sum()))


def auc_roc(real: np.ndarray, escore: np.ndarray) -> float:
    """Área sob a curva ROC = probabilidade de um evento real receber escore maior que um não-evento (empate vale 1/2).
    0,5 = sem habilidade; 1 = perfeito. Não depende de limiar."""
    n1, n0 = int(real.sum()), int((~real).sum())
    if n1 == 0 or n0 == 0:
        return np.nan
    r = sps.rankdata(escore)
    return float((r[real].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def precisao_media(real: np.ndarray, escore: np.ndarray) -> float:
    """Precisão média (AP, área sob a curva precisão × revocação). Para eventos raros é mais informativa que a AUC-ROC:
    um previsor sem habilidade tem AP ≈ prevalência do evento (aqui ≈ 0,067)."""
    if real.sum() == 0:
        return np.nan
    r = real[np.argsort(-escore, kind="stable")]
    acertos = np.cumsum(r)
    return float((acertos / np.arange(1, len(r) + 1))[r].sum() / r.sum())


def limiar_ri_ajustado(y_val: np.ndarray, p_val: np.ndarray, limiar_ri: float) -> float:
    """Limiar do ΔVmax PREVISTO (5 … `limiar_ri` nós, passo 1) que maximiza o CSI de RI na validação interna. O evento real continua
    sendo ΔVmax ≥ `limiar_ri`. Escolhido SÓ com a validação (nunca olha o teste)."""
    ok = ~(np.isnan(y_val) | np.isnan(p_val))
    if not (y_val[ok] >= limiar_ri).any():
        return float(limiar_ri)                      # sem eventos na validação: não há o que ajustar
    grade = np.arange(min(5.0, limiar_ri), limiar_ri + 0.5, 1.0)
    csi = [metricas_ri(y_val, p_val, limiar_ri, limiar_prev=g)["csi"] for g in grade]
    return float(grade[int(np.argmax(np.nan_to_num(csi, nan=-1.0)))])


def metricas_ri_todas(y_dv, p_dv, y_val, p_val, limiar_ri: float) -> dict:
    """RI de três jeitos: (1) alarme em ΔVmax_previsto ≥ limiar_ri (o 'ingênuo'); (2) alarme num limiar AJUSTADO na validação
    interna (sufixo _aj); (3) sem limiar nenhum: AUC-ROC e precisão média (AP) do ΔVmax previsto usado como 'escore' de RI."""
    r = metricas_ri(y_dv, p_dv, limiar_ri)
    out = dict(pod_ri=r["pod"], far_ri=r["far"], csi_ri=r["csi"])
    if y_val is not None and p_val is not None and np.isfinite(p_val).any():
        lim = limiar_ri_ajustado(y_val, p_val, limiar_ri)
        a = metricas_ri(y_dv, p_dv, limiar_ri, limiar_prev=lim)
        out.update(limiar_ri_aj=lim, pod_ri_aj=a["pod"], far_ri_aj=a["far"], csi_ri_aj=a["csi"])
    else:                                            # execução antiga (sem previsões de validação) ou variante sem ΔVmax
        out.update(limiar_ri_aj=np.nan, pod_ri_aj=np.nan, far_ri_aj=np.nan, csi_ri_aj=np.nan)
    ok = ~(np.isnan(y_dv) | np.isnan(p_dv))
    real = y_dv[ok] >= limiar_ri
    out.update(auc_ri=auc_roc(real, p_dv[ok]), ap_ri=precisao_media(real, p_dv[ok]))
    return out


def _autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=usar_amp) if device.type == "cuda" \
        else torch.autocast(device_type="cpu", enabled=False)


def _criar_scaler(ativo: bool):
    try:
        return torch.amp.GradScaler("cuda", enabled=ativo)
    except Exception:  # PyTorch mais antigo
        return torch.cuda.amp.GradScaler(enabled=ativo)


@torch.no_grad()
def prever(modelo: ModeloTC, dados: Dados, idx: np.ndarray, mu, sd, cfg: Config, norm: dict):
    """Predições (em nós) para as janelas `idx`, sem aumento de dados. Devolve {tarefa: array}."""
    modelo.eval()
    brutas = {t: [] for t in modelo.tarefas}
    for i in range(0, len(idx), cfg.tam_lote_aval):
        ib = torch.as_tensor(idx[i:i + cfg.tam_lote_aval], device=device)
        with _autocast():
            saidas, _ = modelo(montar_lote(dados, ib, mu, sd, cfg, treino=False))
        for t in brutas:
            brutas[t].append(saidas[t].float().cpu())
    return {t: torch.cat(v).numpy() * norm[t][1] + norm[t][0] for t, v in brutas.items()}


def perda_multitarefa(saidas: dict, yv_n: torch.Tensor, yd_n: torch.Tensor, pesos: dict) -> torch.Tensor:
    perda = torch.zeros((), device=yv_n.device)
    if "vmax" in saidas:
        perda = perda + pesos["vmax"] * F.mse_loss(saidas["vmax"], yv_n)
    if "dvmax" in saidas:
        m = ~torch.isnan(yd_n)                                    # só janelas com rótulo em t+Δ
        if m.any():
            perda = perda + pesos["dvmax"] * F.mse_loss(saidas["dvmax"][m], yd_n[m])
    return perda

### 5.3 O laço de treino
* **Otimizador:** AdamW (Adam com *weight decay* desacoplado) + agenda *OneCycle* (aquece a taxa de aprendizado e depois
  a reduz suavemente) e **corte de gradiente** — importante em redes recorrentes, onde o gradiente pode explodir.
* **Precisão mista (GPU CUDA):** CNN e LSTM em `float16` (mais rápido, menos memória); as **cabeças KAN ficam em `float32`**
  porque a recursão das B-splines é sensível.
* **Early stopping:** a cada época medimos o RMSE de Vmax na validação; guardamos os melhores pesos e paramos após
  `paciencia` épocas sem melhora. O critério é o mesmo (Vmax, a tarefa principal) para todas as variantes, o que mantém a
  seleção de modelo comparável entre elas.
* **Semente:** `fixar_semente(100·seed)`. A divisão dos dados **não** depende da seed — só a inicialização, a ordem
  dos lotes e o sorteio das rotações.

In [ ]:
def perda_padronizada(pred: dict, idx: np.ndarray, dados: Dados, norm: dict, pesos: dict) -> float:
    """A MESMA perda do treino (MSE dos alvos padronizados, ponderado por `pesos`), mas calculada sobre previsões já em nós (a saída de
    `prever`). Serve para medir a perda em modo de avaliação (sem dropout nem rotação) e comparar treino com validação."""
    total = 0.0
    for t, w in pesos.items():
        if t not in pred:
            continue
        y = dados.jan[t].to_numpy(np.float32)[idx]                    # as colunas "vmax" e "dvmax" da tabela de janelas são os rótulos
        ok = ~np.isnan(y)
        if ok.any():                                                  # erro padronizado = (previsto − real) / desvio-padrão do treino
            total += w * float(np.mean(((pred[t][ok].astype(np.float64) - y[ok]) / norm[t][1]) ** 2))
    return total


def diagnosticos_da_epoca(modelo: ModeloTC, dados: Dados, idx_tr_aval: Optional[np.ndarray], idx_val: np.ndarray, pv: dict,
                          mu, sd, cfg: Config, norm: dict, pesos: dict) -> dict:
    """Métricas extras de uma época, só para as curvas de treino (NÃO influenciam o treino nem o early stopping): perda e RMSE de ΔVmax
    na validação (já prevista em `pv`) e perda/RMSE do TREINO em modo de avaliação, num subconjunto fixo de janelas de treino."""
    nan = float("nan")
    y_dv_val = dados.jan["dvmax"].to_numpy(np.float32)[idx_val]
    out = dict(perda_val=perda_padronizada(pv, idx_val, dados, norm, pesos),
               rmse_val_dvmax=metricas_regressao(y_dv_val, pv["dvmax"])["rmse"] if "dvmax" in pv else nan,
               perda_treino_aval=nan, rmse_treino_vmax=nan, rmse_treino_dvmax=nan)
    if idx_tr_aval is not None:
        pt = prever(modelo, dados, idx_tr_aval, mu, sd, cfg, norm)    # o mesmo caminho da validação: sem dropout, sem rotação
        out["perda_treino_aval"] = perda_padronizada(pt, idx_tr_aval, dados, norm, pesos)
        out["rmse_treino_vmax"] = metricas_regressao(dados.jan["vmax"].to_numpy(np.float32)[idx_tr_aval], pt["vmax"])["rmse"]
        if "dvmax" in pt:
            out["rmse_treino_dvmax"] = metricas_regressao(dados.jan["dvmax"].to_numpy(np.float32)[idx_tr_aval], pt["dvmax"])["rmse"]
    return out


def treinar_execucao(variante: str, seed: int, dados: Dados, particao: Particao, cfg: Config, avaliar_teste: bool = True) -> dict:
    """Treina uma variante com uma seed e devolve as predições do conjunto de teste + histórico. `avaliar_teste=False` (usado no ajuste de
    hiperparâmetros) treina e mede só na validação: o teste nem é previsto, para nunca influenciar a escolha."""
    t0 = time.time()
    fixar_semente(100 * seed)
    idx_tr, idx_val, idx_te = particao.treino, particao.val, particao.teste

    # --- estatísticas calculadas SÓ no treino (nada da validação nem do teste vaza para a normalização)
    mu, sd = estatisticas_canais(dados, np.unique(dados.idx_np[idx_tr]), cfg, seed=cfg.seed_estrutural)
    ytr_v = dados.y_vmax[torch.as_tensor(idx_tr, device=device)]
    ytr_d = dados.y_dvmax[torch.as_tensor(idx_tr, device=device)]
    ytr_d = ytr_d[~torch.isnan(ytr_d)]
    norm = {"vmax": (float(ytr_v.mean()), float(ytr_v.std())),
            "dvmax": (float(ytr_d.mean()), float(ytr_d.std())) if len(ytr_d) > 1 else (0.0, 1.0)}
    yv_n = (dados.y_vmax - norm["vmax"][0]) / norm["vmax"][1]
    yd_n = (dados.y_dvmax - norm["dvmax"][0]) / norm["dvmax"][1]

    modelo = ModeloTC(cfg, variante).to(device)
    opt = torch.optim.AdamW(modelo.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    passos = max(len(idx_tr) // cfg.tam_lote, 1)
    agenda = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg.lr, total_steps=cfg.epocas_max * passos, pct_start=0.15)
    scaler = _criar_scaler(usar_amp)
    pesos = {t: cfg.pesos_tarefas[t] for t in modelo.tarefas}

    idx_tr_t = torch.as_tensor(idx_tr, device=device)
    y_val = dados.jan["vmax"].to_numpy(np.float32)[idx_val]
    # Diagnóstico das curvas (não muda o treino): subconjunto FIXO de janelas de treino para medir o erro de treino em MODO DE AVALIAÇÃO
    # (sem dropout nem rotação). O sorteio usa um gerador LOCAL: não consome o gerador global, então as seeds do treino não mudam.
    n_aval = min(len(idx_tr), cfg.n_janelas_aval_treino)
    idx_tr_aval = np.sort(np.random.default_rng(cfg.seed_estrutural + 2000).choice(idx_tr, size=n_aval, replace=False)) if n_aval > 0 else None
    historico, melhor, melhor_epoca, estado, sem_melhora = [], np.inf, -1, None, 0
    for ep in range(cfg.epocas_max):
        modelo.train()
        te = time.time()
        perm = idx_tr_t[torch.randperm(len(idx_tr_t), device=device)]
        soma = torch.zeros((), device=device)                      # acumula na GPU (evita sincronizar a cada passo)
        for i in range(passos):                                   # descarta o último lote incompleto (BatchNorm)
            ib = perm[i * cfg.tam_lote:(i + 1) * cfg.tam_lote]
            x = montar_lote(dados, ib, mu, sd, cfg, treino=True)  # rotação aleatória só aqui
            with _autocast():
                saidas, _ = modelo(x)
            perda = perda_multitarefa(saidas, yv_n[ib], yd_n[ib], pesos)   # soma ponderada das perdas das cabeças
            opt.zero_grad(set_to_none=True)
            scaler.scale(perda).backward()    # GradScaler: escala a perda p/ os gradientes em float16 não "zerarem" (no-op sem CUDA)
            scaler.unscale_(opt)              # desfaz a escala antes de cortar o gradiente
            nn.utils.clip_grad_norm_(modelo.parameters(), cfg.clip_grad)   # corte de gradiente: evita explosão na LSTM (BPTT)
            scaler.step(opt); scaler.update()
            agenda.step()                     # OneCycle: a taxa de aprendizado muda a CADA passo (sobe e depois desce)
            soma += perda.detach()
        pv = prever(modelo, dados, idx_val, mu, sd, cfg, norm)
        rmse_val = metricas_regressao(y_val, pv["vmax"])["rmse"]
        extra = diagnosticos_da_epoca(modelo, dados, idx_tr_aval, idx_val, pv, mu, sd, cfg, norm, pesos)   # só para as curvas
        historico.append(dict(epoca=ep, perda_treino=float(soma) / passos, rmse_val_vmax=rmse_val, lr=agenda.get_last_lr()[0],
                              segundos=time.time() - te, **extra))
        if cfg.log_por_epoca and (ep + 1) % cfg.log_por_epoca == 0:
            h = historico[-1]
            log(f"    época {ep + 1:2d}/{cfg.epocas_max} | perda treino {h['perda_treino']:.4f} val {h['perda_val']:.4f} | "
                f"RMSE Vmax treino(aval) {h['rmse_treino_vmax']:.2f} val {rmse_val:.2f} nós | {h['segundos']:.1f}s")
        if rmse_val < melhor - 1e-4:
            melhor, melhor_epoca, sem_melhora = rmse_val, ep, 0
            estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
        else:
            sem_melhora += 1
            if sem_melhora >= cfg.paciencia:
                break

    modelo.load_state_dict(estado)                                # volta aos melhores pesos
    if not avaliar_teste:
        idx_te = np.array([], dtype=np.int64)                      # ajuste de hiperparâmetros: o teste fica intocado
    pt = prever(modelo, dados, idx_te, mu, sd, cfg, norm) if len(idx_te) else {t: np.array([], dtype=np.float32) for t in modelo.tarefas}
    pv_final = prever(modelo, dados, idx_val, mu, sd, cfg, norm)     # validação com os MELHORES pesos (ajusta o limiar de RI)
    nan = np.full(len(idx_te), np.nan, dtype=np.float32)
    nan_val = np.full(len(idx_val), np.nan, dtype=np.float32)
    res = dict(
        idx_teste=idx_te, y_vmax=dados.jan["vmax"].to_numpy(np.float32)[idx_te], pred_vmax=pt["vmax"].astype(np.float32),
        y_dvmax=dados.jan["dvmax"].to_numpy(np.float32)[idx_te], pred_dvmax=pt.get("dvmax", nan).astype(np.float32),
        y_dvmax_val=dados.jan["dvmax"].to_numpy(np.float32)[idx_val], pred_dvmax_val=pv_final.get("dvmax", nan_val).astype(np.float32),
        meta=dict(variante=variante, seed=seed, divisao=particao.descricao, melhor_epoca=melhor_epoca, n_epocas=len(historico),
                  rmse_val=melhor, segundos=time.time() - t0, parametros=contar_parametros(modelo),
                  historico=historico, norm=norm, n_treino=len(idx_tr), n_val=len(idx_val), n_teste=len(idx_te)),
        estado=estado, mu=mu.cpu(), sd=sd.cpu())
    return res

## 6. Protocolo: seeds × variantes

### 6.1 Por que várias seeds?
Treinar uma rede neural é estocástico: pesos iniciais aleatórios, ordem aleatória dos lotes, rotações sorteadas e, em GPU, somas de ponto
flutuante feitas em paralelo em ordem não fixa. Mesmo com o mesmo código e os mesmos dados, **uma seed fixa não garante o mesmo resultado** (com
LSTM, a mesma seed na GPU não repete o treino: a RMSE muda em até ≈ 0,2 nó). Repetir o treino com várias seeds mede **quanto da diferença observada
entre dois modelos é efeito real e quanto é ruído do treino**: se um modelo ganha de forma consistente em todas as seeds, a evidência é muito mais forte.

**Nesta versão rápida** o plano usa as seeds do painel (`SEED`, por padrão uma só): 1 treino por variante. O que se perde é a medida do ruído do
treino (uma seed não o mostra); o que se ganha é tempo para iterar. Veja a nota "Como ler o resultado de 1 seed" no início.

### 6.2 Organização para o Kaggle (sessões longas)
* Cada execução é gravada **assim que termina** em `runs/` (predições do conjunto de teste, histórico, metadados).
* Se a sessão cair, basta rodar de novo: execuções já gravadas são **puladas**. Também dá para apontar
  `cfg.dirs_retomada` para a saída de uma sessão anterior (anexada como *input*).
* O nome do arquivo inclui uma **assinatura** dos hiperparâmetros, da divisão e das janelas: mudou a configuração → as execuções antigas
  não são reaproveitadas por engano. Só entram nela os campos que mudam o resultado (hiperparâmetros, dados, janelas, divisão); orçamento de
  tempo, cache e afins ficam de fora.
* A ordem é *seed → variante*: assim todas as variantes completam a mesma seed antes de passar à próxima, e as comparações pareadas já ficam
  possíveis mesmo se o tempo acabar no meio.

In [ ]:
# Campos que NÃO mudam o resultado de uma execução (onde/quanto rodar, estatística, orçamento de tempo, cache, tamanho do lote de
# AVALIAÇÃO e frequência do log): ficam fora da assinatura. Assim um treino feito em outra máquina (treinar_remoto.py) ou numa sessão
# com outro orçamento de tempo é reconhecido aqui como a mesma execução.
_CAMPOS_SO_DE_EXECUCAO = {"modo", "dir_dados", "dir_saida", "detectar_duplicatas", "tol_duplicata_graus", "dirs_retomada", "variantes",
                          "seeds", "variante_referencia", "alfa", "gradcam_amostras", "spline_topk",
                          "bootstrap_reps", "baseline_sem_imagens", "outlier_erro_nos",
                          "max_horas_treino", "usar_cache", "permitir_cpu", "tam_lote_aval", "log_por_epoca",
                          "nome_experimento", "notas", "n_janelas_aval_treino"}   # (estes 3: rótulo da rodada e diagnóstico das curvas)


def assinatura_experimento(cfg: Config, janelas: pd.DataFrame) -> str:
    """Hash curto dos hiperparâmetros que mudam o resultado + do conjunto de janelas."""
    d = {k: v for k, v in asdict(cfg).items() if k not in _CAMPOS_SO_DE_EXECUCAO}
    d["janelas"] = hashlib.md5("|".join(janelas["id_ciclone"] + "@" + janelas["instante"].astype(str)).encode()).hexdigest()
    return hashlib.md5(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:8]


ASSINATURA = assinatura_experimento(cfg, janelas)


def nome_run(variante: str, seed: int) -> str:
    return f"{variante}__s{seed}__{ASSINATURA}"


def salvar_execucao(res: dict, caminho: Path, guardar_modelo: bool) -> None:
    np.savez_compressed(caminho, idx_teste=res["idx_teste"], y_vmax=res["y_vmax"], pred_vmax=res["pred_vmax"],
                        y_dvmax=res["y_dvmax"], pred_dvmax=res["pred_dvmax"],
                        y_dvmax_val=res["y_dvmax_val"], pred_dvmax_val=res["pred_dvmax_val"],
                        meta=np.array(json.dumps(res["meta"], default=float)))
    if guardar_modelo:  # pesos para a explicabilidade (seção 9): só da primeira seed
        torch.save(dict(estado=res["estado"], mu=res["mu"], sd=res["sd"], meta=res["meta"]), caminho.with_suffix(".pt"))


def carregar_execucao(caminho: Path) -> dict:
    z = np.load(caminho, allow_pickle=False)
    out = {k: z[k] for k in ("idx_teste", "y_vmax", "pred_vmax", "y_dvmax", "pred_dvmax")}
    for k in ("y_dvmax_val", "pred_dvmax_val"):        # ausentes nas execuções gravadas por versões antigas do notebook
        out[k] = z[k] if k in z.files else None
    out["meta"] = json.loads(str(z["meta"]))
    out["variante"], out["seed"] = out["meta"]["variante"], out["meta"]["seed"]
    return out


def executar_protocolo(dados: Dados, particao: Particao, cfg: Config) -> int:
    """Treina o que falta do plano (seed × variante). Devolve quantas execuções ficaram pendentes (0 = completo)."""
    pasta = DIR_SAIDA / "runs"
    plano = [(s, v) for s in cfg.seeds for v in cfg.variantes]

    # retomada: traz para a pasta de saída as execuções já concluídas em sessões anteriores
    for d in cfg.dirs_retomada:
        for arq in Path(d).glob(f"runs/*__{ASSINATURA}.*"):
            if not (pasta / arq.name).exists():
                shutil.copy(arq, pasta / arq.name)
    pendentes = [(s, v) for s, v in plano if not (pasta / (nome_run(v, s) + ".npz")).exists()]
    log(f"plano: {len(plano)} execuções ({len(cfg.seeds)} seeds × {len(cfg.variantes)} variantes) | "
        f"já concluídas: {len(plano) - len(pendentes)} | pendentes: {len(pendentes)}")
    if len(plano) - len(pendentes):
        log(f"ATENÇÃO: {len(plano) - len(pendentes)} execuções já existem em runs/ (mesma assinatura) e serão REAPROVEITADAS, sem retreinar. Se você mudou o "
            "CÓDIGO do modelo ou do treino (e não só a configuração), a assinatura não muda: troque o NOME_EXPERIMENTO (ou apague runs/) para treinar de novo.")

    tempos, t_ini = [], time.time()
    orcamento = (cfg.max_horas_treino or 0) * 3600           # segundos; 0 = sem limite
    if orcamento:
        log(f"orçamento de tempo para novos treinos: {cfg.max_horas_treino:g} h (depois disso para e segue para as tabelas)")
    for s, v in pendentes:
        if orcamento and tempos and (time.time() - t_ini) + float(np.mean(tempos)) > orcamento:
            log(f"orçamento de tempo esgotado: {len(pendentes) - len(tempos)} execuções ficaram pendentes. Rode de novo (ou numa nova "
                f"sessão com cfg.dirs_retomada) para completá-las; as tabelas abaixo usam só as {len(plano) - len(pendentes) + len(tempos)} concluídas.")
            break
        caminho = pasta / (nome_run(v, s) + ".npz")
        res = treinar_execucao(v, s, dados, particao, cfg)
        salvar_execucao(res, caminho, guardar_modelo=(s == cfg.seeds[0]))
        tempos.append(res["meta"]["segundos"])
        mv = metricas_regressao(res["y_vmax"], res["pred_vmax"])
        md = metricas_regressao(res["y_dvmax"], res["pred_dvmax"])
        restante = np.mean(tempos) * (len(pendentes) - len(tempos))
        log(f"[{len(tempos)}/{len(pendentes)}] {v:18s} seed {s} | Vmax RMSE {mv['rmse']:.2f} MAE {mv['mae']:.2f} nós"
            + (f" | ΔVmax RMSE {md['rmse']:.2f}" if md["n"] else "")
            + f" | melhor época {res['meta']['melhor_epoca'] + 1}/{res['meta']['n_epocas']} | {formatar_tempo(res['meta']['segundos'])}"
            + f" | faltam ≈ {formatar_tempo(restante)}")
        del res
        if device.type == "cuda":
            torch.cuda.empty_cache()
    n_pend = len(pendentes) - len(tempos)
    log("protocolo concluído." if n_pend == 0 else f"protocolo INCOMPLETO: {n_pend} execuções pendentes.")
    return n_pend

In [ ]:
# [parar-antes-desta-celula]
# (o comentário acima é um marcador: `notebook_local.py` e `ajustar_hiperparametros.py` carregam as definições até aqui, sem treinar o plano)
n_pendentes = executar_protocolo(dados, particao, cfg)

### 6.3 Carregando os resultados
Lê todas as execuções (desta assinatura) e monta a tabela com uma linha por (variante, seed).

In [ ]:
def carregar_todas_execucoes(cfg: Config) -> List[dict]:
    """Execuções gravadas (desta assinatura) que fazem parte do PLANO desta rodada: variantes e seeds do painel. Execuções de
    outras seeds/variantes que estejam em runs/ (por exemplo, trazidas por `dirs_retomada`) ficam de fora das tabelas e figuras."""
    execs = [carregar_execucao(p) for p in sorted((DIR_SAIDA / "runs").glob(f"*__{ASSINATURA}.npz"))]
    return [e for e in execs if e["variante"] in cfg.variantes and e["seed"] in cfg.seeds]


execucoes = carregar_todas_execucoes(cfg)


def tabela_execucoes(execs: List[dict], cfg: Config) -> pd.DataFrame:
    linhas = []
    for e in execs:
        mv, md = metricas_regressao(e["y_vmax"], e["pred_vmax"]), metricas_regressao(e["y_dvmax"], e["pred_dvmax"])
        ri = metricas_ri_todas(e["y_dvmax"], e["pred_dvmax"], e.get("y_dvmax_val"), e.get("pred_dvmax_val"), cfg.limiar_ri_nos)
        linhas.append(dict(variante=e["variante"], seed=e["seed"],
                           rmse_vmax=mv["rmse"], mae_vmax=mv["mae"], vies_vmax=mv["vies"],
                           rmse_dvmax=md["rmse"], mae_dvmax=md["mae"], **ri,
                           melhor_epoca=e["meta"]["melhor_epoca"] + 1, n_epocas=e["meta"]["n_epocas"],
                           segundos=e["meta"]["segundos"], parametros=e["meta"]["parametros"]))
    return pd.DataFrame(linhas).sort_values(["variante", "seed"]).reset_index(drop=True)


df_exec = tabela_execucoes(execucoes, cfg)
df_exec.to_csv(DIR_SAIDA / "tabelas" / "metricas_por_execucao.csv", index=False)
print(f"{len(df_exec)} execuções carregadas ({df_exec['variante'].nunique()} variantes)")
display(df_exec.round(3).head(10))

## 7. Etapa 5 — Resultados e testes estatísticos

### 7.1 Desempenho por variante
Cada linha abaixo resume as execuções (seeds) de uma variante no **conjunto de teste**: **média ± desvio-padrão entre seeds** de RMSE, MAE
(nós) e dos indicadores de intensificação rápida. **Com 1 seed (o padrão desta versão rápida) não há desvio para mostrar**: aparece só o valor,
e o ruído de seed não aparece aqui (veja a nota sobre 1 seed no início).

In [ ]:
def resumo_por_variante(df: pd.DataFrame) -> pd.DataFrame:
    cols = ["rmse_vmax", "mae_vmax", "vies_vmax", "rmse_dvmax", "mae_dvmax", "pod_ri", "far_ri", "csi_ri"]
    ag = df.groupby("variante")[cols].agg(["mean", "std"])
    out = pd.DataFrame(index=ag.index)
    out["execuções"] = df.groupby("variante").size()
    for c in cols:
        out[c] = [("—" if np.isnan(m) else f"{m:.2f}" if np.isnan(s) else f"{m:.2f} ± {s:.2f}") for m, s in zip(ag[(c, "mean")], ag[(c, "std")])]
    out["parâmetros"] = df.groupby("variante")["parametros"].first()
    out["época média"] = df.groupby("variante")["melhor_epoca"].mean().round(1)
    out["tempo/execução"] = df.groupby("variante")["segundos"].mean().map(formatar_tempo)
    return out


tab_resumo = resumo_por_variante(df_exec)
tab_resumo.to_csv(DIR_SAIDA / "tabelas" / "resumo_por_variante.csv")
display(tab_resumo)

### 7.2 Referência: o baseline do TCC
Para situar os números: no TCC, treinando com 2003–2014 e testando em 2017 (N = 4296), o **CNN-TC** teve RMSE 10,06 / MAE 7,53
nós e o melhor modelo (**MobileNetV2 + SDTA**) teve RMSE 9,14 / MAE 7,00. Com `DIVISAO = "ano"` o protocolo daqui é **o mesmo** (treino 2003–2014,
teste 2017), então a comparação é direta, com duas ressalvas: aqui cada amostra é uma **janela de 5 quadros** que exige 7 quadros consecutivos
válidos (o teste tem ≈ 4 mil janelas, e não as 4296 imagens do TCC), e existe uma validação (2015–2016) para o *early stopping*. Com
`DIVISAO = "ciclone"` a comparação é só indicativa. A tabela mostra também as métricas **restritas às janelas de ciclones de 2017**.

In [ ]:
REF_TCC = pd.DataFrame({"modelo": ["CNN-TC (TCC)", "MobileNetV2+SDTA (TCC)"], "RMSE": [10.06, 9.14], "MAE": [7.53, 7.00]})


def predicoes_agrupadas(execs: List[dict], variante: str, seed: int) -> pd.DataFrame:
    """Predições de uma (variante, seed) no conjunto de teste: cada janela aparece uma vez (fora da amostra)."""
    partes = [pd.DataFrame({"janela": e["idx_teste"], "y_vmax": e["y_vmax"], "p_vmax": e["pred_vmax"],
                            "y_dvmax": e["y_dvmax"], "p_dvmax": e["pred_dvmax"]})
              for e in execs if e["variante"] == variante and e["seed"] == seed]
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()


linhas = []
for v in cfg.variantes:
    pg = predicoes_agrupadas(execucoes, v, cfg.seeds[0])
    if pg.empty:
        continue
    anos = janelas["ano"].to_numpy()[pg["janela"].to_numpy()]
    m_all, m_17 = metricas_regressao(pg["y_vmax"].to_numpy(), pg["p_vmax"].to_numpy()), \
        metricas_regressao(pg["y_vmax"].to_numpy()[anos == 2017], pg["p_vmax"].to_numpy()[anos == 2017])
    linhas.append(dict(variante=v, **{"RMSE (todas)": m_all["rmse"], "MAE (todas)": m_all["mae"], "N (todas)": m_all["n"],
                                       "RMSE (só 2017)": m_17["rmse"], "MAE (só 2017)": m_17["mae"], "N (2017)": m_17["n"]}))
if linhas:
    display(pd.DataFrame(linhas).round(2))
display(REF_TCC)

### 7.2b Baselines **sem imagens** para ΔVmax (e RI)

Antes de dizer que as imagens ajudam a prever a intensificação, é preciso saber **quanto dá para prever sem imagem nenhuma**. Três
previsores simples, ajustados no treino e avaliados no **mesmo teste** da rede:

| baseline | o que faz |
|---|---|
| `baseline_zero` | prevê ΔVmax = 0 (nada muda) |
| `baseline_media` | prevê a média do ΔVmax do treino |
| `baseline_historico` | regressão polinomial de grau 2 em Vmax(t) e na tendência recente Vmax(t) − Vmax(t−3h/6h/12h), do *best track* |

O último é o adversário de verdade: usa o **histórico de intensidade**, que a rede (só imagens) **não recebe**, mas que um previsor
operacional tem. Se a rede não o supera, o ΔVmax previsto por imagens + LSTM está só reconstruindo a tendência de intensidade, e o
argumento do artigo para essa cabeça precisa ser outro (por exemplo, a detecção de RI) — ou o histórico de Vmax deveria entrar como
entrada (decisão de projeto, não feita aqui).

**RI três vezes** (colunas da tabela): (i) `pod_ri / far_ri / csi_ri` com o limiar ingênuo (alarme se ΔVmax previsto ≥ 30 nós);
(ii) `*_aj`, com o limiar **ajustado na validação interna** (maximiza o CSI lá; nos baselines, no próprio treino); (iii) `auc_ri`
(AUC-ROC) e `ap_ri` (precisão média), que **não dependem de limiar** — um previsor sem habilidade tem AUC ≈ 0,50 e AP ≈ prevalência
de RI (≈ 0,07).

In [ ]:
def features_historico(jan: pd.DataFrame, serie: pd.DataFrame) -> np.ndarray:
    """(W, 4): Vmax(t) e as variações Vmax(t) − Vmax(t−3h), (t−6h) e (t−12h), todas do best track. NaN se faltar o quadro anterior
    (não acontece: toda janela tem 7 quadros consecutivos)."""
    s = serie.set_index(["id_ciclone", "instante"])["vmax"]
    v0 = jan["vmax"].to_numpy(float)
    cols = [v0]
    for h in (3, 6, 12):
        idx = pd.MultiIndex.from_arrays([jan["id_ciclone"].to_numpy(),
                                         (jan["instante"] - pd.to_timedelta(h, unit="h")).to_numpy()])
        cols.append(v0 - s.reindex(idx).to_numpy(float))
    return np.column_stack(cols)


def polinomio_grau2(Z: np.ndarray) -> np.ndarray:
    """[1, z_i, z_i·z_j (i ≤ j)]: uma regressão linear nesses termos é uma regressão polinomial de grau 2."""
    p = Z.shape[1]
    return np.column_stack([np.ones(len(Z))] + [Z[:, i] for i in range(p)] + [Z[:, i] * Z[:, j] for i in range(p) for j in range(i, p)])


def _execucao_baseline(nome: str, idx_te: np.ndarray, janelas: pd.DataFrame, p_te, y_tr, p_tr) -> dict:
    """Empacota as previsões de um baseline no mesmo formato das execuções da rede (para reaproveitar tabelas e testes)."""
    nan = np.full(len(idx_te), np.nan, dtype=np.float32)
    return dict(variante=nome, seed=0, idx_teste=idx_te,
                y_vmax=janelas["vmax"].to_numpy(np.float32)[idx_te], pred_vmax=nan,                  # só há baseline de ΔVmax
                y_dvmax=janelas["dvmax"].to_numpy(np.float32)[idx_te], pred_dvmax=np.asarray(p_te, np.float32),
                y_dvmax_val=np.asarray(y_tr, np.float32), pred_dvmax_val=np.asarray(p_tr, np.float32),   # o "ajuste" do limiar usa o próprio treino
                meta=dict(variante=nome, seed=0))


def baselines_dvmax(janelas: pd.DataFrame, serie: pd.DataFrame, particao: Particao, cfg: Config) -> Dict[str, List[dict]]:
    """Três previsores de ΔVmax SEM IMAGENS, ajustados no treino e avaliados no MESMO teste da rede:
       'baseline_zero'      — prevê 0 (sem mudança);
       'baseline_media'     — prevê a média do ΔVmax do treino;
       'baseline_historico' — regressão polinomial de grau 2 em Vmax(t) e na tendência recente (3, 6 e 12 h) do best track.
    O último usa informação que a rede de imagens NÃO recebe (o Vmax passado): é o 'adversário' que as imagens precisam bater."""
    X = features_historico(janelas, serie)
    y = janelas["dvmax"].to_numpy(float)
    tr = np.zeros(len(janelas), dtype=bool)
    tr[particao.treino] = True
    tr &= ~np.isnan(X).any(1) & ~np.isnan(y)
    te = particao.teste
    mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-9
    Phi = polinomio_grau2((X - mu) / sd)                                   # padroniza só com o treino (NaN onde faltar histórico)
    w, *_ = np.linalg.lstsq(Phi[tr], y[tr], rcond=None)
    prev = Phi @ w
    return {"baseline_zero": [_execucao_baseline("baseline_zero", te, janelas, np.zeros(len(te)), y[tr], np.zeros(tr.sum()))],
            "baseline_media": [_execucao_baseline("baseline_media", te, janelas, np.full(len(te), y[tr].mean()), y[tr], np.full(tr.sum(), y[tr].mean()))],
            "baseline_historico": [_execucao_baseline("baseline_historico", te, janelas, prev[te], y[tr], prev[tr])]}


def fmt_ms(x) -> str:
    """'média ± desvio' ignorando NaN ('—' se não há nenhum valor)."""
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return "—"
    return f"{x.mean():.2f} ± {x.std(ddof=1):.2f}" if len(x) > 1 else f"{x.mean():.2f}"


execucoes_baseline: List[dict] = []                    # só o baseline de histórico entra nos testes da seção 7.3b
df_base = pd.DataFrame()
resumo_baseline = None
if cfg.baseline_sem_imagens:
    _bl = baselines_dvmax(janelas, tabela_serie, particao, cfg)
    execucoes_baseline = _bl["baseline_historico"]
    linhas_b = []
    for nome, lista in _bl.items():
        for e in lista:
            md = metricas_regressao(e["y_dvmax"], e["pred_dvmax"])
            linhas_b.append(dict(variante=nome, rmse_dvmax=md["rmse"], mae_dvmax=md["mae"],
                                 **metricas_ri_todas(e["y_dvmax"], e["pred_dvmax"], e["y_dvmax_val"], e["pred_dvmax_val"], cfg.limiar_ri_nos)))
    df_base = pd.DataFrame(linhas_b)
    df_base.to_csv(DIR_SAIDA / "tabelas" / "baselines_dvmax.csv", index=False)

    # tabela única: redes (média ± dp entre seeds) e baselines (uma previsão só, avaliada no mesmo teste)
    cols_tab = ["rmse_dvmax", "mae_dvmax", "pod_ri", "far_ri", "csi_ri", "limiar_ri_aj", "pod_ri_aj", "far_ri_aj", "csi_ri_aj", "auc_ri", "ap_ri"]
    linhas_t = []
    for v in cfg.variantes:
        g = df_exec[df_exec["variante"] == v]
        if "dvmax" in VARIANTES[v]["tarefas"] and len(g):
            linhas_t.append({"modelo": v, "tipo": "rede (imagens)", **{c: fmt_ms(g[c]) for c in cols_tab}})
    for nome in df_base["variante"].unique():
        g = df_base[df_base["variante"] == nome]
        linhas_t.append({"modelo": nome, "tipo": "sem imagens", **{c: fmt_ms(g[c]) for c in cols_tab}})
    tab_dvmax = pd.DataFrame(linhas_t).set_index("modelo")
    tab_dvmax.to_csv(DIR_SAIDA / "tabelas" / "dvmax_modelos_vs_baselines.csv")
    prev_ri = float((janelas["dvmax"].dropna() >= cfg.limiar_ri_nos).mean())
    print(f"ΔVmax e RI — redes × baselines sem imagens (prevalência de RI = {100 * prev_ri:.1f}%: uma previsão sem habilidade tem "
          f"AUC ≈ 0,50 e AP ≈ {prev_ri:.2f})")
    display(tab_dvmax)
    ref_rmse = df_exec.loc[df_exec["variante"] == cfg.variante_referencia, "rmse_dvmax"].mean()
    h_rmse = df_base.loc[df_base["variante"] == "baseline_historico", "rmse_dvmax"].mean()
    resumo_baseline = dict(rmse_dvmax_referencia=float(ref_rmse), rmse_dvmax_baseline_historico=float(h_rmse),
                           rmse_dvmax_baseline_zero=float(df_base.loc[df_base["variante"] == "baseline_zero", "rmse_dvmax"].mean()))
    print(f"\nRMSE de ΔVmax: {cfg.variante_referencia} = {ref_rmse:.2f} nós | baseline de histórico (sem imagens) = {h_rmse:.2f} nós | "
          f"{'a rede NÃO bate' if ref_rmse >= h_rmse else 'a rede bate'} o baseline no teste "
          f"(a significância está na seção 7.3b).")
else:
    print("cfg.baseline_sem_imagens = False — baseline sem imagens desligado.")

### 7.3 Teste de Wilcoxon pareado + correção de Holm-Bonferroni

**Teste de Wilcoxon (de postos sinalizados) pareado.** Compara dois modelos nas **mesmas janelas de teste**: para cada janela
calcula a diferença entre os erros absolutos $d_i=|e^{A}_i|-|e^{B}_i|$, ordena $|d_i|$, soma os postos dos $d_i$ positivos e dos
negativos e pergunta: *se não houvesse diferença sistemática, seria plausível uma soma de postos tão desbalanceada?* É a
alternativa **não paramétrica** ao teste *t* pareado — apropriada porque os erros absolutos **não são normais** (Shapiro-Wilk
rejeitou normalidade no TCC) e porque os dados são pareados (mesma janela, dois modelos).

**Muitos testes ao mesmo tempo.** Aqui rodamos um teste para cada (comparação × tarefa × seed). Com α = 5% por teste, a
chance de **pelo menos um falso positivo** cresce rápido (com 20 testes independentes, ≈ 64%). A **correção de Holm-Bonferroni**
controla essa taxa (*family-wise error*): ordena os $m$ p-valores, $p_{(1)}\le\dots\le p_{(m)}$, compara o $k$-ésimo com
$\alpha/(m-k+1)$ e **para** no primeiro que não passa (rejeita só os anteriores). É menos conservadora que a Bonferroni simples
e igualmente rigorosa quanto ao erro. Implementamos a versão em *p-valores ajustados*:
$p^{adj}_{(k)}=\max_{j\le k}\min\{1,(m-j+1)\,p_{(j)}\}$ — significativo se $p^{adj}\le\alpha$.

**Dois níveis de comparação** (a variante de referência é sempre o modelo proposto):
1. **Por janela** (um teste por seed): compara os erros janela a janela no conjunto de teste. Tem muito poder, mas as janelas de um
   mesmo ciclone são correlacionadas (se parecem entre si), então os p-valores são *otimistas*: é só apoio.
2. **Principal — bootstrap por ciclone** (seção 7.3b): a unidade independente é o **ciclone**. Reamostram-se os ciclones do teste com
   reposição e recalcula-se a diferença de MAE/RMSE; o IC 95% e o p-valor vêm da distribuição dessas diferenças. Responde "a diferença
   se manteria com outros ciclones?".

Leitura: um resultado é convincente quando o IC 95% do bootstrap por ciclone **não inclui 0** (com p ajustado por Holm abaixo de α). Como
o teste é um conjunto só (por padrão, os ciclones de 2017), o IC é largo: é o preço de não usar folds. O bootstrap embute a variação entre
ciclones, **não** a entre seeds. (Os níveis «por execução» e «1 ponto por fold» da versão com folds saíram junto com os folds.)

In [ ]:
def holm_bonferroni(p: np.ndarray, alfa: float = 0.05):
    """Correção de Holm-Bonferroni (passo-descendente). Devolve (rejeita H0?, p-valor ajustado)."""
    p = np.asarray(p, dtype=float)
    m = len(p)
    ordem = np.argsort(p)
    ajustado, maximo = np.empty(m), 0.0
    for posto, i in enumerate(ordem):                      # posto 0 = menor p-valor
        maximo = max(maximo, (m - posto) * p[i])           # (m − k + 1)·p_(k), mantido monotônico
        ajustado[i] = min(1.0, maximo)
    return ajustado <= alfa, ajustado


def _wilcoxon(a: np.ndarray, b: np.ndarray) -> float:
    """p-valor bicaudal do Wilcoxon pareado (1.0 se não há nenhuma diferença)."""
    if not np.any(a != b):
        return 1.0
    return float(sps.wilcoxon(a, b, zero_method="wilcox", alternative="two-sided").pvalue)


def comparar_por_janela(execs: List[dict], cfg: Config) -> pd.DataFrame:
    """Wilcoxon pareado dos erros absolutos, janela a janela, em cada seed: referência × cada outra variante."""
    por_chave = {(e["variante"], e["seed"]): e for e in execs}
    ref, linhas = cfg.variante_referencia, []
    for outra in [v for v in cfg.variantes if v != ref]:
        for (v, s), a in por_chave.items():
            b = por_chave.get((outra, s))
            if v != ref or b is None:
                continue
            assert np.array_equal(a["idx_teste"], b["idx_teste"]), "as variantes devem ser avaliadas nas mesmas janelas"
            for tarefa in ("vmax", "dvmax"):
                y, pa, pb = a["y_" + tarefa], a["pred_" + tarefa], b["pred_" + tarefa]
                ok = ~(np.isnan(y) | np.isnan(pa) | np.isnan(pb))
                if ok.sum() < 20:
                    continue                                       # ex.: a variante só-Vmax não tem a tarefa dvmax
                ea, eb = np.abs(pa[ok] - y[ok]), np.abs(pb[ok] - y[ok])
                linhas.append(dict(comparacao=f"{ref} × {outra}", tarefa=tarefa, seed=s, n=int(ok.sum()),
                                   mae_ref=ea.mean(), mae_outra=eb.mean(), pct_ref_melhor=100 * float((ea < eb).mean()),
                                   p_bruto=_wilcoxon(ea, eb)))
    df = pd.DataFrame(linhas)
    if df.empty:
        return df
    df["significativo"], df["p_holm"] = holm_bonferroni(df["p_bruto"].to_numpy(), cfg.alfa)   # UMA família com todos os testes
    df["veredito"] = np.where(~df["significativo"], "sem diferença",
                              np.where(df["mae_ref"] < df["mae_outra"], "referência melhor", "referência pior"))
    return df


# sanidade: nossa correção de Holm confere com a do statsmodels (se estiver instalado)
try:
    from statsmodels.stats.multitest import multipletests
    _p = np.random.default_rng(0).uniform(0, 0.08, 40)
    assert np.allclose(holm_bonferroni(_p)[1], multipletests(_p, alpha=0.05, method="holm")[1]), "Holm diverge do statsmodels"
    print("Holm-Bonferroni confere com statsmodels.multipletests(method='holm')")
except ImportError:
    pass

if cfg.variante_referencia in set(df_exec["variante"]) and df_exec["variante"].nunique() > 1:
    testes_janela = comparar_por_janela(execucoes, cfg)
    testes_janela.to_csv(DIR_SAIDA / "tabelas" / "wilcoxon_por_janela.csv", index=False)
    if len(testes_janela):
        print(f"\n== Nível 1 — por janela: {len(testes_janela)} testes, UMA família de Holm-Bonferroni (α = {cfg.alfa}) ==")
        resumo_j = (testes_janela.groupby(["comparacao", "tarefa"])
                    .agg(testes=("p_bruto", "size"), ref_melhor=("veredito", lambda s: int((s == "referência melhor").sum())),
                         ref_pior=("veredito", lambda s: int((s == "referência pior").sum())),
                         sem_diferenca=("veredito", lambda s: int((s == "sem diferença").sum())),
                         mae_ref=("mae_ref", "mean"), mae_outra=("mae_outra", "mean"),
                         pct_janelas_ref_melhor=("pct_ref_melhor", "mean"), p_holm_mediano=("p_holm", "median")).round(4))
        display(resumo_j)
else:
    testes_janela = pd.DataFrame()
    print("Há só uma variante (ou falta a de referência): rode com ≥ 2 variantes para comparar entre variantes (o baseline de histórico segue na seção 7.3b).")

### 7.3b Teste principal: bootstrap por ciclone

**Por que este teste.** As janelas de um mesmo ciclone são quase a mesma imagem, então o número de observações realmente independentes é o de
**ciclones do teste** (≈ 94 em 2017), não o de janelas.

**Bootstrap por ciclone.** Para cada comparação (referência × outra variante, em Vmax e ΔVmax):
1. para cada janela de teste, toma-se o erro **médio entre as seeds** (as variantes são avaliadas nas mesmas janelas);
2. somam-se esses erros **por ciclone** (estatísticas suficientes: n, Σ|e|, Σe²);
3. repete-se `bootstrap_reps` vezes: sorteiam-se N ciclones **do teste** com reposição, recalculam-se MAE e RMSE de cada variante e a
   diferença (referência − outra);
4. o **IC 95%** é o intervalo central dessas diferenças e o **p-valor** bicaudal é 2·min(P(d ≥ 0), P(d ≤ 0)); aplica-se
   **Holm-Bonferroni** a todos os testes juntos.

Diferença **negativa** = a referência erra menos. O `baseline_historico` (seção 7.2b) também entra, só para ΔVmax. O RMSE aqui é a
raiz do erro quadrático médio entre as seeds (pode diferir milésimos da média dos RMSE por execução da seção 7.1). Com um teste só, o
intervalo é mais largo do que era com 5 folds (todos os ciclones testados): é a incerteza de verdade de avaliar em ≈ 94 ciclones.

In [ ]:
def erros_medios_por_janela(execs: List[dict], variante: str, n_jan: int) -> Dict[str, np.ndarray]:
    """Erro de cada janela de teste MÉDIO ENTRE AS SEEDS (cada janela é testada uma vez por seed). Devolve, para vmax e dvmax,
    |erro| e erro² como vetores (n_jan,) — NaN onde a janela não foi avaliada ou não tem rótulo/previsão."""
    soma = {k: np.zeros(n_jan) for k in ("abs_vmax", "sq_vmax", "abs_dvmax", "sq_dvmax")}
    cont = {t: np.zeros(n_jan) for t in ("vmax", "dvmax")}
    for e in execs:
        if e["variante"] != variante:
            continue
        for t in ("vmax", "dvmax"):
            err = e["pred_" + t].astype(float) - e["y_" + t].astype(float)
            ok = ~np.isnan(err)
            i = e["idx_teste"][ok]
            soma["abs_" + t][i] += np.abs(err[ok])
            soma["sq_" + t][i] += err[ok] ** 2
            cont[t][i] += 1
    return {k: np.where(cont[k.split("_")[1]] > 0, soma[k] / np.maximum(cont[k.split("_")[1]], 1), np.nan) for k in soma}


def comparar_bootstrap(execs: List[dict], ref: str, outras: List[str], ids_ciclone: np.ndarray, idx_teste: np.ndarray, cfg: Config) -> pd.DataFrame:
    """TESTE PRINCIPAL — bootstrap por CICLONE. A unidade independente é o ciclone (janelas do mesmo ciclone são quase a mesma
    imagem; as seeds avaliam as mesmas janelas). Reamostra os ciclones DO TESTE com reposição `cfg.bootstrap_reps` vezes e,
    em cada réplica, recalcula a diferença de MAE e de RMSE entre a referência e a outra variante (nas mesmas janelas, usando o erro
    médio entre seeds). Devolve a diferença observada, o IC 95% e o p-valor bicaudal (+ correção de Holm entre todos os testes)."""
    rng = np.random.default_rng(cfg.seed_estrutural)
    ids, inv_t = np.unique(ids_ciclone[idx_teste], return_inverse=True)                  # só os ciclones do TESTE entram no sorteio
    inv = np.full(len(ids_ciclone), -1, dtype=np.int64)
    inv[idx_teste] = inv_t
    n_c, reps = len(ids), int(cfg.bootstrap_reps)
    pesos = rng.multinomial(n_c, np.full(n_c, 1.0 / n_c), size=reps).astype(float)      # (reps, n_c): vezes que cada ciclone entra
    e_ref = erros_medios_por_janela(execs, ref, len(ids_ciclone))
    linhas = []
    for outra in outras:
        e_out = erros_medios_por_janela(execs, outra, len(ids_ciclone))
        for t in ("vmax", "dvmax"):
            ok = ~(np.isnan(e_ref["abs_" + t]) | np.isnan(e_out["abs_" + t])) & (inv >= 0)
            if ok.sum() < 20:
                continue                                                                   # ex.: a variante só-Vmax não tem ΔVmax
            # estatísticas suficientes por ciclone: [n, Σ|e|_ref, Σ|e|_outra, Σe²_ref, Σe²_outra]
            S = np.column_stack([np.bincount(inv[ok], weights=v, minlength=n_c) for v in
                                 (np.ones(int(ok.sum())), e_ref["abs_" + t][ok], e_out["abs_" + t][ok],
                                  e_ref["sq_" + t][ok], e_out["sq_" + t][ok])])
            difs = lambda T: dict(mae=(T[..., 1] - T[..., 2]) / T[..., 0],
                                  rmse=np.sqrt(T[..., 3] / T[..., 0]) - np.sqrt(T[..., 4] / T[..., 0]))
            obs, bs = difs(S.sum(0)), difs(pesos @ S)
            for m in ("mae", "rmse"):
                d = bs[m]
                lo, hi = np.percentile(d, [2.5, 97.5])
                p = min(1.0, 2.0 * (min(int((d >= 0).sum()), int((d <= 0).sum())) + 1) / (reps + 1))
                linhas.append(dict(comparacao=f"{ref} − {outra}", tarefa=t, metrica=m, ciclones=int(len(np.unique(inv[ok]))),
                                   janelas=int(ok.sum()), dif=float(obs[m]), ic95_inf=float(lo), ic95_sup=float(hi), p_boot=float(p)))
    df = pd.DataFrame(linhas)
    if df.empty:
        return df
    df["significativo"], df["p_holm"] = holm_bonferroni(df["p_boot"].to_numpy(), cfg.alfa)    # UMA família com todos os testes
    df["veredito"] = np.where(~df["significativo"], "sem diferença", np.where(df["dif"] < 0, "referência melhor", "referência pior"))
    return df


execs_todas = execucoes + execucoes_baseline
outras_vars = [v for v in cfg.variantes if v != cfg.variante_referencia] + (["baseline_historico"] if execucoes_baseline else [])
testes_boot = pd.DataFrame()
if cfg.variante_referencia in set(df_exec["variante"]) and outras_vars:
    testes_boot = comparar_bootstrap(execs_todas, cfg.variante_referencia, outras_vars, janelas["id_ciclone"].to_numpy(), particao.teste, cfg)
    testes_boot.to_csv(DIR_SAIDA / "tabelas" / "bootstrap_por_ciclone.csv", index=False)
    if len(testes_boot):
        print(f"== TESTE PRINCIPAL — bootstrap por ciclone ({cfg.bootstrap_reps} réplicas; Holm-Bonferroni sobre {len(testes_boot)} testes; "
              f"dif = referência − outra, negativo = referência erra menos; nós) ==")
        display(testes_boot.round(4))
else:
    print("Falta a variante de referência ou algo contra o que comparar.")

### 7.4 Figuras de desempenho
1. **Predito × real** (densidade; as janelas são muitas demais para pontos) — a diagonal é a previsão perfeita;
2. **RMSE de cada execução** (cada ponto é uma seed): mostra a dispersão devida às seeds;
3. **Curvas de validação** (RMSE de Vmax na validação por época);
4. **MAE por faixa de intensidade** — ciclones intensos são raros e mais difíceis.

In [ ]:
# A COR segue a VARIANTE (não a posição na lista): a mesma variante tem a mesma cor em todas as figuras. Azul = proposta; o baseline de
# histórico (sem imagens) é cinza. Uma variante nova que você acrescentar em VARIANTES ganha a próxima cor livre.
_PALETA = ["#2a78d6", "#eb6834", "#1baf7a", "#6250d6", "#e87ba4", "#008300", "#eda100", "#e34948"]
COR_VARIANTE = {"cnn_lstm_kan": _PALETA[0], "cnn_lstm_mlp": _PALETA[1], "cnn_kan_1frame": _PALETA[2], "cnn_lstm_kan_vmax": _PALETA[3]}
for _v in VARIANTES:
    COR_VARIANTE.setdefault(_v, next((c for c in _PALETA if c not in COR_VARIANTE.values()), "#555555"))
COR_VARIANTE["baseline_historico"] = "#898781"
cores = COR_VARIANTE
nv = len(cfg.variantes)

# (1) predito × real
fig, ax = plt.subplots(1, nv, figsize=(4.6 * nv, 4.4), squeeze=False, sharex=True, sharey=True)
for a, v in zip(ax[0], cfg.variantes):
    pg = predicoes_agrupadas(execucoes, v, cfg.seeds[0])
    if pg.empty:
        continue
    a.hexbin(pg["y_vmax"], pg["p_vmax"], gridsize=45, bins="log", cmap="viridis", mincnt=1)
    lim = [10, max(pg["y_vmax"].max(), pg["p_vmax"].max()) + 5]
    a.plot(lim, lim, "r--", lw=1); a.set_xlim(lim); a.set_ylim(lim)
    m = metricas_regressao(pg["y_vmax"].to_numpy(), pg["p_vmax"].to_numpy())
    a.set_title(f"{v}\nRMSE {m['rmse']:.2f} | MAE {m['mae']:.2f} nós | N={m['n']}", fontsize=10)
    a.set_xlabel("Vmax real (nós)")
ax[0, 0].set_ylabel("Vmax previsto (nós)")
plt.tight_layout(); plt.savefig(DIR_SAIDA / "figuras" / "02_predito_vs_real.png", dpi=110); plt.show()

# (2) RMSE por execução + (3) curvas de validação
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for k, (metrica, rot) in enumerate((("rmse_vmax", "RMSE de Vmax (nós)"), ("rmse_dvmax", "RMSE de ΔVmax (nós)"))):
    dados_box = [df_exec.loc[df_exec["variante"] == v, metrica].dropna().to_numpy() for v in cfg.variantes]
    com_dados = [(v, d) for v, d in zip(cfg.variantes, dados_box) if len(d)]
    if not com_dados:
        continue
    ax[k].boxplot([d for _, d in com_dados], showfliers=False)       # (sem o parâmetro `labels`: mudou entre versões do Matplotlib)
    ax[k].set_xticks(range(1, len(com_dados) + 1)); ax[k].set_xticklabels([v for v, _ in com_dados])
    for i, (v, d) in enumerate(com_dados, 1):
        ax[k].scatter(np.random.default_rng(i).normal(i, 0.05, len(d)), d, s=14, color=cores[v], alpha=0.7, zorder=3)
    ax[k].set_ylabel(rot); ax[k].tick_params(axis="x", rotation=15); ax[k].set_title("uma bolinha = uma seed")
for v in cfg.variantes:
    curvas = [pd.DataFrame(e["meta"]["historico"])["rmse_val_vmax"].to_numpy() for e in execucoes if e["variante"] == v]
    if not curvas:
        continue
    L = max(len(c) for c in curvas)
    M = np.array([np.pad(c, (0, L - len(c)), constant_values=np.nan) for c in curvas])
    ax[2].plot(np.arange(1, L + 1), np.nanmean(M, 0), color=cores[v], label=v)
    ax[2].fill_between(np.arange(1, L + 1), np.nanmean(M, 0) - np.nanstd(M, 0), np.nanmean(M, 0) + np.nanstd(M, 0), color=cores[v], alpha=0.15)
ax[2].set_xlabel("época"); ax[2].set_ylabel("RMSE de Vmax na validação (nós)"); ax[2].legend(fontsize=8)
ax[2].set_title("curvas de validação (média ± desvio entre seeds)")
plt.tight_layout(); plt.savefig(DIR_SAIDA / "figuras" / "03_rmse_e_curvas.png", dpi=110); plt.show()

# (4) MAE por faixa de intensidade
faixas = [(0, 34, "TD (<34)"), (34, 64, "TS (34–63)"), (64, 96, "Cat 1–2 (64–95)"), (96, 999, "Cat 3–5 (≥96)")]
fig, ax = plt.subplots(figsize=(8, 4))
larg = 0.8 / nv
for k, v in enumerate(cfg.variantes):
    pg = predicoes_agrupadas(execucoes, v, cfg.seeds[0])
    if pg.empty:
        continue
    maes = [np.abs(pg["p_vmax"] - pg["y_vmax"])[(pg["y_vmax"] >= lo) & (pg["y_vmax"] < hi)].mean() for lo, hi, _ in faixas]
    ax.bar(np.arange(len(faixas)) + k * larg, maes, larg, color=cores[v], label=v)
ax.set_xticks(np.arange(len(faixas)) + 0.4 - larg / 2); ax.set_xticklabels([f for _, _, f in faixas])
ax.set_ylabel("MAE de Vmax (nós)"); ax.legend(fontsize=8); ax.set_title("erro por faixa de intensidade (seed %d)" % cfg.seeds[0])
plt.tight_layout(); plt.savefig(DIR_SAIDA / "figuras" / "04_erro_por_intensidade.png", dpi=110); plt.show()

### 7.4b Curvas de treino (perda e RMSE, treino × validação)
O treino registra, **a cada época**, tudo o que é preciso para enxergar como a rede aprendeu. Cada variante ganha uma figura
(`10_curvas_treino_<variante>.png`) com uma coluna por seed e estas linhas:

| linha | treino (cinza) | validação (cor da variante) | o que mostra |
|---|---|---|---|
| **perda padronizada** | a perda que o otimizador vê, **com dropout e rotação** (por isso fica *acima* do que seria em avaliação) | a mesma perda em modo de avaliação | se a rede continua aprendendo e se a validação acompanha |
| **RMSE de Vmax (nós)** | RMSE em modo de avaliação (sem dropout nem rotação) numa amostra **fixa** de janelas de treino (`cfg.n_janelas_aval_treino`) | RMSE na validação | a **folga** entre as duas curvas é o sobreajuste, em nós |
| **RMSE de ΔVmax (nós)** | idem | idem | o mesmo para a tarefa de intensificação (variantes com ΔVmax) |

A linha vertical e o ponto marcam a **melhor época** (a que o *early stopping* guarda; é com esses pesos que o teste é previsto).
Para a parte final do treino ficar legível, o eixo y ignora as 2 primeiras épocas (a rede ainda está longe do ajuste): elas podem sair do quadro.
`11_curvas_resumo.png` compara as variantes (média entre seeds, faixa = mínimo e máximo entre seeds) e mostra a taxa de aprendizado (*OneCycle*).
As perdas **não** são comparáveis entre variantes (a variante só-Vmax não tem o termo de ΔVmax), por isso o resumo compara RMSE. A tabela
completa fica em `tabelas/historico_treino.csv`.

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

# ---- estilo comum das figuras novas (7.4b e 7.4c): fundo branco, grade fina e contínua, texto em tons neutros. A COR identifica a
#      variante (COR_VARIANTE, da seção 7.4) e o formato do marcador repete a identidade onde várias variantes se sobrepõem.
TINTA, TINTA2, TINTA_FRACA, GRADE, EIXO = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
MARCADOR_VARIANTE = {"cnn_lstm_kan": "o", "cnn_lstm_mlp": "s", "cnn_kan_1frame": "^", "cnn_lstm_kan_vmax": "D", "baseline_historico": "P"}


def estilo_figuras():
    """Contexto de estilo das figuras novas (use com `with estilo_figuras():`)."""
    return plt.rc_context({
        "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white", "axes.edgecolor": EIXO,
        "axes.labelcolor": TINTA2, "axes.titlecolor": TINTA, "axes.titlesize": 10, "axes.labelsize": 9, "axes.titlelocation": "left",
        "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRADE, "grid.linewidth": 0.8,
        "axes.axisbelow": True, "xtick.color": TINTA_FRACA, "ytick.color": TINTA_FRACA, "xtick.labelsize": 8, "ytick.labelsize": 8,
        "text.color": TINTA, "legend.fontsize": 8, "legend.frameon": False, "lines.linewidth": 1.8, "font.size": 9})


def salvar_e_mostrar(fig, nome: str, dpi: int = 110) -> None:
    fig.savefig(DIR_SAIDA / "figuras" / nome, dpi=dpi, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def tentar(nome: str, desenhar) -> None:
    """Roda a função que desenha uma figura. Se ela falhar, avisa e SEGUE: um erro numa figura de diagnóstico não pode impedir
    o relatório e o .zip no fim (o treino, que é o que custa horas, já está gravado)."""
    try:
        desenhar()
    except Exception as erro:
        import traceback
        log(f"ATENÇÃO: a figura «{nome}» falhou ({type(erro).__name__}: {erro}); a rodada continua.")
        traceback.print_exc(limit=4)
        plt.close("all")


# ---- histórico de treino de todas as execuções, em formato longo (uma linha por variante × seed × época)
COLUNAS_HIST = ["perda_treino", "perda_val", "perda_treino_aval", "rmse_treino_vmax", "rmse_val_vmax", "rmse_treino_dvmax",
                "rmse_val_dvmax", "lr", "segundos"]


def historico_longo(execs: List[dict]) -> pd.DataFrame:
    """Execuções gravadas por versões antigas do notebook não têm as colunas novas (perda_val, rmse_treino_*...): ficam NaN e as
    figuras pulam o que não existe."""
    partes = []
    for e in execs:
        h = pd.DataFrame(e["meta"]["historico"]).reindex(columns=["epoca"] + COLUNAS_HIST)
        h["epoca"] = h["epoca"] + 1                                       # 1ª época = 1
        h.insert(0, "melhor_epoca", e["meta"]["melhor_epoca"] + 1)
        for chave, valor in (("seed", e["seed"]), ("variante", e["variante"])):
            h.insert(0, chave, valor)
        partes.append(h)
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()


df_hist = historico_longo(execucoes)
df_hist.to_csv(DIR_SAIDA / "tabelas" / "historico_treino.csv", index=False)
print(f"histórico de treino: {len(df_hist)} linhas (variante × seed × época) → tabelas/historico_treino.csv")


def limites_y(valores: np.ndarray, epocas: np.ndarray, ep_ini: int = 3):
    """Limites do eixo y que ignoram as primeiras épocas (valores enormes): a parte final do treino fica legível."""
    v = valores[np.isfinite(valores)]
    if len(v) == 0:
        return None
    tardio = valores[np.isfinite(valores) & (epocas >= ep_ini)]
    lo, hi = float(v.min()), float(np.percentile(tardio if len(tardio) else v, 99))
    folga = 0.06 * max(hi - lo, 1e-6)
    return lo - folga, hi + folga


def desenhar_curvas_variante(v: str) -> None:
    g = df_hist[df_hist["variante"] == v]
    seeds, cor = sorted(g["seed"].unique()), COR_VARIANTE[v]
    linhas = [("perda padronizada", "perda_treino", "perda_val", "treino (com dropout e rotação)"),
              ("RMSE de Vmax (nós)", "rmse_treino_vmax", "rmse_val_vmax", "treino (modo avaliação)"),
              ("RMSE de ΔVmax (nós)", "rmse_treino_dvmax", "rmse_val_dvmax", "treino (modo avaliação)")]
    linhas = [ln for ln in linhas if g[ln[2]].notna().any()]               # pula o que a variante/execução não tem
    if g.empty or not linhas:
        print(f"{v}: sem histórico de treino para desenhar")
        return
    with estilo_figuras():
        fig, ax = plt.subplots(len(linhas), len(seeds), figsize=(4.6 * len(seeds) + 0.7, 2.4 * len(linhas) + 0.9), squeeze=False, sharex=True)
        for r, (rotulo, c_tr, c_va, nome_tr) in enumerate(linhas):
            lim = limites_y(g[[c_tr, c_va]].to_numpy(float).ravel(), np.repeat(g["epoca"].to_numpy(), 2))
            for k, s in enumerate(seeds):
                a = ax[r, k]
                h = g[g["seed"] == s].sort_values("epoca")
                if h.empty:
                    continue
                if h[c_tr].notna().any():
                    a.plot(h["epoca"], h[c_tr], color=TINTA_FRACA, lw=1.8)
                a.plot(h["epoca"], h[c_va], color=cor, lw=1.8)
                melhor = int(h["melhor_epoca"].iloc[0])
                a.axvline(melhor, color=EIXO, lw=0.8, zorder=1)
                y_melhor = h.loc[h["epoca"] == melhor, c_va]
                if len(y_melhor) and np.isfinite(y_melhor.iloc[0]):
                    a.plot([melhor], [y_melhor.iloc[0]], "o", color=cor, ms=6, mec="white", mew=1.5, zorder=5)
                if lim:
                    a.set_ylim(*lim)
                a.xaxis.set_major_locator(MaxNLocator(integer=True))                 # épocas são inteiras
                if r == 0:
                    a.set_title(f"seed {s}")
                if k == 0:
                    a.set_ylabel(rotulo)
                if r == len(linhas) - 1:
                    a.set_xlabel("época")
            itens_leg = ([Line2D([], [], color=TINTA_FRACA, lw=1.8, label=nome_tr)] if g[c_tr].notna().any() else []) + \
                        [Line2D([], [], color=cor, lw=1.8, label="validação")]
            ax[r, 0].legend(handles=itens_leg, loc="upper right")
        fig.suptitle(f"{v} — treino × validação; linha vertical e ponto = melhor época (early stopping)", y=0.995, fontsize=10)
        fig.tight_layout(rect=(0, 0, 1, 0.97))
    salvar_e_mostrar(fig, f"10_curvas_treino_{v}.png")


def desenhar_curvas_resumo() -> None:
    d = df_hist.assign(folga_vmax=df_hist["rmse_val_vmax"] - df_hist["rmse_treino_vmax"])
    paineis = [("rmse_val_vmax", "RMSE de Vmax na validação (nós)"), ("folga_vmax", "folga: validação − treino, RMSE de Vmax (nós)"),
               ("rmse_val_dvmax", "RMSE de ΔVmax na validação (nós)"), ("lr", "taxa de aprendizado (OneCycle)")]
    with estilo_figuras():
        fig, ax = plt.subplots(2, 2, figsize=(11.5, 7.2), sharex=True)
        for a, (col, titulo) in zip(ax.ravel(), paineis):
            a.set_title(titulo)
            if col == "lr":                                               # a taxa de aprendizado é a mesma em todas as execuções
                lr = d.groupby("epoca")["lr"].mean()
                a.plot(lr.index, lr.values, color=TINTA_FRACA)
                a.set_ylim(bottom=0)
                continue
            desenhou = False
            for v in cfg.variantes:
                g = d[d["variante"] == v]
                if not g[col].notna().any():
                    continue
                por_seed = g.groupby(["epoca", "seed"])[col].mean().unstack("seed")
                m, lo, hi = por_seed.mean(axis=1), por_seed.min(axis=1), por_seed.max(axis=1)   # só seeds ainda treinando naquela época
                a.fill_between(m.index, lo, hi, color=COR_VARIANTE[v], alpha=0.10, lw=0)
                a.plot(m.index, m.values, color=COR_VARIANTE[v], marker=MARCADOR_VARIANTE.get(v, "o"), ms=3.5, label=v)
                desenhou = True
            if not desenhou:                                                          # ex.: execuções antigas, sem as métricas novas
                a.text(0.5, 0.5, "sem dados nesta rodada", transform=a.transAxes, ha="center", va="center", color=TINTA_FRACA)
                a.axis("off")
                continue
            if col == "folga_vmax":
                a.axhline(0, color=EIXO, lw=1.0)
            lim = limites_y(d[col].to_numpy(float), d["epoca"].to_numpy())
            if lim:
                a.set_ylim(*lim)
        for a in ax.ravel():
            a.xaxis.set_major_locator(MaxNLocator(integer=True))
        for a in ax[1]:
            a.set_xlabel("época")
        ax[0, 0].legend(loc="upper right")
        fig.suptitle("Curvas de treino — média entre seeds (faixa = mínimo e máximo entre seeds)", y=0.995, fontsize=10)
        fig.tight_layout(rect=(0, 0, 1, 0.97))
    salvar_e_mostrar(fig, "11_curvas_resumo.png")


if len(df_hist):
    for _v in cfg.variantes:
        tentar(f"curvas de treino de {_v}", lambda v=_v: desenhar_curvas_variante(v))
    tentar("resumo das curvas de treino", desenhar_curvas_resumo)

### 7.4c Diagnósticos do teste: onde o modelo erra
Figuras feitas com as previsões do **conjunto de teste** (cada janela é prevista uma vez, por um modelo que não viu o ciclone dela; com
mais de uma seed, usa-se a primeira). Servem para saber **onde** uma mudança no modelo ajudou ou piorou, e não só quanto.

* `12_validacao_vs_teste.png` — RMSE de Vmax e de ΔVmax na **validação** (na melhor época, a que o *early stopping* escolheu) e no **teste**, por
  variante. Com a divisão por ano a validação (2015–2016) e o teste (2017) são anos diferentes: uma diferença grande entre as duas barras
  diz que o resultado depende do conjunto de ciclones, e não só do modelo (o teste é um conjunto só; não há folds para mostrar essa variação).
* `13_erro_por_categoria.png` — viés (previsto − real), RMSE e nº de janelas por faixa de intensidade (depressão a categoria 5), mais a distribuição do
  erro de Vmax. O modelo **regride à média**: superestima as fracas e subestima as intensas; um viés que encolhe nas categorias altas é sinal de ganho real.
* `14_dvmax_predito_vs_real.png` — ΔVmax previsto × real. A regressão por erro quadrático encolhe as previsões para perto de zero, e a
  intensificação rápida (≥ 30 nós em 24 h) é um evento raro; por isso há também as curvas abaixo.
* `15_ri_roc_pr.png` — curvas ROC e de precisão × revocação da intensificação rápida, usando o ΔVmax previsto como escore (sem fixar limiar),
  contra o baseline de histórico de Vmax (sem imagens). Linhas de referência: sorteio (ROC) e prevalência de RI (precisão × revocação).

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

CMAP_CONTAGEM = LinearSegmentedColormap.from_list("azul", ["#cde2fb", "#3987e5", "#0d366b"])   # uma cor só, do claro (pouco) ao escuro (muito)
BINS_INTENSIDADE = [(0, 34, "DT\n< 34"), (34, 64, "TS\n34–63"), (64, 83, "Cat 1\n64–82"), (83, 96, "Cat 2\n83–95"),
                    (96, 113, "Cat 3\n96–112"), (113, 137, "Cat 4\n113–136"), (137, 10_000, "Cat 5\n≥ 137")]


def pares_dvmax(nome: str):
    """(ΔVmax real, ΔVmax previsto) do conjunto de teste, sem NaN. Redes: 1ª seed do plano; baseline_historico: o baseline.
    Devolve None se não há previsão de ΔVmax (ex.: a variante só-Vmax)."""
    if nome == "baseline_historico":
        partes = [(e["y_dvmax"], e["pred_dvmax"]) for e in execucoes_baseline]
    else:
        partes = [(e["y_dvmax"], e["pred_dvmax"]) for e in execucoes if e["variante"] == nome and e["seed"] == cfg.seeds[0]]
    if not partes:
        return None
    y = np.concatenate([a for a, _ in partes]).astype(float)
    p = np.concatenate([b for _, b in partes]).astype(float)
    ok = ~(np.isnan(y) | np.isnan(p))
    return (y[ok], p[ok]) if ok.sum() > 20 else None


def desenhar_validacao_vs_teste() -> None:
    """RMSE na validação (na melhor época, a que o early stopping guarda) e no teste, por variante e por tarefa (média entre seeds)."""
    painel = (("Vmax", "rmse_val_vmax", "rmse_vmax"), ("ΔVmax", "rmse_val_dvmax", "rmse_dvmax"))
    melhor = df_hist[df_hist["epoca"] == df_hist["melhor_epoca"]]
    with estilo_figuras():
        fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
        for a, (tarefa, c_val, c_te) in zip(ax, painel):
            nomes = [v for v in cfg.variantes if df_exec.loc[df_exec["variante"] == v, c_te].notna().any()]
            val = [float(melhor.loc[melhor["variante"] == v, c_val].mean()) for v in nomes]
            te = [float(df_exec.loc[df_exec["variante"] == v, c_te].mean()) for v in nomes]
            cores_b = [COR_VARIANTE[v] for v in nomes]
            if tarefa == "ΔVmax" and len(df_base):                           # o baseline sem imagens só tem teste
                nomes.append("baseline_historico"); val.append(np.nan)
                te.append(float(df_base.loc[df_base["variante"] == "baseline_historico", c_te].mean())); cores_b.append(COR_VARIANTE["baseline_historico"])
            x = np.arange(len(nomes))
            a.bar(x - 0.2, val, 0.4, color="#c3c2b7", label="validação (melhor época)")
            a.bar(x + 0.2, te, 0.4, color=cores_b, label="teste")
            for xi, vv, tt in zip(x, val, te):
                for xx, yy in ((xi - 0.2, vv), (xi + 0.2, tt)):
                    if np.isfinite(yy):
                        a.text(xx, yy, f"{yy:.2f}", ha="center", va="bottom", fontsize=7.5, color=TINTA2)
            todos = [y for y in val + te if np.isfinite(y)]
            a.set_ylim(0, 1.12 * max(todos))                              # barras partem de zero (eixo cortado exageraria as diferenças)
            a.set_xticks(x); a.set_xticklabels(nomes, fontsize=8, rotation=10); a.grid(axis="x", visible=False)
            a.set_title(f"RMSE de {tarefa} (nós)"); a.legend(loc="upper left", fontsize=8)
        fig.tight_layout()
    salvar_e_mostrar(fig, "12_validacao_vs_teste.png")


def desenhar_erro_por_categoria() -> None:
    estat = {}
    for v in cfg.variantes:
        pg = predicoes_agrupadas(execucoes, v, cfg.seeds[0])
        if pg.empty:
            continue
        y, p = pg["y_vmax"].to_numpy(float), pg["p_vmax"].to_numpy(float)
        linhas = []
        for lo, hi, _ in BINS_INTENSIDADE:
            m = (y >= lo) & (y < hi)
            e = p[m] - y[m]
            linhas.append((float(e.mean()) if m.any() else np.nan, float(np.sqrt(np.mean(e ** 2))) if m.any() else np.nan, int(m.sum())))
        estat[v] = linhas
    if not estat:
        print("sem previsões de Vmax para desenhar")
        return
    x = np.arange(len(BINS_INTENSIDADE))
    nomes = [n for _, _, n in BINS_INTENSIDADE]
    bins_erro = np.arange(-40, 41, 2.0)
    with estilo_figuras():
        fig, ax = plt.subplots(2, 2, figsize=(12.5, 8.4))
        a_vies, a_rmse, a_hist, a_n = ax[0, 0], ax[0, 1], ax[1, 0], ax[1, 1]
        for v, linhas in estat.items():
            for a, k in ((a_vies, 0), (a_rmse, 1)):
                a.plot(x, [ln[k] for ln in linhas], color=COR_VARIANTE[v], marker=MARCADOR_VARIANTE.get(v, "o"), ms=5.5, mec="white", mew=1.0, label=v)
            pg = predicoes_agrupadas(execucoes, v, cfg.seeds[0])
            dens, _ = np.histogram((pg["p_vmax"] - pg["y_vmax"]).to_numpy(float), bins=bins_erro, density=True)
            a_hist.plot(0.5 * (bins_erro[1:] + bins_erro[:-1]), dens, color=COR_VARIANTE[v], lw=1.6)
        a_vies.axhline(0, color=EIXO, lw=1.0)
        a_vies.set_title("viés de Vmax por faixa: previsto − real (nós)"); a_rmse.set_title("RMSE de Vmax por faixa (nós)")
        a_hist.axvline(0, color=EIXO, lw=1.0)
        a_hist.set_title("distribuição do erro de Vmax (|erro| > 40 nós fica fora do quadro)"); a_hist.set_xlabel("previsto − real (nós)"); a_hist.set_ylabel("densidade")
        n_jan = [ln[2] for ln in next(iter(estat.values()))]
        a_n.bar(x, n_jan, width=0.55, color="#c3c2b7")
        for xi, ni in zip(x, n_jan):                                              # valor na ponta de cada barra
            a_n.text(xi, ni, f"{ni:,}".replace(",", "."), ha="center", va="bottom", fontsize=7.5, color=TINTA2)
        a_n.set_ylim(0, 1.12 * max(n_jan)); a_n.set_title("janelas de teste por faixa de intensidade"); a_n.grid(axis="x", visible=False)
        for a in (a_vies, a_rmse, a_n):
            a.set_xticks(x); a.set_xticklabels(nomes, fontsize=7.5)
        a_vies.legend(loc="lower left")
        fig.tight_layout()
    salvar_e_mostrar(fig, "13_erro_por_categoria.png")


def desenhar_dvmax_predito_vs_real() -> None:
    itens = [(n, pares_dvmax(n)) for n in list(cfg.variantes) + (["baseline_historico"] if execucoes_baseline else [])]
    itens = [(n, d) for n, d in itens if d is not None]
    if not itens:
        print("sem previsões de ΔVmax para desenhar")
        return
    with estilo_figuras():
        fig, ax = plt.subplots(1, len(itens), figsize=(4.4 * len(itens), 4.3), squeeze=False, sharex=True, sharey=True)
        for a, (n, (y, p)) in zip(ax[0], itens):
            a.hexbin(y, p, gridsize=40, bins="log", cmap=CMAP_CONTAGEM, mincnt=1, extent=(-80, 80, -80, 80), linewidths=0.2)
            a.plot([-80, 80], [-80, 80], color=TINTA_FRACA, lw=1.0)                            # previsão perfeita
            a.axvline(cfg.limiar_ri_nos, color=EIXO, lw=0.9); a.axhline(cfg.limiar_ri_nos, color=EIXO, lw=0.9)   # limiar de RI
            m = metricas_regressao(y, p)
            a.set_title(f"{n}\nRMSE {m['rmse']:.2f} | MAE {m['mae']:.2f} nós | N = {m['n']}", fontsize=9)
            a.set_xlabel("ΔVmax real (nós)"); a.set_xlim(-80, 80); a.set_ylim(-80, 80); a.grid(False)
        ax[0, 0].set_ylabel("ΔVmax previsto (nós)")
        fig.tight_layout()
    salvar_e_mostrar(fig, "14_dvmax_predito_vs_real.png")


def curva_roc_pr(real: np.ndarray, escore: np.ndarray, max_pontos: int = 1500):
    """Pontos das curvas ROC e de precisão × revocação, ordenando as janelas pelo escore (do maior para o menor)."""
    ordem = np.argsort(-escore, kind="stable")
    r = real[ordem]
    tp, fp = np.cumsum(r), np.cumsum(~r)
    P, N = int(r.sum()), int((~r).sum())
    sel = np.unique(np.linspace(0, len(r) - 1, min(max_pontos, len(r))).astype(int))      # poucos pontos bastam para desenhar
    fpr, tpr = np.r_[0.0, fp[sel] / N], np.r_[0.0, tp[sel] / P]
    return fpr, tpr, tp[sel] / P, tp[sel] / np.maximum(tp[sel] + fp[sel], 1)


def desenhar_ri_roc_pr() -> None:
    itens = [(n, pares_dvmax(n)) for n in list(cfg.variantes) + (["baseline_historico"] if execucoes_baseline else [])]
    itens = [(n, d) for n, d in itens if d is not None and (d[0] >= cfg.limiar_ri_nos).any()]
    if not itens:
        print("sem eventos de intensificação rápida nas janelas de teste (comum no modo teste): nada a desenhar")
        return
    prevalencia = float(np.mean(np.concatenate([d[0] for _, d in itens[:1]]) >= cfg.limiar_ri_nos))
    with estilo_figuras():
        fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.7))
        for n, (y, p) in itens:
            real = y >= cfg.limiar_ri_nos
            fpr, tpr, rev, prec = curva_roc_pr(real, p)
            rot = f"{n} (AUC {auc_roc(real, p):.3f} | AP {precisao_media(real, p):.3f})"
            kw = dict(color=COR_VARIANTE[n], lw=1.8)
            ax[0].plot(fpr, tpr, label=rot, **kw)
            ax[1].plot(rev, prec, label=rot, **kw)
        ax[0].plot([0, 1], [0, 1], color=EIXO, lw=1.0)                                        # sem habilidade
        ax[1].axhline(prevalencia, color=EIXO, lw=1.0)
        ax[0].set_title("ROC da intensificação rápida"); ax[0].set_xlabel("taxa de falsos alarmes"); ax[0].set_ylabel("taxa de acertos (POD)")
        ax[1].set_title("precisão × revocação"); ax[1].set_xlabel("revocação (POD)"); ax[1].set_ylabel("precisão")
        for a in ax:
            a.set_xlim(0, 1); a.set_ylim(0, 1.02)
        ax[0].legend(loc="lower right", fontsize=7.5)
        ax[1].text(0.98, prevalencia - 0.012, f"sem habilidade ≈ prevalência ({prevalencia:.3f})", ha="right", va="top", fontsize=7.5, color=TINTA2)
        fig.suptitle(f"RI = ΔVmax ≥ {cfg.limiar_ri_nos:.0f} nós em {cfg.horizonte_horas} h; escore = ΔVmax previsto; conjunto de teste", y=1.0, fontsize=10)
        fig.tight_layout()
    salvar_e_mostrar(fig, "15_ri_roc_pr.png")


tentar("validação × teste", desenhar_validacao_vs_teste)
tentar("erro por categoria de intensidade", desenhar_erro_por_categoria)
tentar("ΔVmax predito × real", desenhar_dvmax_predito_vs_real)
tentar("curvas ROC e PR da intensificação rápida", desenhar_ri_roc_pr)

### 7.5 Janelas com erro grande: o que são?

No gráfico predito × real da primeira execução completa apareceu uma **faixa horizontal**: janelas com Vmax real entre 40 e 140 nós
previstas em ~25 nós **pelos três modelos**. Se modelos diferentes erram as mesmas janelas do mesmo jeito, a causa mais provável está
na **entrada ou no rótulo**, e não na arquitetura. Esta seção lista essas janelas (`|erro| ≥ cfg.outlier_erro_nos`), mede quanto do
erro quadrático elas respondem por, compara as características da imagem (temperatura IR média/desvio/mínimo, fração de nuvens frias,
fração de IR ausente nos 7 quadros) com as das demais e desenha as piores. Tabela completa em
`tabelas/janelas_erro_grande_vmax.csv`.

Hipóteses para conferir: imagem quase constante/vazia (desvio de IR ≈ 0) apesar de ter < 10% de NaN; centro do ciclone fora do
recorte; rótulo de Vmax incompatível com a imagem (por exemplo, ciclone já extratropical).

In [ ]:
@torch.no_grad()
def estatisticas_imagem_janelas(dados: Dados, idx_jan: np.ndarray, cfg: Config, lote: int = 512) -> Dict[str, np.ndarray]:
    """Resumo do quadro t (recorte central 64×64) de cada janela: média, desvio e mínimo da temperatura IR, fração de pixels 'frios'
    (IR < 220 K, topos de nuvem altos) e média do PMW. Ignora pixels inválidos (NaN). Servem para perguntar: 'esta imagem é normal?'."""
    m0 = (cfg.recorte_armazenado - cfg.recorte_final) // 2
    m1 = m0 + cfg.recorte_final
    chaves = ("ir_media", "ir_dp", "ir_min", "frac_frio", "pmw_media")
    saida = {k: [] for k in chaves}
    for i in range(0, len(idx_jan), lote):
        ib = torch.as_tensor(idx_jan[i:i + lote], device=dados.quadros.device)
        q = dados.quadros[dados.idx[ib][:, -1]][:, :, m0:m1, m0:m1].float()          # (B, 2, 64, 64): IR e PMW do quadro t
        ir, pmw = q[:, 0], q[:, 1]
        v, vp = ~torch.isnan(ir), ~torch.isnan(pmw)
        n, npmw = v.sum((1, 2)).clamp_min(1), vp.sum((1, 2)).clamp_min(1)
        zero = torch.zeros_like(ir)
        media = torch.where(v, ir, zero).sum((1, 2)) / n
        dp = (torch.where(v, (ir - media.view(-1, 1, 1)) ** 2, zero).sum((1, 2)) / n).sqrt()
        minimo = torch.where(v, ir, torch.full_like(ir, float("inf"))).amin((1, 2))
        frio = ((ir < 220.0) & v).sum((1, 2)).float() / n
        sem_ir = v.sum((1, 2)) == 0                                                     # quadro todo inválido: sem estatística
        for k, x in zip(chaves, (media, dp, minimo, frio, torch.where(vp, pmw, torch.zeros_like(pmw)).sum((1, 2)) / npmw)):
            saida[k].append(torch.where(sem_ir, torch.full_like(x, float("nan")), x).cpu())
    return {k: torch.cat(v).numpy() for k, v in saida.items()}


def previsao_media_por_janela(execs: List[dict], variante: str, tarefa: str, n_jan: int):
    """Previsão da tarefa, média entre as seeds, de cada janela de teste (NaN onde a janela não foi avaliada)."""
    soma, cont = np.zeros(n_jan), np.zeros(n_jan)
    for e in execs:
        if e["variante"] == variante:
            p = e["pred_" + tarefa].astype(float)
            ok = ~np.isnan(p)
            soma[e["idx_teste"][ok]] += p[ok]
            cont[e["idx_teste"][ok]] += 1
    return np.where(cont > 0, soma / np.maximum(cont, 1), np.nan)


v_ref = cfg.variante_referencia
y_all = janelas["vmax"].to_numpy(float)
prev_ref = previsao_media_por_janela(execucoes, v_ref, "vmax", len(janelas))
erro_ref = prev_ref - y_all
avaliadas = np.flatnonzero(~np.isnan(erro_ref))
grande = avaliadas[np.abs(erro_ref[avaliadas]) >= cfg.outlier_erro_nos]
sq_tot = float(np.nansum(erro_ref ** 2))
print(f"{len(grande)} de {len(avaliadas)} janelas ({100 * len(grande) / max(len(avaliadas), 1):.2f}%) têm |erro de Vmax| ≥ "
      f"{cfg.outlier_erro_nos:.0f} nós em {v_ref}; elas respondem por {100 * float(np.sum(erro_ref[grande] ** 2)) / max(sq_tot, 1e-9):.1f}% "
      f"da soma dos erros quadráticos (RMSE sem elas: "
      f"{np.sqrt((sq_tot - float(np.sum(erro_ref[grande] ** 2))) / max(len(avaliadas) - len(grande), 1)):.2f} nós).")

if len(grande):
    img = estatisticas_imagem_janelas(dados, avaliadas, cfg)
    df_img = pd.DataFrame(img, index=avaliadas)
    df_img["frac_nan_ir_max"] = frac_invalido_ir[idx_quadros[avaliadas]].max(axis=1)       # pior quadro (entre os 7) em IR ausente
    df_img["vmax_real"] = y_all[avaliadas]
    df_img["grande"] = np.isin(avaliadas, grande)
    ordem_cols = ["vmax_real", "frac_nan_ir_max", "ir_media", "ir_dp", "ir_min", "frac_frio", "pmw_media"]
    comp = df_img.groupby("grande")[ordem_cols].median().rename(index={False: "demais janelas", True: "erro grande"})
    print("\nMediana das características da imagem (quadro t): janelas de erro grande × demais")
    display(comp.round(3))

    tab_g = pd.DataFrame({"id_ciclone": janelas["id_ciclone"].to_numpy()[grande], "instante": janelas["instante"].to_numpy()[grande],
                          "bacia": janelas["bacia"].to_numpy()[grande], "ano": janelas["ano"].to_numpy()[grande],
                          "vmax_real": y_all[grande], f"prev_{v_ref}": prev_ref[grande], "erro": erro_ref[grande]}, index=grande)
    for v in cfg.variantes:                                                                # o MESMO erro nas outras variantes?
        if v != v_ref:
            tab_g[f"erro_{v}"] = previsao_media_por_janela(execucoes, v, "vmax", len(janelas))[grande] - y_all[grande]
    tab_g = tab_g.join(df_img.loc[grande, ["frac_nan_ir_max", "ir_media", "ir_dp", "ir_min", "frac_frio", "pmw_media"]])
    tab_g = tab_g.reindex(tab_g["erro"].abs().sort_values(ascending=False).index)
    tab_g.to_csv(DIR_SAIDA / "tabelas" / "janelas_erro_grande_vmax.csv")
    outras_v = [c for c in tab_g.columns if c.startswith("erro_")]
    if outras_v:
        mesma = np.mean([(np.sign(tab_g[c]) == np.sign(tab_g["erro"])) & (tab_g[c].abs() >= cfg.outlier_erro_nos) for c in outras_v], axis=0)
        print(f"\nFração média das janelas de erro grande que as OUTRAS variantes também erram (≥ {cfg.outlier_erro_nos:.0f} nós, mesmo "
              f"sinal): {100 * float(np.mean(mesma)):.0f}%  → próxima de 100% sugere problema no dado/rótulo, não no modelo.")
    print("\nPor ano e por bacia (janelas de erro grande):")
    display(tab_g["ano"].value_counts().sort_index().to_frame("janelas").T)
    display(tab_g["bacia"].value_counts().to_frame("janelas").T)
    print("\nAs 15 piores janelas:")
    display(tab_g.head(15).round({c: 2 for c in tab_g.columns if pd.api.types.is_float_dtype(tab_g[c])}))

    # figura: IR do quadro t das piores janelas (pixels inválidos em vermelho)
    n_fig = min(8, len(tab_g))
    fig, ax = plt.subplots(2, 4, figsize=(13, 7), squeeze=False)
    cmap = plt.get_cmap("gray_r").copy(); cmap.set_bad("red")
    for a, (j, r) in zip(ax.ravel(), tab_g.head(n_fig).iterrows()):
        q = dados.quadros[int(dados.idx[int(j)][-1]), 0].float().cpu().numpy()
        a.imshow(q, cmap=cmap); a.axis("off")
        a.set_title(f"{r['id_ciclone']}  {pd.Timestamp(r['instante']):%Y-%m-%d %Hh}\nreal {r['vmax_real']:.0f} | previsto {r[f'prev_{v_ref}']:.0f} nós", fontsize=9)
    for a in ax.ravel()[n_fig:]:
        a.axis("off")
    fig.suptitle("IR (quadro t) das janelas de maior erro de Vmax — vermelho = pixel inválido", y=0.995)
    plt.tight_layout(); plt.savefig(DIR_SAIDA / "figuras" / "09_janelas_erro_grande.png", dpi=105); plt.show()
else:
    print("Nenhuma janela acima do limite — nada a diagnosticar (no modo teste isso é comum).")

## 8. Etapa 6 — Explicabilidade

Dois métodos **complementares**: o Grad-CAM responde **"onde"** na imagem o modelo olhou; as splines da KAN respondem
**"que relação"** o modelo aprendeu entre as variáveis internas e cada saída.

### 8.1 Grad-CAM no encoder
*Gradient-weighted Class Activation Mapping* (Selvaraju et al., 2017). Para uma saída (aqui, a previsão de Vmax ou de ΔVmax):

1. pegamos os **mapas de características** $A^k$ da última camada convolucional do encoder (256 mapas de 4×4);
2. calculamos o **gradiente** da saída em relação a cada mapa e tiramos a média espacial → um **peso de importância** por
   canal: $\alpha_k=\frac1{HW}\sum_{h,w}\frac{\partial y}{\partial A^k_{hw}}$;
3. combinamos: $\mathrm{CAM}=\mathrm{ReLU}\big(\sum_k\alpha_kA^k\big)$ (o ReLU mantém só as regiões que *aumentam* a previsão);
4. ampliamos o mapa 4×4 até 64×64 e sobrepomos à imagem.

Como o encoder processa **cada quadro** da janela, obtemos **5 mapas** (um por instante): dá para ver *quando* e *onde*
o modelo prestou atenção. É o papel que o SHAP teve no TCC, mas muito mais barato (um passe de gradiente, em vez de
milhares de avaliações) — em contrapartida é menos rigoroso teoricamente (não tem as propriedades axiomáticas do SHAP) e,
com uma camada 4×4, é grosseiro espacialmente.

In [ ]:
def carregar_modelo_treinado(variante: str, seed: int):
    """Recarrega os pesos gravados de uma execução (só existem para a primeira seed)."""
    pt = DIR_SAIDA / "runs" / (nome_run(variante, seed) + ".pt")
    if not pt.exists():
        return None
    ck = torch.load(pt, map_location=device, weights_only=False)
    modelo = ModeloTC(cfg, variante).to(device)
    modelo.load_state_dict(ck["estado"])
    modelo.eval()
    return modelo, ck["mu"].to(device), ck["sd"].to(device), ck["meta"]


class GradCAM:
    """Grad-CAM sobre o último bloco convolucional do encoder (um mapa por quadro da janela)."""

    def __init__(self, modelo: ModeloTC):
        self.modelo, self.ativ, self.grad = modelo, None, None
        self._h = modelo.encoder.convs[-1].register_forward_hook(self._guardar)

    def _guardar(self, _mod, _inp, saida):
        self.ativ = saida
        saida.register_hook(lambda g: setattr(self, "grad", g))   # captura d(saída)/d(ativação) na volta

    def remover(self):
        self._h.remove()

    def __call__(self, x: torch.Tensor, tarefa: str) -> torch.Tensor:
        """x: (B, T, 4, H, W) → mapas (B, T', H, W) em [0, ∞); T' = 5 (ou 1 na variante sem sequência)."""
        self.modelo.eval(); self.modelo.zero_grad(set_to_none=True)
        # o backward da LSTM do cuDNN só funciona em modo treino; sem cuDNN (só neste trecho) o modelo segue em eval
        with torch.backends.cudnn.flags(enabled=False):
            saidas, _ = self.modelo(x)
            saidas[tarefa].sum().backward()      # amostras não interagem (BatchNorm em modo eval) → um backward serve p/ o lote todo
        A, G = self.ativ.detach(), self.grad.detach()                      # (B·T', C, h, w)
        alfa = G.mean(dim=(2, 3), keepdim=True)                            # peso de cada canal
        cam = F.relu((alfa * A).sum(1, keepdim=True))                      # (B·T', 1, h, w)
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
        return cam.view(x.shape[0], -1, *x.shape[-2:])


def _janelas_do_teste(variante: str, seed: int):
    e = next(e for e in execucoes if e["variante"] == variante and e["seed"] == seed)
    return e["idx_teste"]


def explicar_gradcam(variante: str, seed: int, tarefa: str = "vmax") -> None:
    carregado = carregar_modelo_treinado(variante, seed)
    if carregado is None:
        print("sem pesos gravados para", variante, seed); return
    modelo, mu, sd, meta = carregado
    idx_te = _janelas_do_teste(variante, seed)
    vmax = janelas["vmax"].to_numpy()[idx_te]
    cam = GradCAM(modelo)
    try:
        # --- (a) três janelas de exemplo: fraca, intermediária e intensa (como no SHAP do TCC)
        alvos = [30, 70, 120]
        escolhidas = [int(idx_te[np.argmin(np.abs(vmax - a))]) for a in alvos]
        ib = torch.as_tensor(escolhidas, device=device)
        x = montar_lote(dados, ib, mu, sd, cfg, treino=False)
        mapas = cam(x, tarefa).cpu().numpy()                                # (3, T', 64, 64)
        T = mapas.shape[1]
        fig, ax = plt.subplots(3, T, figsize=(2.6 * T, 8), squeeze=False)
        for r in range(3):
            m = mapas[r] / (mapas[r].max() + 1e-9)                          # mesma escala nos quadros de uma janela
            for f in range(T):
                q = f + (5 - T)
                ir = x[r, q, 0].cpu().numpy()
                ax[r, f].imshow(ir, cmap="gray_r", vmin=np.percentile(ir, 1), vmax=np.percentile(ir, 99))
                ax[r, f].imshow(m[f], cmap="jet", alpha=0.45, vmin=0, vmax=1); ax[r, f].axis("off")
                if r == 0:
                    ax[r, f].set_title(f"t-{(4 - q) * cfg.passo_horas}h" if q < 4 else "t", fontsize=11)
            ax[r, 0].text(-0.08, 0.5, f"Vmax real = {janelas['vmax'].iloc[escolhidas[r]]:.0f} nós", transform=ax[r, 0].transAxes,
                          rotation=90, va="center", ha="right", fontsize=10)
        fig.suptitle(f"Grad-CAM no encoder — {variante} — tarefa {tarefa} (vermelho = região que mais pesou)", y=0.995)
        plt.tight_layout(); plt.savefig(DIR_SAIDA / "figuras" / f"05_gradcam_exemplos_{tarefa}.png", dpi=105); plt.show()

        # --- (b) mapa MÉDIO sobre várias janelas de teste (equivalente ao SHAP médio do TCC)
        rng = np.random.default_rng(0)
        amostra = rng.choice(idx_te, size=min(cfg.gradcam_amostras, len(idx_te)), replace=False)
        soma, n = None, 0
        for i in range(0, len(amostra), 64):                       # lotes pequenos: o backward guarda ativações
            xb = montar_lote(dados, torch.as_tensor(amostra[i:i + 64], device=device), mu, sd, cfg, treino=False)
            mb = cam(xb, tarefa)
            soma = mb.sum(0) if soma is None else soma + mb.sum(0)
            n += xb.shape[0]
        medio = (soma / n).cpu().numpy()
        fig, ax = plt.subplots(1, medio.shape[0], figsize=(2.6 * medio.shape[0], 3), squeeze=False)
        for f in range(medio.shape[0]):
            im = ax[0, f].imshow(medio[f], cmap="hot"); ax[0, f].axis("off")
            ax[0, f].set_title(f"t-{(4 - f - (5 - medio.shape[0])) * cfg.passo_horas}h" if f + (5 - medio.shape[0]) < 4 else "t")
        fig.suptitle(f"Grad-CAM médio sobre {n} janelas de teste — {variante} ({tarefa})", y=1.02)
        plt.tight_layout(); plt.savefig(DIR_SAIDA / "figuras" / f"06_gradcam_medio_{tarefa}.png", dpi=105, bbox_inches="tight"); plt.show()
    finally:
        cam.remover()


explicar_gradcam(cfg.variante_referencia, cfg.seeds[0], "vmax")
if "dvmax" in VARIANTES[cfg.variante_referencia]["tarefas"]:
    explicar_gradcam(cfg.variante_referencia, cfg.seeds[0], "dvmax")

### 8.2 Funções spline da KAN
Cada cabeça KAN é uma soma de funções de uma variável: na primeira camada,
$u_j=\sum_i\varphi_{0,j,i}(h_i)$, onde $h_i$ é a $i$-ésima variável latente de $h(t)$ (o resumo da LSTM). Cada $\varphi$ é uma curva que
pode ser **desenhada** — mostra *como* a saída muda quando só aquela variável latente varia.

Para saber **quais** variáveis importam, usamos a ideia de "importância de aresta" do artigo da KAN: a média de $|\varphi(h_i)|$ nos
dados (quanto a curva realmente contribui), ponderada pela importância do nó para o qual ela aponta (propagada da saída para a
entrada). Abaixo, para cada tarefa: (a) as variáveis latentes mais importantes e (b) as splines das arestas mais fortes de cada
uma, sobre o histograma dos valores que a variável de fato assume. Se a cabeça tem duas camadas, mostramos também a função da
segunda camada do nó oculto envolvido.

Observação: as variáveis latentes da LSTM não têm nome físico ("h₁₇"); o que se interpreta é a **forma** da curva
(crescente, decrescente, saturando, com degrau) e quais variáveis dominam cada tarefa — e se as duas tarefas usam variáveis
diferentes ou compartilhadas.

In [ ]:
@torch.no_grad()
def estados_h(modelo: ModeloTC, mu, sd, idx: np.ndarray) -> torch.Tensor:
    """h(t) de cada janela de `idx` (a entrada das cabeças)."""
    modelo.eval()
    hs = []
    for i in range(0, len(idx), cfg.tam_lote_aval):
        x = montar_lote(dados, torch.as_tensor(idx[i:i + cfg.tam_lote_aval], device=device), mu, sd, cfg, treino=False)
        hs.append(modelo(x)[1])
    return torch.cat(hs)


@torch.no_grad()
def importancia_kan(cabeca: CabecaKAN, H: torch.Tensor):
    """Importância de cada variável latente (e de cada aresta da 1ª camada) numa cabeça KAN.

    Importância de aresta = média de |φ(x)| nos dados. A importância do nó de saída é 1; propagando de trás para frente,
    a de um nó = soma das importâncias das arestas que saem dele (ponderadas pela importância do nó de destino)."""
    entradas, x = [], H
    for camada in cabeca.camadas:
        entradas.append(x)
        x = camada(x)
    imp_aresta = [c.valores_arestas(e).abs().mean(0) for c, e in zip(cabeca.camadas, entradas)]   # (n_out, n_in) por camada
    w = torch.ones(1, device=H.device)
    pesos_aresta = []
    for imp in reversed(imp_aresta):
        pa = imp * w[:, None]
        pesos_aresta.append(pa)
        w = pa.sum(0)                                   # importância dos nós de entrada daquela camada
    pesos_aresta.reverse()
    return w.cpu().numpy(), pesos_aresta, entradas       # w: (dim_oculta,) = importância de cada variável latente


def explicar_kan(variante: str, seed: int, tarefa: str) -> None:
    carregado = carregar_modelo_treinado(variante, seed)
    if carregado is None or VARIANTES[variante]["cabeca"] != "kan" or tarefa not in carregado[0].cabecas:
        print("sem KAN/pesos para explicar:", variante, tarefa); return
    modelo, mu, sd, _ = carregado
    idx = _janelas_do_teste(variante, seed)
    idx = np.random.default_rng(0).choice(idx, size=min(2000, len(idx)), replace=False)
    H = estados_h(modelo, mu, sd, idx)
    cabeca = modelo.cabecas[tarefa]
    score, pesos_aresta, entradas = importancia_kan(cabeca, H)
    topo = np.argsort(-score)[:cfg.spline_topk]
    p_imp = score / score.sum()                  # esparsidade: uma KAN "interpretável" concentraria a importância em poucas variáveis
    print(f"KAN ({tarefa}): as 10 variáveis mais importantes concentram {100 * np.sort(p_imp)[::-1][:10].sum():.1f}% da importância "
          f"(se fosse uniforme: {100 * 10 / len(p_imp):.1f}%) | nº efetivo de variáveis = {1 / np.sum(p_imp ** 2):.0f} de {len(p_imp)}")

    fig, ax = plt.subplots(1, 1, figsize=(7, 3))
    ordem = np.argsort(-score)[:20]
    ax.bar(range(len(ordem)), score[ordem], color="#1f5fa8"); ax.set_xticks(range(len(ordem)))
    ax.set_xticklabels([f"h{i}" for i in ordem], rotation=90, fontsize=8)
    ax.set_ylabel("importância"); ax.set_title(f"KAN ({tarefa}) — 20 variáveis latentes mais importantes de h(t)")
    plt.tight_layout(); plt.savefig(DIR_SAIDA / "figuras" / f"07_kan_importancia_{tarefa}.png", dpi=105); plt.show()

    camadas = cabeca.camadas
    linhas = 2 if len(camadas) > 1 else 1
    fig, ax = plt.subplots(linhas, len(topo), figsize=(3.6 * len(topo), 3.2 * linhas), squeeze=False)
    xs = torch.linspace(-1, 1, 200, device=device)
    nos_usados = []                                                          # nós ocultos distintos (p/ a 2ª linha)
    for c, i in enumerate(topo):
        j = int(pesos_aresta[0][:, i].argmax())                              # nó oculto mais ligado a h_i
        if j not in nos_usados:
            nos_usados.append(j)
        y = camadas[0].curva(j, int(i), xs).cpu().numpy()
        a2 = ax[0, c].twinx(); a2.hist(H[:, i].cpu().numpy(), bins=30, range=(-1, 1), color="gray", alpha=0.25); a2.set_yticks([])
        ax[0, c].plot(xs.cpu().numpy(), y, color="#c9731b", lw=2)
        ax[0, c].set_zorder(a2.get_zorder() + 1); ax[0, c].patch.set_visible(False)
        ax[0, c].set_title(f"φ(h{i}) → nó oculto {j}", fontsize=10); ax[0, c].set_xlabel(f"h{i}")
    if len(camadas) > 1:                                                     # 2ª camada: uma curva por nó oculto distinto
        for c in range(len(topo)):
            if c >= len(nos_usados):
                ax[1, c].axis("off"); continue
            j = nos_usados[c]
            u = entradas[1][:, j].cpu().numpy()
            lo, hi = np.percentile(u, [0.5, 99.5]); lo, hi = min(lo, -1), max(hi, 1)
            ug = torch.linspace(lo, hi, 200, device=device)
            a3 = ax[1, c].twinx(); a3.hist(u, bins=30, range=(lo, hi), color="gray", alpha=0.25); a3.set_yticks([])
            ax[1, c].plot(ug.cpu().numpy(), camadas[1].curva(0, j, ug).cpu().numpy(), color="#3a8f5a", lw=2)
            ax[1, c].set_zorder(a3.get_zorder() + 1); ax[1, c].patch.set_visible(False)
            ax[1, c].set_title(f"φ₂(u{j}) → saída", fontsize=10); ax[1, c].set_xlabel(f"u{j} (soma da 1ª camada)")
    ax[0, 0].set_ylabel("contribuição à saída (padronizada)")
    fig.suptitle(f"Splines aprendidas pela cabeça KAN — tarefa {tarefa} (cinza: distribuição dos valores)", y=1.0)
    plt.tight_layout(); plt.savefig(DIR_SAIDA / "figuras" / f"08_kan_splines_{tarefa}.png", dpi=105, bbox_inches="tight"); plt.show()

    return set(map(int, np.argsort(-score)[:10]))


_top = {}
for _t in VARIANTES[cfg.variante_referencia]["tarefas"]:
    _top[_t] = explicar_kan(cfg.variante_referencia, cfg.seeds[0], _t)
if len(_top) == 2 and all(v is not None for v in _top.values()):
    _a, _b = _top["vmax"], _top["dvmax"]
    _acaso = 10 * 10 / cfg.dim_oculta                                            # sobreposição esperada com dois sorteios independentes
    _p_acaso = float(sps.hypergeom(cfg.dim_oculta, 10, 10).sf(len(_a & _b) - 1))   # P(sobreposição ≥ observada | acaso)
    print(f"variáveis latentes entre as 10 mais importantes nas duas tarefas: {sorted(_a & _b)} "
          f"({len(_a & _b)}/10 em comum; por acaso esperaríamos {_acaso:.1f}, P(≥ observado) = {_p_acaso:.2f})")

### 8.3 O que cada variável latente "mede"?

As splines mostram *como* a saída depende de cada variável latente $h_i$, mas $h_i$ é uma coordenada **arbitrária** do espaço
aprendido pela LSTM (girar esse espaço daria outra rede igualmente boa): "h26" não tem significado físico por si só. Para dar algum,
calculamos a correlação de Spearman (por postos) de cada $h_i$ com grandezas observáveis: Vmax e ΔVmax reais e resumos do quadro
$t$ (temperatura IR média/desvio/mínimo, fração de nuvens frias, PMW médio). Se as variáveis mais importantes de uma cabeça
tiverem correlação alta com alguma grandeza, a interpretação ganha base ("h64 cresce com a cobertura de nuvens frias"); se todas
forem baixas, as curvas descrevem uma representação que ainda não sabemos nomear — e a **esparsificação** da KAN (regularização
L1 + entropia e poda, como no artigo original) seria o próximo passo; ela **não** está implementada aqui porque muda o modelo
proposto.

In [ ]:
def spearman_colunas(H: np.ndarray, grandezas: Dict[str, np.ndarray]) -> pd.DataFrame:
    """Correlação de Spearman (por postos) de cada variável latente (colunas de H) com cada grandeza; pares com NaN são ignorados."""
    out = np.full((H.shape[1], len(grandezas)), np.nan)
    for j, g in enumerate(grandezas.values()):
        ok = ~np.isnan(g)
        if ok.sum() < 10 or np.ptp(g[ok]) == 0:
            continue
        rg = sps.rankdata(g[ok])
        rh = sps.rankdata(H[ok], axis=0)
        rg = (rg - rg.mean()) / (rg.std() + 1e-12)
        rh = (rh - rh.mean(0)) / (rh.std(0) + 1e-12)
        out[:, j] = (rh * rg[:, None]).mean(0)
    return pd.DataFrame(out, index=[f"h{i}" for i in range(H.shape[1])], columns=list(grandezas))


def explicar_latentes(variante: str, seed: int) -> None:
    carregado = carregar_modelo_treinado(variante, seed)
    if carregado is None or VARIANTES[variante]["cabeca"] != "kan":
        print("sem KAN/pesos para explicar:", variante); return
    modelo, mu, sd, _ = carregado
    idx_te = _janelas_do_teste(variante, seed)
    idx = np.random.default_rng(0).choice(idx_te, size=min(2000, len(idx_te)), replace=False)   # as MESMAS janelas de explicar_kan
    H = estados_h(modelo, mu, sd, idx)
    img = estatisticas_imagem_janelas(dados, idx, cfg)
    grandezas = {"Vmax real": janelas["vmax"].to_numpy(float)[idx], "ΔVmax real": janelas["dvmax"].to_numpy(float)[idx],
                 "IR média (K)": img["ir_media"], "IR desvio (K)": img["ir_dp"], "IR mínimo (K)": img["ir_min"],
                 "fração fria (IR<220K)": img["frac_frio"], "PMW média": img["pmw_media"]}
    corr = spearman_colunas(H.cpu().numpy(), grandezas)
    corr.to_csv(DIR_SAIDA / "tabelas" / f"kan_latentes_correlacoes_{variante}.csv")
    for tarefa in modelo.cabecas:
        score, _, _ = importancia_kan(modelo.cabecas[tarefa], H)
        topo = [f"h{i}" for i in np.argsort(-score)[:cfg.spline_topk]]
        print(f"\nKAN ({tarefa}) — correlação de Spearman das {cfg.spline_topk} variáveis latentes mais importantes com grandezas físicas:")
        display(corr.loc[topo].round(2))
    ca = corr.abs().fillna(0.0)
    melhor = pd.DataFrame({"melhor variável (de %d)" % corr.shape[0]: ca.idxmax(), "|ρ|": ca.max().round(2)})
    print("\nA variável latente mais correlacionada com cada grandeza (qualquer uma das variáveis, importante ou não):")
    display(melhor)


explicar_latentes(cfg.variante_referencia, cfg.seeds[0])

## 9. Fechamento: resumo, limitações e próximos passos

In [ ]:
resumo_final = dict(
    modo=cfg.modo, assinatura=ASSINATURA, dispositivo=str(device),
    experimento=dict(nome=cfg.nome_experimento, notas=cfg.notas, seeds=list(cfg.seeds)),
    dados=dict(imagens=int(len(tabela_bruta)), ciclones=int(tabela_bruta["id_ciclone"].nunique()), janelas=int(len(janelas)),
               duplicatas=resumo_duplicatas, quadros_invalidos=int((~tabela_serie["quadro_valido"]).sum())),
    protocolo=dict(divisao=cfg.divisao, descricao_divisao=particao.descricao, seeds=list(cfg.seeds), variantes=list(cfg.variantes),
                   referencia=cfg.variante_referencia,
                   janelas=dict(treino=int(len(particao.treino)), validacao=int(len(particao.val)), teste=int(len(particao.teste))),
                   ciclones=dict(treino=len(_ids_conjunto[0]), validacao=len(_ids_conjunto[1]), teste=len(_ids_conjunto[2]))),
    execucoes=int(len(df_exec)),
    resultados={v: {c: (None if np.isnan(m) else float(m))
                    for c in ("rmse_vmax", "mae_vmax", "rmse_dvmax", "mae_dvmax", "auc_ri", "ap_ri", "csi_ri", "csi_ri_aj")
                    for m in [df_exec.loc[df_exec.variante == v, c].mean()]}
                for v in cfg.variantes if (df_exec.variante == v).any()},
    baseline_sem_imagens=resumo_baseline,
    bootstrap_por_ciclone=testes_boot.round(5).to_dict("records") if len(testes_boot) else [],
)
(DIR_SAIDA / "resumo_final.json").write_text(json.dumps(resumo_final, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
resumo_final["execucoes_pendentes"] = int(n_pendentes)
(DIR_SAIDA / "resumo_final.json").write_text(json.dumps(resumo_final, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
print(json.dumps(resumo_final["resultados"], indent=2, ensure_ascii=False))
if n_pendentes:
    print(f"\n⚠ {n_pendentes} execuções do plano ficaram PENDENTES (orçamento de tempo). Rode o notebook de novo para completá-las.")
print("\nArquivos gerados em", DIR_SAIDA.resolve())
if cfg.modo == "teste":
    print("\n⚠ MODO TESTE: os números acima NÃO são resultados. Troque MODO para \"completo\" (seção 1) para o experimento real.")
for p in sorted(DIR_SAIDA.rglob("*")):
    if p.is_file() and p.suffix in (".csv", ".png", ".json"):
        print("  ", p.relative_to(DIR_SAIDA))

### Limitações conhecidas (e como tratá-las)
* **Uma seed só (esta versão).** Serve para iterar, não para concluir: o ruído de treino não é medido e uma melhora pequena pode ser acaso
  (veja "Como ler o resultado de 1 seed" no início). O protocolo de 5 seeds é o que sustenta os números do artigo.
* **Horizonte fixo e discreto.** A LSTM supõe passos regulares (3 h). Janelas com buracos são descartadas (não "consertadas").
  Uma Neural ODE modelaria o tempo de forma contínua — fica como trabalho futuro.
* **Pmin fora de escopo.** O TCIR já traz a coluna `MSLP` (pressão); a cabeça de Pmin pode ser adicionada como uma terceira
  tarefa em `VARIANTES` (basta um novo alvo e um novo termo na perda).
* **Testes por janela são otimistas** (janelas do mesmo ciclone são correlacionadas). O veredito é o **bootstrap por ciclone** (seção 7.3b);
  com um teste único (por padrão, os ≈ 94 ciclones de 2017) o intervalo é largo.
* **Sem folds.** O teste é um conjunto só: o resultado depende de quais ciclones caíram nele (a validação cruzada em 5 folds, que está na pipeline
  completa, testa todos os ciclones). Veja «Folds: o que foi removido» no início.
* **ΔVmax × baseline sem imagens (seção 7.2b).** Se a rede não superar a regressão sobre o histórico de Vmax, a cabeça `dvmax`
  não está extraindo das imagens nada além da tendência de intensidade. Dar o histórico de Vmax como entrada da rede é uma
  decisão de projeto (muda o modelo proposto) e **não** foi feita aqui.
* **RI.** O limiar é ajustado na validação interna e há métricas sem limiar (AUC/AP), mas a cabeça continua sendo de regressão
  (MSE). Uma cabeça de classificação (BCE com peso na classe rara) é uma alternativa **não implementada** (muda o modelo).
* **KAN sem esparsificação.** As variáveis latentes têm importância espalhada (seção 8.2). Regularização L1/entropia e poda das
  splines, como no artigo original da KAN, **não** foram aplicadas (mudam o treino do modelo proposto); a seção 8.3 só
  correlaciona as variáveis latentes com grandezas físicas.
* **Valores a confirmar:** horizonte Δ (24 h), probabilidade de rotação (0,55), encoder (aqui, estilo CNN-TC),
  quais cabeças entram na primeira versão do artigo.

## 10. Relatório da rodada e pacote `.zip` para baixar

Duas células, **nesta ordem** e por último (elas só empacotam o que as anteriores já geraram):

1. **`RELATORIO.md`** — resumo da rodada num arquivo só: quem é a rodada (nome, seed, notas), o plano, o desempenho por variante, o treino
   (melhor época e sobreajuste), as comparações estatísticas e a lista de arquivos. Também é impresso aqui.
2. **`resultados_<NOME_EXPERIMENTO>_seed<SEED>.zip`** — a pasta de saída inteira (figuras, tabelas, predições `runs/*.npz`, pesos `runs/*.pt`,
   `config.json`, `resumo_final.json`, `RELATORIO.md`). No Kaggle, baixe-o no painel *Output*; as figuras também ficam soltas em `figuras/`.
   O `cache/` de quadros (≈ 2,5 GB) fica de fora e é apagado da saída no final: ele é refeito em 1–5 min na próxima sessão.


In [ ]:
def fmt2(x) -> str:
    """Número com 2 casas, ou '—' se for NaN/None."""
    return "—" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.2f}"


def tabela_md(df: pd.DataFrame, nome_indice: str = "") -> str:
    """DataFrame → tabela em Markdown (sem depender do pacote `tabulate`)."""
    cab = [nome_indice or (df.index.name or "")] + [str(c) for c in df.columns]
    linhas = ["| " + " | ".join(cab) + " |", "|" + "|".join("---" for _ in cab) + "|"]
    for ind, r in df.iterrows():
        linhas.append("| " + " | ".join([str(ind)] + ["—" if (isinstance(x, float) and np.isnan(x)) else str(x) for x in r.tolist()]) + " |")
    return "\n".join(linhas)


def tabela_treino() -> pd.DataFrame:
    """Melhor época e folga treino × validação (RMSE de Vmax na melhor época de cada execução, média entre execuções)."""
    linhas = []
    for v in cfg.variantes:
        g, e = df_hist[df_hist["variante"] == v], df_exec[df_exec["variante"] == v]
        if g.empty or e.empty:
            continue
        melhor = g[g["epoca"] == g["melhor_epoca"]]                          # a linha da melhor época de cada execução
        tr, va = float(melhor["rmse_treino_vmax"].mean()), float(melhor["rmse_val_vmax"].mean())
        linhas.append({"variante": v, "melhor época (média)": f"{e['melhor_epoca'].mean():.1f}",
                       "épocas rodadas (média)": f"{e['n_epocas'].mean():.1f} de {cfg.epocas_max}",
                       "pararam antes do fim": f"{int((e['n_epocas'] < cfg.epocas_max).sum())}/{len(e)}",
                       "RMSE Vmax treino": fmt2(tr), "RMSE Vmax validação": fmt2(va), "folga (val − treino)": fmt2(va - tr),
                       "tempo/execução": formatar_tempo(float(e["segundos"].mean()))})
    return pd.DataFrame(linhas).set_index("variante")


DESCRICAO_FIGURAS = {
    "00": "distribuição de Vmax, de ΔVmax e das janelas por ano", "01": "tensor de entrada de uma janela (sem e com rotação)",
    "02": "Vmax predito × real (conjunto de teste)", "03": "RMSE de cada execução e curvas de validação",
    "04": "MAE de Vmax por faixa de intensidade", "05": "Grad-CAM de exemplos (onde a CNN olhou)", "06": "Grad-CAM médio",
    "07": "importância das variáveis latentes na KAN", "08": "splines aprendidas pela KAN", "09": "janelas de maior erro de Vmax",
    "10": "curvas de treino por seed: perda e RMSE, treino × validação", "11": "curvas de treino: resumo entre variantes",
    "12": "RMSE na validação × no teste", "13": "viés, RMSE e nº de janelas por categoria de intensidade; distribuição do erro", "14": "ΔVmax predito × real",
    "15": "curvas ROC e precisão × revocação da intensificação rápida"}
DESCRICAO_TABELAS = {
    "metricas_por_execucao.csv": "uma linha por (variante, seed) com todas as métricas", "resumo_por_variante.csv": "média ± desvio por variante",
    "historico_treino.csv": "época a época: perdas, RMSE de treino e de validação, taxa de aprendizado, tempo",
    "bootstrap_por_ciclone.csv": "teste principal (IC 95% e p ajustado por Holm)",
    "wilcoxon_por_janela.csv": "Wilcoxon por janela (apoio)",
    "baselines_dvmax.csv": "baselines de ΔVmax sem imagens", "dvmax_modelos_vs_baselines.csv": "ΔVmax e RI: redes × baselines",
    "janelas_erro_grande_vmax.csv": "janelas com erro grande de Vmax",
    "kan_latentes_correlacoes_": "correlação de Spearman das variáveis latentes da KAN com grandezas físicas"}


def descricao_tabela(nome: str) -> str:
    """Descrição de uma tabela: pelo nome exato ou pelo começo do nome (tabelas com o nome da variante no fim)."""
    return next((d for k, d in DESCRICAO_TABELAS.items() if nome == k or nome.startswith(k)), "")


def gerar_relatorio() -> str:
    gpu = f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""
    t = [f"# Relatório da rodada «{cfg.nome_experimento}» — seed {', '.join(str(s) for s in cfg.seeds)}", ""]
    if cfg.modo == "teste":
        t += ["> ⚠ **MODO TESTE**: poucos ciclones e 2 épocas. Os números abaixo **não são resultado** — só confirmam que o ambiente funciona.", ""]
    if n_pendentes:
        t += [f"> ⚠ **{n_pendentes} execuções do plano ficaram PENDENTES** (orçamento de tempo). Rode de novo para completá-las.", ""]
    t += [f"- **Gerado em:** {time.strftime('%Y-%m-%d %H:%M')} · **modo:** {cfg.modo} · **dispositivo:** {device.type}{gpu} · **assinatura:** `{ASSINATURA}`",
          f"- **Plano:** {len(df_exec)} treinos = {len(cfg.seeds)} seed(s) {list(cfg.seeds)} × "
          f"{len(cfg.variantes)} variantes · até {cfg.epocas_max} épocas (paciência {cfg.paciencia}) · tempo de treino somado: "
          f"{formatar_tempo(float(df_exec['segundos'].sum()))}",
          f"- **Dados:** {len(janelas)} janelas de {janelas['id_ciclone'].nunique()} ciclones "
          f"({resumo_duplicatas['registros_removidos']} registros duplicados removidos; {int((~tabela_serie['quadro_valido']).sum())} quadros inválidos)",
          f"- **Divisão dos dados:** {particao.descricao} — treino {len(particao.treino)} janelas ({len(_ids_conjunto[0])} ciclones), "
          f"validação {len(particao.val)} ({len(_ids_conjunto[1])}), teste {len(particao.teste)} ({len(_ids_conjunto[2])})",
          f"- **Variantes:** {', '.join(cfg.variantes)} (referência: `{cfg.variante_referencia}`)",
          f"- **Notas:** {cfg.notas or '(sem notas)'}", ""]

    t += ["## Desempenho no teste", "",
          "Conjunto de teste. Média ± desvio entre seeds" + (" (com 1 seed só aparece o valor)" if len(cfg.seeds) == 1 else "") + ". Erros em nós.", ""]
    cols = {"execuções": "execuções", "rmse_vmax": "Vmax RMSE", "mae_vmax": "Vmax MAE", "vies_vmax": "Vmax viés",
            "rmse_dvmax": "ΔVmax RMSE", "mae_dvmax": "ΔVmax MAE", "parâmetros": "parâmetros"}
    t += [tabela_md(tab_resumo[list(cols)].rename(columns=cols), "variante"), ""]

    if "tab_dvmax" in globals() and len(tab_dvmax):
        t += ["## ΔVmax e intensificação rápida (RI) contra os baselines sem imagens", "",
              f"RI = ΔVmax ≥ {cfg.limiar_ri_nos:.0f} nós em {cfg.horizonte_horas} h. Sem habilidade: AUC ≈ 0,50 e AP ≈ prevalência "
              f"({100 * float((janelas['dvmax'].dropna() >= cfg.limiar_ri_nos).mean()):.1f}%). `csi_ri_aj` usa o limiar ajustado na validação interna.", ""]
        ver = [c for c in ("tipo", "rmse_dvmax", "mae_dvmax", "auc_ri", "ap_ri", "csi_ri_aj") if c in tab_dvmax.columns]
        t += [tabela_md(tab_dvmax[ver], "modelo"), ""]

    if len(df_hist):
        t += ["## Treino: melhor época e sobreajuste", "",
              "RMSE de Vmax na melhor época (a que o *early stopping* guarda). O treino é medido em **modo de avaliação** (sem dropout nem rotação) "
              "numa amostra fixa de janelas de treino; folga positiva = a rede erra mais na validação do que no treino (sobreajuste).", "",
              tabela_md(tabela_treino(), "variante"), ""]

    if len(testes_boot):
        b = testes_boot[["comparacao", "tarefa", "metrica", "dif", "ic95_inf", "ic95_sup", "p_holm", "veredito"]].copy()
        b["dif [IC 95%]"] = [f"{d:+.3f} [{lo:+.3f}; {hi:+.3f}]" for d, lo, hi in zip(b["dif"], b["ic95_inf"], b["ic95_sup"])]
        b["p_holm"] = b["p_holm"].map(lambda p: f"{p:.3f}")
        t += ["## Comparações (bootstrap por ciclone)", "",
              "dif = referência − outra; negativo = a referência erra menos (nós). Convincente quando o IC 95% não inclui 0 (o teste é um conjunto só: o IC é largo).", "",
              tabela_md(b[["comparacao", "tarefa", "metrica", "dif [IC 95%]", "p_holm", "veredito"]].set_index("comparacao"), "comparação"), ""]

    t += ["## Como ler esta rodada", "",
          "- **Seed única:** duas rodadas com a mesma configuração e seeds diferentes não dão o mesmo número (e, com LSTM, nem a mesma seed repete o treino na GPU). "
          "Com um teste único, a margem de ruído de uma seed é de ≈ 0,45 nó em Vmax e ≈ 0,67 nó em ΔVmax (medida na execução completa anterior, um fold por vez); "
          "só diferenças maiores que isso entre duas rodadas de 1 seed destoam do ruído. Confirme com mais seeds antes de adotar uma mudança que ganhe pouco.",
          "- **Teste único:** o teste é um conjunto só (por padrão, os ciclones de 2017): o resultado depende de quais ciclones caíram nele. Compare rodadas na **mesma divisão** "
          "e olhe `12_validacao_vs_teste.png` (validação × teste).",
          "- **Sobreajuste:** veja `10_curvas_treino_*.png` e a folga acima; se a validação sobe enquanto o treino cai, a rede está decorando.", ""]

    arq_fig = [p.name for p in sorted((DIR_SAIDA / "figuras").glob("*.png"))]
    arq_tab = [p.name for p in sorted((DIR_SAIDA / "tabelas").glob("*.csv"))]
    t += ["## Arquivos desta rodada", "", "**figuras/**", ""]
    t += [f"- `{n}` — {DESCRICAO_FIGURAS.get(n[:2], '')}" for n in arq_fig]
    t += ["", "**tabelas/**", ""] + [f"- `{n}` — {descricao_tabela(n)}" for n in arq_tab]
    t += ["", "**runs/** — `*.npz` (predições do conjunto de teste, rótulos e histórico de cada treino) e `*.pt` (pesos).",
          "**config.json** — todos os parâmetros · **resumo_final.json** — resumo em JSON.", ""]
    return "\n".join(t)


texto_relatorio = gerar_relatorio()
(DIR_SAIDA / "RELATORIO.md").write_text(texto_relatorio, encoding="utf-8")
if "ipykernel" in sys.modules:
    from IPython.display import Markdown
    display(Markdown(texto_relatorio))
else:
    print(texto_relatorio)
print("\nRelatório salvo em", (DIR_SAIDA / "RELATORIO.md").resolve())

In [ ]:
import zipfile

INCLUIR_PESOS = True            # .pt de cada execução (≈ 4 MB cada). False = zip bem menor (as figuras de Grad-CAM e KAN já estão nos PNGs)
INCLUIR_CACHE = False           # cache/ de quadros (≈ 2,5 GB): é refeito em 1–5 min, não precisa ir junto
APAGAR_CACHE_NO_FINAL = True    # apaga cache/ da pasta de saída depois de zipar (no Kaggle, 2,5 GB a menos no "Output" da versão)

NOME_PACOTE = NOME_RODADA + ("_TESTE" if cfg.modo == "teste" else "")
# No Kaggle o zip fica em /kaggle/working (aparece no painel Output). Fora dele, ao lado da pasta de saída (nunca dentro dela).
RAIZ_ZIP = Path("/kaggle/working") if os.path.exists("/kaggle/working") else DIR_SAIDA.parent
ZIP_SAIDA = RAIZ_ZIP / f"resultados_{NOME_PACOTE}.zip"
ZIP_SAIDA.unlink(missing_ok=True)                  # não acumula versões nem zipa o zip antigo

n_arquivos, bytes_origem = 0, 0
with zipfile.ZipFile(ZIP_SAIDA, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for dirpath, dirnames, filenames in os.walk(DIR_SAIDA):
        dirnames[:] = [d for d in dirnames if d not in {"__pycache__", ".ipynb_checkpoints"} and (INCLUIR_CACHE or d != "cache")]
        for nome in filenames:
            f = Path(dirpath) / nome
            if f.is_symlink() or f.suffix == ".zip" or (not INCLUIR_PESOS and f.suffix == ".pt"):
                continue
            zf.write(f, arcname=Path(NOME_PACOTE) / f.relative_to(DIR_SAIDA))   # dentro do zip tudo fica numa pasta com o nome da rodada
            n_arquivos += 1
            bytes_origem += f.stat().st_size

print(f"{n_arquivos} arquivos | {bytes_origem / 1e6:.1f} MB → {ZIP_SAIDA.stat().st_size / 1e6:.1f} MB (zip)")
print("Zip:", ZIP_SAIDA.resolve())
if os.path.exists("/kaggle/working"):
    try:
        onde = f"{DIR_SAIDA.relative_to(RAIZ_ZIP)}/figuras/"
    except ValueError:                              # pasta de saída fora de /kaggle/working
        onde = f"{DIR_SAIDA}/figuras/"
    print(f"Para baixar: painel Output → {ZIP_SAIDA.name}  (as figuras também ficam soltas em {onde})")

if APAGAR_CACHE_NO_FINAL and not INCLUIR_CACHE and (DIR_SAIDA / "cache").exists():
    shutil.rmtree(DIR_SAIDA / "cache", ignore_errors=True)
    print("cache de quadros apagado da pasta de saída (a próxima sessão o refaz em 1–5 min)")
